In [1]:
:set -XOverloadedStrings
:set -XScopedTypeVariables
:set -XRankNTypes
:set -XFlexibleInstances
:set -XFlexibleContexts
:set -XDeriveFunctor
:set -XGeneralizedNewtypeDeriving
:load ../lib/src/Quantale.hs ../lib/src/KanExtension.hs ../lib/src/Bitopos.hs ../lib/src/Distribution.hs ../lib/src/SubjectiveModel.hs
import Quantale
import KanExtension
import Bitopos
import Distribution
import SubjectiveModel
import Data.List (sortBy, nub, minimumBy, maximumBy)
import Data.Ord (comparing, Down(..))
import Data.Maybe (fromMaybe, mapMaybe)
import qualified Data.Map.Strict as Map
import Control.Monad (filterM)
import System.IO (hSetBuffering, stdout, BufferMode(..))
hSetBuffering stdout LineBuffering
putStrLn "\x2705 SETUP OK: Subjective Modeling (Pyt'ev) + lib (Quantale/Kan/Bitopos/Distribution/SubjectiveModel)"

✅ SETUP OK: Subjective Modeling (Pyt'ev) + lib (Quantale/Kan/Bitopos/Distribution/SubjectiveModel)

# ❓ Теория Субъективного Моделирования Пытьева

> **Источник:** Ю.П. Пытьев, «Математические методы субъективного моделирования в научных исследованиях»  
> **Часть 1** — Математические и эмпирические основы, Вестник МГУ, Физика. Астрономия, 2018, №1  
> **Часть 2** — Приложения, Вестник МГУ, Физика. Астрономия, 2018, №2

---

## Содержание

| № | Раздел | Уровень |
|---|--------|---------|
| 0 | Введение: связи с Kan и Toposes | Категорное |
| 1 | Пространство субъективной модели и неопределённый элемент | Пытьев |
| 2 | Дуальные шкалы $L$ и $\bar{L}$ | Пытьев |
| 3 | Меры правдоподобия $\mathrm{Pl}$ и доверия $\mathrm{Bel}$ | Пытьев |
| 4 | Принцип относительности — группа автоморфизмов $\Gamma$ | Пытьев |
| 5 | Pl-интегралы и Bel-интегралы | Пытьев |
| 6 | Субъективная независимость НОЭ | Пытьев |
| 7 | Условные субъективные распределения | Пытьев |
| 8 | Альтернативные варианты мер Pl и Bel | Пытьев |
| 9 | Эмпирические основы: восстановление модели | Пытьев |
| 10 | Комбинирование субъективных и эмпирических данных | Пытьев |
| 11 | Энтропия субъективной неопределённости (Часть 2) | Пытьев |
| 12 | Идентификация состояний НО.НЧ.О. (Часть 2) | Пытьев |
| 13 | $[0,1]$ как Quantale: внутренняя топологическая структура | Категорное |
| 14 | Двойственность Исбелла и гипотеза эквивалентности подходов | Категорное |
| 14.5 | Теоремы: состоятельность и эквивалентность Pl/Bel | Категорное |
| 14.6 | Изоморфизм конструкций: битопос ≅ Кан | Категорное |
| 14.7 | Бесконечный случай: заметки и тропы (research notes) | Категорное |
| 14.8 | Бесконечный X: теорема тропы 1 (CompMaxMeas ≅ Filt) | Категорное |
| 14.9 | Бесточечная тропа: мера = сопряжение (ядро тропы 3) | Категорное |
| 14.10 | d-меры и билатис вердиктов (тропа 3b, дискретный этаж) | Категорное |
| 15 | Три сравнительных примера: битопос vs расширения Кана | Практика |
| 16 | Диагностика двигателя: три эксперта | Практика |
| 17 | Монада возможности — поссибилистский двойник монады Гири | Категорное |
| 18 | Обогащённая $\mathbf{X}$ и нетривиальная двойственность Исбелла | Категорное |

> **📦 Зависимости**
> **Пакеты:** `base`, `containers`
> **Библиотека курса:** `Bitopos`, `Distribution`, `KanExtension`, `Quantale`, `SubjectiveModel` (`src/lib`)
> **Расширения:** `DeriveFunctor` — deriving Functor ([→](Extensions.ipynb#deriving)) · `FlexibleContexts` — произвольные ограничения в контекстах ([→](Extensions.ipynb#flexiblecontexts)) · `FlexibleInstances` — инстансы для конкретных типов-применений ([→](Extensions.ipynb#flexibleinstances)) · `GeneralizedNewtypeDeriving` — newtype наследует инстансы обёрнутого типа ([→](Extensions.ipynb#deriving)) · `OverloadedStrings` — строковые литералы любого типа IsString ([→](Extensions.ipynb#overloadedstrings)) · `RankNTypes` — forall внутри аргументов функций ([→](Extensions.ipynb#rankntypes)) · `ScopedTypeVariables` — переменные типа из сигнатуры видны в теле ([→](Extensions.ipynb#scopedtypevariables))


## 0. Введение и зависимости

**SubjectiveModeling** — синтез двух глубоких идей из предшествующих ноутбуков:
- **[Расширения Кана](KanExtensions.ipynb)**: меры Pl и Bel суть левое (Lan) и правое (Ran) расширения Кана;
- **[Топосы](Toposes.ipynb)**: $[0,1]$ является полной алгеброй Хейтинга (фреймом, квантальным пространством),
  а пара $(\Omega_+, \Omega_-) = (\mathrm{Pl}, \mathrm{Bel})$ образует **битопос**;
- **[Неопределённость](Uncertainty.ipynb)**: тот же формальный аппарат — это **теория возможностей Пытьева**
  ($\Pi, N$ с нормировкой $\sup\tau=1$); субъективное моделирование отличается лишь интерпретацией (раздел 9).

### Таблица ключевых связей

| Понятие в этом ноутбуке | Откуда | Раздел |
|------------------------|--------|--------|
| $\mathrm{Pl}(E) = \mathrm{Lan}_J\,\tau(E)$ | [Расширения Кана](KanExtensions.ipynb) | Lan — левое расширение |
| $\mathrm{Bel}(E) = \mathrm{Ran}_J\,\bar{\tau}(E)$ | [Расширения Кана](KanExtensions.ipynb) | Ran — правое расширение |
| $[0,1]$ как фрейм (полная алгебра Хейтинга) | [Топосы](Toposes.ipynb) | T4 — внутренняя логика |
| Битопос $(\Omega_+, \Omega_-) = (\mathrm{Pl}, \mathrm{Bel})$ | [Топосы](Toposes.ipynb) | T5, T9 — битопос |
| $\tau: X \to [0,1]$ как пресноп | [Топосы](Toposes.ipynb) | T1 — предпучки |
| Двойственность Исбелла | [Расширения Кана](KanExtensions.ipynb) | Ran/Lan над $[0,1]$-Frm |
| Теория возможностей $\Pi$, $N$ ($\sup\tau=1$) | [Неопределённость](Uncertainty.ipynb) | Раздел 9 — возможность/необходимость |
| Группа автоморфизмов $\Gamma$ | — | Принцип относительности Пытьева |

### Граф зависимостей

![Граф зависимостей](../diagrams/subj/subj_deps.svg)


In [2]:
-- Концептуальная карта: SubjectiveModeling зависит от...

-- Из KanExtensions.ipynb:
-- Lan_J tau (E) = sup_{x in E} tau(x) = Pl(E)
-- Ran_J tauBar (E) = inf_{x not in E} tauBar(x) = Bel(E)

-- Из Toposes.ipynb:
-- [0,1] -- complete Heyting algebra (quantale, фрейм)
-- BiTopos: два классификатора Omega+ и Omega- <=> Pl и Bel

-- Принцип относительности Пытьева:
-- Группа Gamma = автоморфизмы [0,1] как решётки
-- <=> инвариантность под Gamma = только порядок имеет смысл

putStrLn "Карта зависимостей загружена."


Карта зависимостей загружена.

### Связь с теорией возможностей

Аппарат субъективного моделирования (правдоподобие $\mathrm{Pl}$, доверие $\mathrm{Bel}$) и
аппарат **теории возможностей** Пытьева (возможность $\Pi$, необходимость $N$) — это
**одна и та же** математическая конструкция. Общие дуальные шкалы
$L=([0,1],\max,\min)$, $\bar L=([0,1],\min,\max)$; совпадают формулы
$$\mathrm{Pl}(A)=\Pi(A)=\sup_{x\in A}\tau(x), \qquad
\mathrm{Bel}(A)=N(A)=\inf_{x\notin A}\bar\tau(x),$$
и интегралы Сугено. Возможность и необходимость вводятся **совместно** на дуальных шкалах с
самого начала; обе функции $\tau,\bar\tau$ задаются формально одинаково в обеих теориях.

Различается только **интерпретация**:

| | Теория возможностей | Субъективное моделирование |
|--|--|--|
| Природа | объективная, частотная альтернатива вероятности | выражение интуиции эксперта |
| Источник $\tau,\bar\tau$ | эмпирические частоты | суждение эксперта (эмпирика — один из вариантов) |
| Приоритет | согласие с наблюдаемыми частотами | мнение эксперта |

Базовая теория возможностей (частотное построение, построение по вероятностям, пример
$w_n=1/p^n$) — в ноутбуке [Uncertainty.ipynb](Uncertainty.ipynb), раздел 9.

# 1. Пространство Субъективной Модели и Неопределённый Элемент

> **Зачем.** Прежде чем считать, нужно договориться, *что* мы моделируем: не частоту события, а **суждение исследователя** о неслучайном объекте. Этот раздел вводит носитель таких суждений — пару мер Pl/Bel на всех подмножествах $X$.

## Определение (Пытьев 2018, Часть 1, п. 1.1)

Модельер-исследователь (МИ) задаёт **субъективную модель** случайной величины $\tilde{x}$ с множеством значений $X$ как **пространство с мерами правдоподобия и доверия**:

$$\boxed{(X,\;\mathcal{P}(X),\;\mathrm{Pl}^{\tilde{x}},\;\mathrm{Bel}^{\tilde{x}})}$$

где $\mathcal{P}(X)$ — класс всех подмножеств $X$, а меры задаются распределениями:

$$\tau^{\tilde{x}}(x) = \mathrm{Pl}^{\tilde{x}}(\tilde{x} = x), \quad x \in X, \quad \sup_{x \in X} \tau^{\tilde{x}}(x) = 1$$

$$\bar{\tau}^{\tilde{x}}(x) = \mathrm{Bel}^{\tilde{x}}(\tilde{x} = x), \quad x \in X, \quad \inf_{x \in X} \bar{\tau}^{\tilde{x}}(x) = 0$$

и для любого $E \subseteq X$:

$$\mathrm{Pl}^{\tilde{x}}(E) \;=\; \sup_{x \in E}\, \tau^{\tilde{x}}(x), \qquad \mathrm{Pl}^{\tilde{x}}(\varnothing) = 0,\quad \mathrm{Pl}^{\tilde{x}}(X) = 1$$

$$\mathrm{Bel}^{\tilde{x}}(E) \;=\; \inf_{x \in X \setminus E}\, \bar{\tau}^{\tilde{x}}(x), \qquad \mathrm{Bel}^{\tilde{x}}(X) = 1,\quad \mathrm{Bel}^{\tilde{x}}(\varnothing) = 0$$

## Интерпретация

**Неопределённый элемент (НОЭ)** $\tilde{x}$ — это модель субъективных суждений МИ о неизвестном значении $x \in X$:

- $\mathrm{Pl}^{\tilde{x}}(\tilde{x} = x)$ — насколько **относительно правдоподобно** равенство $\tilde{x} = x$
- $\mathrm{Bel}^{\tilde{x}}(\tilde{x} \neq x)$ — насколько следует **относительно доверять** неравенству $\tilde{x} \neq x$

«Относительно» означает: численные значения мер в $(0,1)$ **не допускают абсолютного содержательного толкования** — существенна лишь их упорядоченность.

## НОЭ как неопределённая высказывательная переменная (п. 1.2)

В теоретико-множественном представлении логики высказываний:
- $X$ — множество элементарных высказываний
- $\mathcal{P}(X)$ — класс всех высказываний
- Любое высказывание $a$ взаимно однозначно представлено множеством $A \subseteq X$ тех элементарных высказываний $x$, каждое из которых влечёт $a$

Правдоподобие $\mathrm{Pl}^{\tilde{x}}(A)$ — **относительное правдоподобие истинности** неопределённого высказывания $\tilde{x} \in A$.  
Доверие $\mathrm{Bel}^{\tilde{x}}(A)$ — **относительное доверие** неравенству $\tilde{x} \notin X \setminus A$ (т.е. $\tilde{x} \in A$).

## Эквивалентность нечётким мерам (Замечание 1.1)

Пространство $(X, \mathcal{P}(X), \mathrm{Pl}^{\tilde{x}}, \mathrm{Bel}^{\tilde{x}})$ **формально эквивалентно** нечёткому пространству $(X, \mathcal{P}(X), \mathrm{P}, \mathrm{N})$ с мерами **возможности** $\mathrm{P}$ и **необходимости** $\mathrm{N}$.

**Границы.** Pl/Bel — не вероятность: нет аддитивности, и $\mathrm{Pl}(E)+\mathrm{Pl}(X\setminus E)$ не обязана равняться 1. Аппарат уместен, когда частотная база отсутствует или нестабильна; при стабильном вероятностном пространстве честнее работает теория вероятностей.


# 2. Дуальные Шкалы $L$ и $\bar{L}$ — Структуры Решётки

> **Зачем.** Значения Pl и Bel живут не в «числах», а в **шкалах с порядком**; их две, и они дуальны. Без этого раздела непонятно, почему min/max, а не +/×.

## Теорема о единственности операций (Пытьев 2018, п. 1.3)

Из **принципа относительности** следует, что бинарные операции в шкалах значений мер Pl и Bel **однозначно определяются** условиями непрерывности, коммутативности и граничными условиями.

### Шкала $L$ (для правдоподобия)

$$\boxed{L = ([0,1],\, +,\, \times) = ([0,1],\, \max,\, \min)}$$

$$a + b = \max\{a, b\}, \qquad a \times b = \min\{a, b\}, \qquad a, b \in [0,1]$$

Граничные условия: $a + 0 = a \times 1 = a$, $\;a + 1 = 1$, $\;a \times 0 = 0$.

### Дуальная шкала $\bar{L}$ (для доверия)

$$\boxed{\bar{L} = ([0,1],\, \bar{+},\, \bar{\times}) = ([0,1],\, \min,\, \max)}$$

$$a\,\bar{+}\,b = \min\{a,b\}, \qquad a\,\bar{\times}\,b = \max\{a,b\}, \qquad a, b \in [0,1]$$

Граничные условия: $a\,\bar{+}\,1 = 1$, $\;a\,\bar{\times}\,0 = 0$ и пр.

### Откуда берётся дуальность?

Шкалы $L$ и $\bar{L}$ связаны **дуальным изоморфизмом** $\theta \in \Theta$, где $\Theta$ — класс строго монотонных убывающих функций $\theta: [0,1] \to [0,1]$, $\theta(0) = 1$, $\theta(1) = 0$:

$$\mathrm{Bel}^{\tilde{x}}(A) = \theta\!\left(\mathrm{Pl}^{\tilde{x}}(X \setminus A)\right), \qquad \bar{\tau}^{\tilde{x}}(x) = \theta(\tau^{\tilde{x}}(x))$$

Такое соответствие называется **дуальной согласованностью** мер Pl и Bel.

## Структура решётки

Как $L$, так и $\bar{L}$ являются **полными решётками** (complete lattices):
- В $L$: $\sup = \max$, $\inf = \min$
- В $\bar{L}$: $\sup = \min$, $\inf = \max$

### Уточнение: где настоящий билатис

Пара $(L, \bar{L})$ — это **одна** решётка с двумя взаимно обратными порядками, а не билатис: в билатисе (Гинзберг, Фиттинг) два порядка **независимы** — порядок истинности $\le_t$ и порядок информации $\le_k$. Настоящий билатис в теории Пытьева образуют **интервалы** $[\mathrm{Bel}(E), \mathrm{Pl}(E)] \subseteq [0,1]$:

$[a,b] \le_t [c,d] \iff a \le c \,\wedge\, b \le d \qquad\text{(истиннее)}$
$[a,b] \le_k [c,d] \iff a \le c \,\wedge\, d \le b \qquad\text{(информативнее: } [c,d] \subseteq [a,b])$

Выделенные элементы — в точности модели знания Пытьева (ср. п. 1.5):

| Элемент билатиса | Интервал | Интерпретация по Пытьеву |
|------------------|----------|--------------------------|
| $\top_t$ (истина) | $[1,1]$ | точное знание «событие достоверно» |
| $\bot_t$ (ложь) | $[0,0]$ | точное знание «событие невозможно» |
| $\bot_k$ (незнание) | $[0,1]$ | **абсолютное незнание**: $\mathrm{Bel}=0$, $\mathrm{Pl}=1$ |
| $\top_k$ (противоречие) | $[1,0]$ | $\mathrm{Bel} > \mathrm{Pl}$ — несогласованные данные (ср. критерий п. 2.2) |

Это структура Белнапа FOUR, растянутая на континуум — interlacing-законы проверяются в коде ниже.

**Границы.** Структуры решётки фиксируют только **порядок** — арифметика значений ($1-t$ и т.п.) есть лишь координатное представление (см. раздел 4).

In [3]:
-- | Раздел 2+: интервальный билатис — из библиотеки (../lib/Bitopos.hs)

demoBilattice :: IO ()
demoBilattice = do
  putStrLn "=== Razdel 2+: intervalnyj bilatis ==="
  putStrLn $ "  Reshyotka po <=t: " ++ show (checkLatticeLaws leqT joinT meetT)
  putStrLn $ "  Reshyotka po <=k: " ++ show (checkLatticeLaws leqK joinK meetK)
  putStrLn $ "  Interlacing:      " ++ show checkInterlacing
  putStrLn $ "  bot_k = " ++ show bUnknown ++ " (absolyutnoe neznanie)"
  putStrLn $ "  top_k = " ++ show bContra  ++ " (protivorechie, Bel > Pl)"
  putStrLn $ "  joinK neznanie tochn.znanie: " ++ show (joinK bUnknown bTrue)

demoBilattice

=== Razdel 2+: intervalnyj bilatis ===
  Reshyotka po <=t: True
  Reshyotka po <=k: True
  Interlacing:      True
  bot_k = IV {ivBel = 0.0, ivPl = 1.0} (absolyutnoe neznanie)
  top_k = IV {ivBel = 1.0, ivPl = 0.0} (protivorechie, Bel > Pl)
  joinK neznanie tochn.znanie: IV {ivBel = 1.0, ivPl = 1.0}

# 3. Меры Правдоподобия $\mathrm{Pl}$ и Доверия $\mathrm{Bel}$

> **Зачем.** Главные операции теории: как из поточечных распределений $\tau,\bar\tau$ получаются меры событий. Здесь же — формальная стыковка с теорией возможностей: тот же аппарат $\Pi/N$ из [Uncertainty.ipynb](Uncertainty.ipynb) (раздел 9), но числа выражают интуицию эксперта, а не частоту.

## Полная аддитивность мер (Пытьев 2018, п. 1.1)

Меры $\mathrm{Pl}_{\tau}$ и $\mathrm{Bel}_{\bar{\tau}}$ **вполне аддитивны** в смысле операций шкал $L$ и $\bar{L}$:

$$\mathrm{Pl}_{\tau}\!\left(\bigcup_{j \in J} E_j\right) = \sup_{j \in J} \mathrm{Pl}_{\tau}(E_j) = \bigoplus_{j \in J}^{L} \mathrm{Pl}_{\tau}(E_j)$$

$$\mathrm{Bel}_{\bar{\tau}}\!\left(\bigcap_{j \in J} E_j\right) = \inf_{j \in J} \mathrm{Bel}_{\bar{\tau}}(E_j) = \bigoplus_{j \in J}^{\bar{L}} \mathrm{Bel}_{\bar{\tau}}(E_j)$$

## Правдоподобия характеристик ОИ (п. 1.4)

Любая функция $\varphi: X \to Y$ задаёт НОЭ $\tilde{y} = \varphi(\tilde{x})$ со значениями в $Y$ и пространство $(Y, \mathcal{P}(Y), \mathrm{Pl}^{\tilde{y}}, \mathrm{Bel}^{\tilde{y}})$, где:

$$\tau^{\tilde{y}}(y) = \mathrm{Pl}^{\tilde{y}}(\tilde{y} = y) = \sup_{x: \varphi(x)=y} \tau^{\tilde{x}}(x)$$

$$\bar{\tau}^{\tilde{y}}(y) = \mathrm{Bel}^{\tilde{y}}(\tilde{y} = y) = \inf_{x: \varphi(x) \neq y} \bar{\tau}^{\tilde{x}}(x)$$

Для любого $A \subseteq Y$:

$$\mathrm{Pl}^{\tilde{y}}(A) = \sup_{y \in A} \tau^{\tilde{y}}(y), \qquad \mathrm{Bel}^{\tilde{y}}(A) = \inf_{y \notin A} \bar{\tau}^{\tilde{y}}(y)$$

> **⚠️ Тонкость двух прочтений $\bar\tau$ (доказательство и проверка — §14.6, C7).** Формула $\bar\tau^{\tilde y}(y) = \inf_{\varphi(x)\neq y}\bar\tau(x)$ вычисляет $\mathrm{Bel}_X(\varphi^{-1}\{y\})$ — доверие к **событию** $\tilde y = y$. Но мера $\mathrm{Bel}$ восстанавливается формулой $\inf_{y\notin A}$ не из доверий к синглетонам, а из **ко-синглетонного** распределения $\bar\tau_Y(y) = \mathrm{Bel}_Y(Y{\setminus}\{y\})$ (Теорема 2, §14.5: $\mathrm{Bel}$ определяется на ко-синглетонах). Подстановка «событийной» версии в $\inf_{y\notin A}$ ломается уже при $\varphi = \mathrm{id}$: для $\bar\tau = \{a{:}0.9,\ b{:}0.5,\ c{:}0.1\}$ она даёт $\mathrm{Bel}(\{b,c\}) = 0.1$ вместо $\bar\tau(a) = 0.9$. Функториально корректный образ ко-синглетонного распределения — $\bar\tau_Y(y) = \inf_{\varphi(x)=y}\bar\tau(x) = \mathrm{Ran}_\varphi\bar\tau$ (inf по слою), двойственно $\tau_Y = \mathrm{Lan}_\varphi\tau$ (sup по слою); тогда $\mathrm{Pl}_Y = \mathrm{Pl}_X\circ\varphi^{-1}$ и $\mathrm{Bel}_Y = \mathrm{Bel}_X\circ\varphi^{-1}$ — стандартные формулы образов мер возможности/необходимости (Dubois–Prade 1988). То же двойное прочтение объясняет столбец $\bar\tau$ «точного знания» в п. 1.5 ниже: как ко-синглетонное распределение ему отвечает $\bar\tau = \theta\tau = [x \neq x_0]$, а вариант $[x = x_0]$ — это синглетонные доверия. Библиотека курса (`belMeasure`) всюду использует ко-синглетонную семантику.

## Субъективные модели «абсолютного незнания» и «точного знания» (п. 1.5)

Инвариантные относительно выбора шкал $\gamma L$, $\bar{\gamma} \bar{L}$ модели:

$$\textbf{Абсолютное незнание:}\quad \tau^{\tilde{x}}(x) = 1,\; \bar{\tau}^{\tilde{x}}(x) = 0 \quad \forall x \in X$$

$$\Rightarrow \mathrm{Pl}^{\tilde{y}}(A) = 1,\; \mathrm{Bel}^{\tilde{y}}(A) = 0 \quad \forall \varphi: X \to Y,\; A \neq \varnothing$$

$$\textbf{Точное знание } x_0:\quad \tau^{\tilde{x}}(x) = \begin{cases} 1 & x = x_0 \\ 0 & x \neq x_0 \end{cases}, \quad \bar{\tau}^{\tilde{x}}(x) = \begin{cases} 1 & x = x_0 \\ 0 & x \neq x_0 \end{cases}$$

$$\Rightarrow \text{«точное знание» влечёт «точное знание» любого следствия модели}$$

**Границы.** $\sup$/$\inf$ делают меры **макситивными/минитивными**: вклад «многих средних свидетельств» не накапливается, как у суммы — это осознанная плата за работу без частот.


# 4. Принцип Относительности — Группа Автоморфизмов $\Gamma$

> **Зачем.** Если эксперт скажет «0.7», а другой «0.8» — значимо ли различие? Принцип относительности отвечает: инвариантен только **порядок**; конкретные числа — координаты.

## Группа $\Gamma$ (Пытьев 2018, п. 1.3)

**Группа автоморфизмов шкалы** $\Gamma$ — группа непрерывных строго монотонных функций:

$$\Gamma = \{\gamma: [0,1] \to [0,1] \mid \gamma(0) = 0,\; \gamma(1) = 1,\; \gamma \text{ строго монотонно возрастает}\}$$

с групповой операцией $(\gamma \circ \gamma')(a) = \gamma(\gamma'(a))$.

## Принцип относительности

Меры $\mathrm{Pl}^{\tilde{x}}$ и $\mathrm{Pl}'^{\tilde{x}}$ **эквивалентны**, если:

$$\exists\, \gamma \in \Gamma\; \forall E \in \mathcal{P}(X):\quad \gamma(\mathrm{Pl}^{\tilde{x}}(E)) = \mathrm{Pl}'^{\tilde{x}}(E)$$

Каждый МИ может формулировать модель в **своей шкале** $\gamma L$, $\bar{\gamma} \bar{L}$, $\gamma, \bar{\gamma} \in \Gamma$.

## Инвариантность автоморфизмов

Для любого $\gamma \in \Gamma$ и любых $a, b \in [0,1]$:

$$\gamma(a + b) = \gamma(a) + \gamma(b), \qquad \gamma(a \times b) = \gamma(a) \times \gamma(b)$$

$$a < b \;\Leftrightarrow\; \gamma(a) < \gamma(b)$$

Это означает: $\gamma$ — **автоморфизм решётки** $(L, \max, \min)$.

## Следствия принципа относительности

1. Численные значения $\mathrm{Pl}(E) \in (0,1)$ **не имеют абсолютного смысла** — только упорядоченность инвариантна
2. Содержательны лишь равенства $\mathrm{Pl}(E) = 0$ или $\mathrm{Pl}(E) = 1$ (не зависят от $\gamma$)
3. МИ может выбрать $\gamma$ так, чтобы значения мер были сколь угодно близки к 0 или 1

## Группы $\Gamma_S$ с выделенными значениями (п. 1.9.1)

Если коллективу МИ нужно **содержательно интерпретировать** некоторые значения $\alpha_i \in (0,1)$, они договариваются использовать подгруппу:

$$\Gamma_S = \{\gamma \in \Gamma \mid \gamma(\alpha_i) = \alpha_i,\; i = 1,\ldots,n\}$$

В частности, $\Gamma_{\{1/2\}}$ оставляет неподвижным значение $1/2$ (индифферентность МИ).

## Третий вариант: психофизическая группа (п. 1.9.2)

Группа $\Gamma'$ порождена преобразованиями $\gamma_\alpha(a) = a^\alpha$, $\alpha > 0$ (степенные функции Стивенса):

$$\gamma_\alpha(a + ' b) = \gamma_\alpha(a) +' \gamma_\alpha(b), \qquad \gamma_\alpha(a \times' b) = \gamma_\alpha(a) \times' \gamma_\alpha(b)$$

$\Gamma'$-инвариант: $\dfrac{\log \mathrm{Pl}'(A)}{\log \mathrm{Pl}'(B)} = \text{const}$ — допускает психофизическую интерпретацию.

**Границы.** Инвариантность к $\Gamma$ означает, что выводы, зависящие от арифметики значений (а не порядка), методологически нелегитимны в этой теории.


### 🔺 Категорный взгляд: Γ = Aut-группа квантали

Условия на $\gamma$ (сохранение $\max$, $\min$, порядка, концов отрезка) означают ровно одно: $\Gamma = \mathrm{Aut}\bigl([0,1], \max, \min\bigr)$ — **группа автоморфизмов квантали**. Принцип относительности тогда формулируется без слов «шкала»: субъективная модель — это объект не в $[0,1]$-значных функциях, а в **фактор-группоиде** $[\,\mathrm{Mod}/\Gamma\,]$; содержательны только $\Gamma$-инварианты (значения $0$, $1$ и порядок). Подгруппы $\Gamma_S$ с неподвижными точками $\alpha_i$ — это автоморфизмы, сохраняющие подквантали $[\alpha_i, \alpha_{i+1}]$; третий вариант мер — переход к другой квантали $([0,1], \max, \cdot)$, у которой $\mathrm{Aut}$ порождён $a \mapsto a^{\alpha}$. Действие $\Gamma$ **эквивариантно** относительно Pl: $\gamma(\mathrm{Pl}_{\tau}(E)) = \mathrm{Pl}_{\gamma\tau}(E)$ — это функториальность Lan по $\tau$, проверяется ниже.

In [4]:
-- | Раздел 4+: Gamma = Aut квантали — из библиотеки (../lib/Quantale.hs)

demoGammaAut :: IO ()
demoGammaAut = do
  putStrLn "=== Razdel 4+: Gamma kak avtomorfizmy kvantali ==="
  putStrLn $ "  t^2  - avtomorfizm:  " ++ show (isQuantaleAuto gammaSq)
  putStrLn $ "  sqrt - avtomorfizm:  " ++ show (isQuantaleAuto gammaSqrt)
  putStrLn $ "  theta - avtomorfizm: " ++ show (isQuantaleAuto theta)
             ++ " (eto dualnost, ne avtomorfizm)"
  -- эквивариантность: gamma(Pl_tau E) = Pl_{gamma.tau} E
  let dom = "abc"
      tau c = case c of { 'a' -> 1.0; 'b' -> 0.6; _ -> 0.2 }
      subs = filterM (const [True, False]) dom
      plW t e = unUI (plMeasure dom (ui . t) e)
      equiv gD = all (\e -> ui (gD (plW tau e)) =~ ui (plW (gD . tau) e)) subs
  putStrLn $ "  Ekvivariantnost Pl dlya t^2:  " ++ show (equiv (\t -> t * t))
  putStrLn $ "  Ekvivariantnost Pl dlya sqrt: " ++ show (equiv sqrt)
  let ordInv = and [ (a < b) == (gammaSq a < gammaSq b) | a <- gammaGrid, b <- gammaGrid ]
  putStrLn $ "  Poryadok invarianten: " ++ show ordInv

demoGammaAut

=== Razdel 4+: Gamma kak avtomorfizmy kvantali ===
  t^2  - avtomorfizm:  True
  sqrt - avtomorfizm:  True
  theta - avtomorfizm: False (eto dualnost, ne avtomorfizm)
  Ekvivariantnost Pl dlya t^2:  True
  Ekvivariantnost Pl dlya sqrt: True
  Poryadok invarianten: True

# 5. Pl-Интегралы и Bel-Интегралы

> **Зачем.** Чтобы принимать решения, нужны «средние» — аналоги интеграла Лебега. Здесь строятся интегралы по неаддитивным мерам (родственники интеграла Сугено).

## Определение 1.1 (Пытьев 2018, п. 1.6)

Пусть $\mathcal{L}(X)$ — класс функций $g: X \to L$ и $\bar{\mathcal{L}}(X)$ — класс функций $\bar{g}: X \to \bar{L}$.

**pl-интеграл** $\mathrm{pl}(\cdot): \mathcal{L}(X) \to L$ — **однородный и вполне аддитивный** функционал:

- **Однородность:** $\mathrm{pl}((a \times g)(\cdot)) = a \times \mathrm{pl}(g(\cdot))$ для любого $a \in [0,1]$
- **Полная аддитивность:** $\mathrm{pl}\!\left(\sum_j g_j\right)(\cdot) = \sum_j \mathrm{pl}(g_j(\cdot))$ в $L$

Аналогично **bel-интеграл** $\mathrm{bel}(\cdot): \bar{\mathcal{L}}(X) \to \bar{L}$ однороден и вполне аддитивен.

## Теорема 1.1: явное представление (Пытьев 2018, п. 1.6)

Для любого pl-интеграла $\exists!\; \tau: X \to L$ такая, что:

$$\boxed{\mathrm{pl}_{\tau}(g(\cdot)) = \sup_{x \in X} \min\{\tau(x),\, g(x)\}}$$

Для любого bel-интеграла $\exists!\; \bar{\tau}: X \to \bar{L}$ такая, что:

$$\boxed{\mathrm{bel}_{\bar{\tau}}(\bar{g}(\cdot)) = \inf_{x \in X} \max\{\bar{\tau}(x),\, \bar{g}(x)\}}$$

## Связь с мерами Pl и Bel

$$\mathrm{Pl}(E) = \mathrm{pl}_{\tau}(\chi_E(\cdot)) = \sup_{x \in E} \tau(x)$$

$$\mathrm{Bel}(E) = \mathrm{bel}_{\bar{\tau}}(\chi_E(\cdot)) = \inf_{x \in X \setminus E} \bar{\tau}(x)$$

## Интерпретация интегралов

Для функции потерь или ценности $f: X \to [0,1]$:

$$\mathrm{pl}_{\tau}(f) = \sup_{x \in X} \min\{\tau(x), f(x)\} \qquad \text{(«оптимистический» — наилучший сценарий)}$$

$$\mathrm{bel}_{\bar{\tau}}(f) = \inf_{x \in X} \max\{\bar{\tau}(x), f(x)\} \qquad \text{(«пессимистический» — наихудший сценарий)}$$

**Границы.** Pl/Bel-интегралы идемпотентны и устойчивы к выбросам, но «не чувствуют» массу повторений — там, где важна аккумуляция, нужен Лебег по вероятности.


In [5]:
-- | Разделы 1-5: субъективная модель — из библиотеки (../lib/SubjectiveModel.hs)

data State = SA | SB | SC deriving (Show, Eq, Ord, Enum, Bounded)

smDemo :: SubjModel State
smDemo = dualConsistent [SA, SB, SC] tau
  where
    tau SA = 1.0
    tau SB = 0.7
    tau SC = 0.3

demoS15 :: IO ()
demoS15 = do
  putStrLn "=== Razdely 1-5: SubjModel iz biblioteki ==="
  putStrLn $ "  Pl{SA,SB}  = " ++ show (smPl smDemo [SA, SB])
  putStrLn $ "  Bel{SA,SB} = " ++ show (smBel smDemo [SA, SB])
  putStrLn $ "  Dualno soglasovana: " ++ show (isDuallyConsistent smDemo)
  -- pl/bel-интегралы (Теорема 1.1)
  let f SA = 0.9
      f SB = 0.5
      f SC = 0.1
  putStrLn $ "  pl-integral f  = " ++ show (plIntegral [SA,SB,SC] (smTau smDemo) f)
  putStrLn $ "  bel-integral f = " ++ show (belIntegral [SA,SB,SC] (smTauBar smDemo) f)
  -- модели знания (п. 1.5)
  let ign = absoluteIgnorance [SA,SB,SC]
      knw = exactKnowledge [SA,SB,SC] SB
  putStrLn $ "  Neznanie:  Pl{SB}=" ++ show (smPl ign [SB]) ++ " Bel{SB}=" ++ show (smBel ign [SB])
  putStrLn $ "  Znanie SB: Pl{SB}=" ++ show (smPl knw [SB]) ++ " Bel{SB}=" ++ show (smBel knw [SB])
  -- образ под phi: X -> Bool
  let img = imageModel smDemo (== SA) [True, False]
  putStrLn $ "  Obraz phi: tau(True)=" ++ show (smTau img True)
             ++ " tau(False)=" ++ show (smTau img False)
  -- Полиморфизм ядра: Pl над Bool = exists, Bel = forall
  putStrLn $ "  [Bool] Pl{2,3}(x>1)  = " ++ show (plMeasure [1,2,3::Int] (> 1) [2,3])
  putStrLn $ "  [Bool] Bel{2,3}(x>1) = " ++ show (belMeasure [1,2,3::Int] (> 1) [2,3])

demoS15

=== Razdely 1-5: SubjModel iz biblioteki ===
  Pl{SA,SB}  = 1.0
  Bel{SA,SB} = 0.7
  Dualno soglasovana: True
  pl-integral f  = 0.9
  bel-integral f = 0.5
  Neznanie:  Pl{SB}=1.0 Bel{SB}=0.0
  Znanie SB: Pl{SB}=1.0 Bel{SB}=0.0
  Obraz phi: tau(True)=1.0 tau(False)=0.7
  [Bool] Pl{2,3}(x>1)  = True
  [Bool] Bel{2,3}(x>1) = False

# 6. Субъективная Независимость НОЭ

> **Зачем.** Совместные модели нескольких неопределённых элементов требуют понятия независимости — здесь оно задаётся через min/max вместо произведения.

## Определение 1.2 (Пытьев 2018, п. 1.7)

Пусть $\tilde{z} = (\tilde{z}_1, \ldots, \tilde{z}_n)$ — НОЭ со значениями в $Z_1 \times \cdots \times Z_n$.

**НОЭ $\tilde{z}_1, \ldots, \tilde{z}_n$ взаимно $\mathrm{Pl}$-независимы**, если правдоподобие события «$\forall i: \tilde{z}_i = z_i$»:

$$\tau^{\tilde{z}_1, \ldots, \tilde{z}_n}(z_1, \ldots, z_n) = \min_{1 \leq i \leq n} \tau^{\tilde{z}_i}(z_i) = \bigtimes_{i=1}^{n} \tau^{\tilde{z}_i}(z_i)$$

**НОЭ $\tilde{z}_1, \ldots, \tilde{z}_n$ взаимно $\mathrm{Bel}$-независимы**, если:

$$\bar{\tau}^{\tilde{z}_1, \ldots, \tilde{z}_n}(z_1, \ldots, z_n) = \max_{1 \leq i \leq n} \bar{\tau}^{\tilde{z}_i}(z_i) = \bar{\bigtimes}_{i=1}^{n} \bar{\tau}^{\tilde{z}_i}(z_i)$$

## Независимость событий

**События** $B_1, \ldots, B_n \in \mathcal{P}(Y)$ **взаимно $\mathrm{Pl}$-независимы**, если:

$$\mathrm{Pl}\!\left(\bigcap_{i=1}^n B_i\right) = \min_{1 \leq i \leq n} \mathrm{Pl}(B_i)$$

**$\mathrm{Bel}$-независимы**, если:

$$\mathrm{Bel}\!\left(\bigcup_{i=1}^n B_i\right) = \max_{1 \leq i \leq n} \mathrm{Bel}(B_i)$$

## Субъективная независимость (Определение 1.3)

**События $B_1, B_2$ субъективно $\mathrm{Pl}$-независимы**, если $\exists$ непрерывная $f: [0,1]^2 \to [0,1]$ такая, что:

$$\mathrm{Pl}(B_1 \cap B_2) = f(\mathrm{Pl}(B_1), \mathrm{Pl}(B_2))$$

**Теорема Пытьева:** взаимная независимость и субъективная независимость **эквивалентны**, причём $f(a,b) = \min(a,b)$.

## Связь Pl- и Bel-независимостей

Если меры Pl и Bel **дуально согласованы** ($\exists \theta \in \Theta: \mathrm{Bel}(A) = \theta(\mathrm{Pl}(X \setminus A))$), то $\mathrm{Pl}$- и $\mathrm{Bel}$-независимости **эквивалентны**.

**Границы.** Pl-независимость не влечёт Bel-независимость и наоборот; интуиция вероятностной независимости переносится лишь частично.


# 7. Условные Субъективные Распределения

> **Зачем.** Обновление суждений по наблюдению — субъективный аналог условной вероятности и основа применений (диагностика в разделе 16).

## Условное распределение правдоподобий (Пытьев 2018, п. 1.8)

Пусть $\tilde{z} = (\tilde{z}_1, \tilde{z}_2)$, $z_1 \in Z_1$, $z_2 \in Z_2$.

**Вариантом условного распределения правдоподобий** $z_1$ при условии $z_2 = z_2$ называется любое решение уравнения:

$$\min\{\tau^{\tilde{z}_1 | \tilde{z}_2}(z_1 | z_2),\; \tau^{\tilde{z}_2}(z_2)\} = \tau^{\tilde{z}_1, \tilde{z}_2}(z_1, z_2)$$

где $\tau^{\tilde{z}_2}(z_2) = \sup_{z_1 \in Z_1} \tau^{\tilde{z}_1, \tilde{z}_2}(z_1, z_2)$.

## Проблема субъективной шкалы (Определение 1.6)

Когда $0 < \tau^{\tilde{z}_2}(z_2) < 1$, условное распределение может быть не распределением правдоподобий в шкале $L$.

Решение: **субъективная шкала** $\gamma_{z_2} L$, где $\gamma_{z_2}: [0, \tau^{\tilde{z}_2}(z_2)] \to [0,1]$ — строго монотонная функция с $\gamma_{z_2}(0)=0$, $\gamma_{z_2}(\tau^{\tilde{z}_2}(z_2))=1$.

Тогда **условное правдоподобие** в субъективной шкале $\gamma_{z_2} L$:

$$\tau^{\tilde{z}_1 | \tilde{z}_2}_s(z_1 | z_2) = \gamma_{z_2} \circ \tau^{\tilde{z}_1, \tilde{z}_2}(z_1, z_2), \qquad z_1 \in Z_1$$

является распределением правдоподобий в $\gamma_{z_2} L$, в котором $z_2 = z_2$ — достоверное событие.

## Аналогия с условной вероятностью

В теории вероятностей: $\mathrm{Pr}(A | B) = \mathrm{Pr}(A \cap B) / \mathrm{Pr}(B)$ — нормирующий множитель $1/\mathrm{Pr}(B)$ задаёт «субъективную шкалу».

В теории Пытьева: переход в шкалу $\gamma_{z_2} L$ играет ту же роль, но в рамках принципа относительности.

**Границы.** Условные распределения определяются неоднозначно (есть варианты определения); выбор варианта — модельное решение.


### 🔺 Категорный взгляд: кондиционирование = residuation

Уравнение Пытьева $\min\{c,\; \tau^{\tilde{z}_2}(z_2)\} = \tau^{\tilde{z}_1,\tilde{z}_2}(z_1,z_2)$ — это вопрос о **правом сопряжённом** к функтору $\min(-, a)$ в квантали $([0,1], \min, 1)$. Его максимальное решение — в точности внутренний hom (residuation):

$$\tau^{\tilde{z}_1|\tilde{z}_2}(z_1|z_2) \;=\; \tau^{\tilde{z}_2}(z_2) \multimap \tau^{\tilde{z}_1,\tilde{z}_2}(z_1,z_2), \qquad a \multimap b = \begin{cases} 1 & a \le b \\ b & a > b \end{cases}$$

Сопряжение $\min(a,c) \le b \iff c \le (a \multimap b)$ гарантирует: (1) это решение уравнения, поскольку всегда $\tau^{\tilde{z}_1,\tilde{z}_2}(z_1,z_2) \le \tau^{\tilde{z}_2}(z_2)$; (2) оно **наибольшее** из решений; (3) $\sup_{z_1} \tau(z_1|z_2) = 1$ — условное распределение нормировано без перехода в субъективную шкалу $\gamma_{z_2}L$. Аналогия с $\Pr(A|B) = \Pr(A \cap B)/\Pr(B)$ точная: деление — это residuation квантали $([0,1], \cdot, 1)$ третьего варианта мер.

In [6]:
-- | Раздел 7+: кондиционирование = residuation (библиотека: condTau, qHom)

demoResiduation :: IO ()
demoResiduation = do
  putStrLn "=== Razdel 7+: kondicionirovanie = residuation ==="
  let dom = [(z1, z2) | z1 <- [1,2::Int], z2 <- [1,2::Int]]
      tauJ :: (Int, Int) -> Double
      tauJ (1,1) = 1.0
      tauJ (2,1) = 0.6
      tauJ (1,2) = 0.4
      tauJ (2,2) = 0.4
      tauJ _     = 0.0
      z1s = [1,2]
      z2s = [1,2]
      marg = margZ2 dom tauJ
      cnd z1 z2 = condTau dom tauJ z2 z1
  mapM_ (\(z1,z2) -> putStrLn $ "  tau(" ++ show z1 ++ "|" ++ show z2 ++ ") = "
         ++ show (cnd z1 z2)) [(a,b) | b <- z2s, a <- z1s]
  let chkEq = and [ ui (min (cnd z1 z2) (marg z2)) =~ ui (tauJ (z1, z2))
                  | z1 <- z1s, z2 <- z2s ]
      gridD = map (\k -> fromIntegral k / 20) [0..20 :: Int]
      chkMax = and [ c <= cnd z1 z2 + 1e-9
                   | z1 <- z1s, z2 <- z2s, c <- gridD
                   , ui (min c (marg z2)) =~ ui (tauJ (z1, z2)) ]
      chkNorm = and [ ui (maximum [cnd z1 z2 | z1 <- z1s]) =~ ltop | z2 <- z2s ]
  putStrLn $ "  Reshenie uravneniya: " ++ show chkEq
  putStrLn $ "  Maksimalnost:        " ++ show chkMax
  putStrLn $ "  Normirovka sup=1:    " ++ show chkNorm
  putStrLn $ "  Adjointness [0,1]:   " ++ show (checkResiduationAdj gammaGrid)
  putStrLn $ "  Adjointness Bool:    " ++ show (checkResiduationAdj [False, True])

demoResiduation

=== Razdel 7+: kondicionirovanie = residuation ===
  tau(1|1) = 1.0
  tau(2|1) = 0.6
  tau(1|2) = 1.0
  tau(2|2) = 1.0
  Reshenie uravneniya: True
  Maksimalnost:        True
  Normirovka sup=1:    True
  Adjointness [0,1]:   True
  Adjointness Bool:    True

# 8. Альтернативные Варианты Мер Pl и Bel

> **Зачем.** Первая пара (sup-min) — не единственная согласованная конструкция; здесь — систематика альтернатив и критерии выбора.

## Три варианта теории (Пытьев 2018, п. 1.9)

Пытьев рассматривает три варианта мер правдоподобия и доверия.

### Первый вариант (основной)

$$a + b = \max\{a,b\}, \quad a \times b = \min\{a,b\}$$

Группа $\Gamma$ — все непрерывные строго монотонные функции $[0,1] \to [0,1]$.  
**Числовые значения** мер, отличные от 0 и 1, **не допускают содержательной интерпретации**.

### Второй вариант (с фиксированными точками, п. 1.9.1)

Для содержательной интерпретации значений $\alpha_1, \ldots, \alpha_n \in (0,1)$ используется подгруппа $\Gamma_S \subset \Gamma$, оставляющая эти значения неподвижными.

Проекторы на интервалы $[\alpha_i, \alpha_{i+1}]$:

$$(u)^{\alpha} = \max\{\alpha, u\}, \qquad (u)_{\alpha} = \min\{\alpha, u\}$$

Шкала $L_{S'}$ разбивается на интервалы $[\alpha_i, \alpha_{i+1}]$, $i = 0, \ldots, n$, с группой $\Gamma_{S'} = \Gamma_{[\alpha_0,\alpha_1]} \otimes \cdots \otimes \Gamma_{[\alpha_n,\alpha_{n+1}]}$.

МИ могут содержательно интерпретировать **принадлежность** значений pl-интеграла неподвижным интервалам.

### Третий вариант (психофизический, п. 1.9.2)

Операции в шкале $L'$:

$$a +' b = \max\{a,b\}, \qquad a \times' b = a \cdot b \; (\text{обычное произведение})$$

Группа $\Gamma'$ порождена $\gamma_\alpha(a) = a^\alpha$, $\alpha > 0$.

Шкала $\bar{L}'$ значений $\mathrm{Bel}'$ — дуально изоморфная $L'$:

$$\theta_\beta(u) = \log_\beta u^{-1}, \quad 0 < u \leq 1, \qquad \bar{v} +' \bar{w} = \min\{\bar{v}, \bar{w}\}, \qquad \bar{v} \times' \bar{w} = \bar{v} + \bar{w}$$

Интегралы третьего варианта:

$$\mathrm{pl}'_{\tau}(f) = \max_{x \in X} (\tau(x) \cdot f(x))$$

$$\mathrm{bel}'_{\bar{\tau}}(\bar{g}) = \min_{x \in X} (\log_\beta \tau(x)^{-1} + \bar{g}(x))$$

**Инвариант третьего варианта** (допускает содержательную интерпретацию):

$$\frac{\log \mathrm{Pl}'(A)}{\log \mathrm{Pl}'(B)} = \text{const при всех } \gamma_\alpha \in \Gamma'$$

Это **отношение степеней** вероятностных оценок — аналог психофизических функций Стивенса.

**Границы.** Разные варианты дают разные численные ответы при одинаковых данных — сравнивать результаты можно только внутри одного варианта.


# 9. Эмпирические Основы: Восстановление Модели НО.НЧ.О.

> **Зачем.** Откуда брать $\tau$? Раздел показывает, как восстановить модель из упорядочивающих ответов эксперта/наблюдений — мост от слов к числам.

## Нечёткий неопределённый объект (НО.НЧ.О.)

**Неопределённый нечёткий объект** — объект, моделью которого является нечёткое пространство $(Y, \mathcal{P}(Y), \mathrm{P}(\cdot; x), \mathrm{N}(\cdot; x))$, зависящее от **неизвестного параметра** $x \in X$.

МИ предлагает субъективную модель НОЭ $\tilde{x}$ — пространство $(X, \mathcal{P}(X), \mathrm{Pl}^{\tilde{x}}, \mathrm{Bel}^{\tilde{x}})$.

## Эмпирическое оценивание по данным наблюдений (п. 2.1.2)

Если МИ доступны данные наблюдений $\eta = y_0 \in Y$ за НО.НЧ.О., то **нечёткий неопределённый элемент (НЧ.НОЭ)** $\hat{x} = \hat{x}(\eta)$, эмпирически оценивающий параметр $x \in X$, задаётся:

$$\tau^{\hat{x}}(x; y_0) = \gamma \circ g^{\eta}(y_0; x)$$

$$\bar{\tau}^{\hat{x}}(x; y_0) = \theta \circ g^{\eta}(y_0; x)$$

где $g^{\eta}(y; x)$ — распределение возможностей наблюдения $\eta = y$ при параметре $x$, $\gamma \in \Gamma$, $\theta \in \Theta$.

## Семейства оценивающих множеств максимального правдоподобия (п. 2.1.2)

$$\Phi^{-1}(y_0; \Lambda) = \{x \in X: g^{\eta}(y_0; x) \geq \min\{\Lambda, \max_{x' \in X} g^{\eta}(y_0; x')\}\}, \qquad \Lambda \in (0,1)$$

Вариант правдоподобия $\tau^{\hat{x}}(x; y_0) = \gamma \circ g^{\eta}(y_0; x)$ — **нечёткое правдоподобие** равенства $x = x \in X$ при наблюдении $\eta = y_0$.

## Правдоподобие согласия модели с данными (п. 2.3)

$$\mathrm{Pl}^{\tilde{x}}(\tilde{x} \sim \eta) = 1 - \sup\{\Lambda \in (0,1): \mathrm{Pl}^{\tilde{x}}(x \in \Phi^{-1}(\eta)) = 1\}$$

Чем меньше это значение, тем значительнее наблюдение $\eta$ **свидетельствует против** субъективной модели $(X, \mathcal{P}(X), \mathrm{Pl}^{\tilde{x}}, \mathrm{Bel}^{\tilde{x}})$.

**Границы.** Восстановление определено с точностью до $\Gamma$ (порядок!), поэтому претензии на «точные значения» здесь бессмысленны.


# 10. Комбинирование Субъективных и Эмпирических Данных

> **Зачем.** На практике есть и мнения экспертов, и данные; нужен способ их честно склеивать (используется в примере с двигателем, раздел 16).

## Задача согласованности (Пытьев 2018, п. 2.2)

Пусть $X = \{x_1, \ldots, x_m\}$, а $\tau^{(0)}(x_i)$ и $\tau^{(1)}(x_i)$ — субъективное и эмпирическое распределения правдоподобий.

Поскольку распределения задаются **в разных шкалах**, их нельзя непосредственно сравнивать числами. Можно сравнивать только **упорядоченности** значений.

## Матрица парных сравнений

Каждому распределению $\tau^{(a)}(x_i)$, $i = 1, \ldots, m$, $a = 0, 1$, сопоставляется **матрица парных сравнений** $M(a) = \{m^{(a)}_{kl}\}$:

$$m^{(a)}_{kl} = \begin{cases} 1 & \text{если } \tau^{(a)}_k > \tau^{(a)}_l \\ 0 & \text{если } \tau^{(a)}_k = \tau^{(a)}_l \\ -1 & \text{если } \tau^{(a)}_k < \tau^{(a)}_l \end{cases}$$

## Евклидово расстояние между матрицами

На классе $\mathcal{M}_{m+1}$ всех матриц парных сравнений вводится расстояние:

$$\rho(M', M'') = \left(\sum_{k,l=1}^{m+1} (m'_{kl} - m''_{kl})^2\right)^{1/2}$$

## Оптимальное совместное распределение (задача минимизации)

$$M^* = \arg\min_{M \in \mathcal{M}_{m+1}} \sum_{a=0,1} w_a^2 \rho^2(M(a), M)$$

где $w_a$ — «веса» субъективного и эмпирического распределений.

**Задача эквивалентна** нахождению матрицы парных сравнений, ближайшей к средневзвешенной матрице:

$$\bar{M} = w_0 M(0) + w_1 M(1), \qquad w_0 + w_1 = 1$$

Элементы $M^*$: $m^*_{kl} = \mathrm{sign}(\bar{m}_{kl})$ при $|\bar{m}_{kl}| \geq 1/2$, иначе $0$.

## Критерий согласованности

Если $M^* \notin \mathcal{M}_{m+1}$ (не является матрицей парных сравнений) — уровень противоречия субъективных и эмпирических данных **превышает критический**; матрице $M^*$ доверять не следует.

**Границы.** Комбинирование чувствительно к согласованности источников; противоречивые эксперты требуют отдельной обработки (взвешивание, отбраковка).


In [7]:
-- | Разделы 6-10: независимость, кондиционирование (residuation), комбинирование

sm1 :: SubjModel Int
sm1 = dualConsistent [1,2,3] (\x -> case x of { 1 -> 1.0; 2 -> 0.5; _ -> 0.3 })

sm2 :: SubjModel Bool
sm2 = dualConsistent [True, False] (\y -> if y then 1.0 else 0.4)

demoS610 :: IO ()
demoS610 = do
  putStrLn "=== Razdely 6-10: nezavisimost i kombinirovanie ==="
  putStrLn $ "  tau12(1,True)  = min(1.0,1.0) = " ++ show (plJointDist sm1 sm2 (1, True))
  putStrLn $ "  tau12(2,False) = min(0.5,0.4) = " ++ show (plJointDist sm1 sm2 (2, False))
  -- условное распределение через residuation (раздел 7+)
  let dom = [(z1, z2) | z1 <- [1,2::Int], z2 <- [1,2::Int]]
      tauJ :: (Int, Int) -> Double
      tauJ (1,1) = 1.0
      tauJ (2,1) = 0.6
      tauJ (1,2) = 0.4
      tauJ (2,2) = 0.4
      tauJ _     = 0.0
  putStrLn $ "  tau(1|2) = " ++ show (condTau dom tauJ 2 1)
  putStrLn $ "  tau(2|1) = " ++ show (condTau dom tauJ 1 2)
  -- комбинирование субъективного и эмпирического (п. 2.2)
  let subj = [1.0, 0.8, 0.5, 0.2]
      empi = [0.9, 0.7, 0.6, 0.3]
  putStrLn $ "  Rangi kombinirovaniya: " ++ show (combineDistributions subj empi 0.5 0.5)

demoS610

Line 4: Use lambda-case
Found:
\ x
  -> case x of
       1 -> 1.0
       2 -> 0.5
       _ -> 0.3
Why not:
\case
  1 -> 1.0
  2 -> 0.5
  _ -> 0.3

=== Razdely 6-10: nezavisimost i kombinirovanie ===
  tau12(1,True)  = min(1.0,1.0) = 1.0
  tau12(2,False) = min(0.5,0.4) = 0.4
  tau(1|2) = 1.0
  tau(2|1) = 0.6
  Rangi kombinirovaniya: [3.0,2.0,1.0,0.0]

# 11. Энтропия Субъективной Неопределённости (Часть 2)

> **Зачем.** Сколько неопределённости в модели? Аналог шенноновской энтропии в min/max-арифметике даёт меру информативности суждений.

## Шенноновская энтропия как отправная точка (Пытьев 2018, Часть 2, п. 2.1)

Шеннон определил энтропию как **среднюю информативность** случайного исхода испытания $(X, \mathcal{P}(X), \mathrm{Pr})$:

$$H(\mathrm{pr}_{\cdot}) = \sum_{i=1}^k \mathrm{pr}_i \log_a \frac{1}{\mathrm{pr}_i}$$

Связь с законом больших чисел (ЗБЧ): число «типичных последовательностей» длины $n$ растёт как $a^{nH}$.

## Информативность и неопределённость НОЭ (п. 2.2)

Для НОЭ $\tilde{x}$ с распределениями $\tau^{\tilde{x}}$ и $\bar{\tau}^{\tilde{x}}$ Пытьев определяет пару энтропий — формальных аналогов шенноновской:

**Информативность** НОЭ $\tilde{x}$ (в первом варианте мер Pl, Bel):

$$\boxed{H(\tau^{\tilde{x}}) = \sup_{x \in X} \min\{\tau^{\tilde{x}}(x),\; \bar{\tau}^{\tilde{x}}(x)\} = \mathrm{pl}_{\tau}(\bar{\tau}(\cdot))}$$

**Неопределённость** НОЭ $\tilde{x}$:

$$\boxed{H(\bar{\tau}^{\tilde{x}}) = \inf_{x \in X} \max\{\bar{\tau}^{\tilde{x}}(x),\; \tau^{\tilde{x}}(x)\} = \mathrm{bel}_{\bar{\tau}}(\tau(\cdot))}$$

Если меры Pl и Bel **дуально согласованы** ($\bar{\tau} = \theta \circ \tau$), то:

$$H_\theta(\tau) = \mathrm{pl}_{\tau}(\theta \circ \tau(\cdot)) = \sup_{x \in X} \min\{\tau(x),\; \theta(\tau(x))\}$$

## Свойства энтропий (формальное подобие шенноновским)

Для независимых НОЭ $\tilde{x}_1$ и $\tilde{x}_2$ (с Pl-независимым совместным распределением):

$$H_\theta(\tau^{\tilde{x}_1} \times \tau^{\tilde{x}_2}) = \max\{H_\theta(\tau^{\tilde{x}_1}),\; H_\theta(\tau^{\tilde{x}_2})\}$$

Независимо повторённое суждение **не несёт дополнительной информации** (в отличие от шенноновской, которая удваивается!). Причина: **отсутствие ЗБЧ** в первом варианте мер.

## Энтропия в третьем варианте (п. 2.3) — со своим ЗБЧ

В третьем варианте мер $\mathrm{Pl}'$ (с $\gamma_\alpha(a) = a^\alpha$):

$$H'(\tau^{\tilde{x}}) = \max_{x \in X} \bigl(\tau(x) \cdot \log_\beta \tau(x)^{-1}\bigr)$$

Здесь есть аналог ЗБЧ, и математическое ожидание субъективной информативности **связано с шенноновской энтропией**:

$$\sum_{i=1}^k \frac{\mathrm{pr}_i}{\mathrm{pr}_1} \cdot \log_a \frac{\mathrm{pr}_i}{\mathrm{pr}_1} = \delta \cdot H(\mathrm{pr}_{\cdot})$$

где $\delta = \lim_{n \to \infty} \delta(n)$ — параметр связи субъективных и вероятностных шкал.

**Границы.** Это формальный аналог: без ЗБЧ интерпретации «числа типичных последовательностей» нет; величина осмыслена как порядковая характеристика.


# 12. Идентификация Состояний НО.НЧ.О. — Оптимальное Правило (Часть 2)

> **Зачем.** Кульминация Части 2: оптимальное правило решения по субъективной модели — аналог байесовского классификатора.

## Задача идентификации (Пытьев 2018, Часть 2, Раздел 3)

**Неопределённый нечёткий объект (НО.НЧ.О.)** находится в состоянии $k \in K = \{1, \ldots, q\}$.  
Его модель $M(x) = (Z, \mathcal{P}(Z), g^{\zeta,\varkappa}(\cdot, \cdot; x))$ зависит от неизвестного параметра $x \in X$.

По наблюдению $\zeta = z \in Z$ требуется принять решение $d(z; x) \in K$ о состоянии объекта.

## Матрица «потерь» (функция возможности потерь)

Субъект, принимающий решения (СПР), задаёт возможность «потерь» $pl_{k,d} \in L$ при истинном состоянии $k$ и принятом решении $d \in K$.

## Оптимальное субъективное правило (Теорема Пытьева)

**Возможность потерь** при правиле $d(\cdot; x): Z \to K$:

$$PL(d(\cdot; x); x) = \sup_{z \in Z} \max_{k \in K} \min\{pl_{k,d(z;x)},\; g^{\zeta,\varkappa}(z, k; x)\}$$

**Оптимальное правило** $d^*(z; x)$ при каждом $x \in X$ минимизирует возможность потерь:

$$d^*(z; x) \in D^*(z; x) = \arg\min_{d \in K} \max_{k \in K} \min\{pl_{k,d},\; g^{\zeta,\varkappa}(z, k; x)\}$$

## Субъективная минимальная возможность потерь

НОЭ $\tilde{x}$ задаёт **субъективную** минимальную возможность потерь $\widetilde{pl}$ с распределением:

$$\tau^{\widetilde{pl}}(p) = \sup\{\tau^{\tilde{x}}(x) \mid x \in X,\; pl(x) = p\}$$

Качество оптимального правила $d^*(\cdot)$ характеризуется значениями:

$$p^* = \arg\max_{p \in L} \tau^{\widetilde{pl}}(p), \qquad \bar{p}^* = \arg\min_{p \in L} \bar{\tau}^{\widetilde{pl}}(p)$$

Чем меньше $p^*$ и $\bar{p}^*$, тем **лучше** оптимальное правило идентификации.

**Границы.** Оптимальность — относительно выбранных мер и функции потерь; смена варианта мер (раздел 8) меняет правило.


In [8]:
-- | Разделы 11-12: энтропии и идентификация — из библиотеки

smE :: SubjModel Int
smE = dualConsistent [1,2,3] (\x -> case x of { 1 -> 1.0; 2 -> 0.7; _ -> 0.3 })

demoS1112 :: IO ()
demoS1112 = do
  putStrLn "=== Razdely 11-12: entropii i identifikaciya ==="
  putStrLn $ "  Informativnost    = " ++ show (subjInformativity smE)
  putStrLn $ "  Neopredelennost   = " ++ show (subjUncertainty smE)
  putStrLn $ "  Dvojnaya entropiya = " ++ show (dualEntropy smE)
  putStrLn $ "  Entropiya 3-go var = " ++ show (thirdVariantEntropy smE)
  let ign = absoluteIgnorance [1,2,3::Int]
  putStrLn $ "  Neznanie: Inf=" ++ show (subjInformativity ign)
             ++ " Neopr=" ++ show (subjUncertainty ign)
  let obs k z = if k == z then 1.0 else 0.3 :: Double
      loss k d = if k == d then 0.0 else 0.8 :: Double
  putStrLn "  Optimalnoe pravilo identifikacii:"
  mapM_ (\(z, d) -> putStrLn $ "    d*(" ++ show z ++ ") = " ++ show d)
        (optimalDecision [1,2] [1,2] obs loss)

demoS1112

Line 4: Use lambda-case
Found:
\ x
  -> case x of
       1 -> 1.0
       2 -> 0.7
       _ -> 0.3
Why not:
\case
  1 -> 1.0
  2 -> 0.7
  _ -> 0.3

=== Razdely 11-12: entropii i identifikaciya ===
  Informativnost    = 0.30000000000000004
  Neopredelennost   = 0.7
  Dvojnaya entropiya = 0.30000000000000004
  Entropiya 3-go var = 0.5210896782498619
  Neznanie: Inf=0.0 Neopr=1.0
  Optimalnoe pravilo identifikacii:
    d*(1) = 1
    d*(2) = 2

# 13. $[0,1]$ как Quantale: Скрытая Топологическая Структура

> **Зачем.** Категорный взгляд: $[0,1]$ с min/⊗ — квантале, и весь аппарат Pl/Bel — обогащённая теория над ним. Это стыкует субъективное моделирование с топосной картиной ([Toposes.ipynb](Toposes.ipynb), T8: строка Sub.Mod. в таблице трёх топосов).

![Quantale structure: три проекции одного объекта](../diagrams/subj/quantale_structure.svg)

## Что такое quantale и почему $[0,1]$ им является

**Quantale** — это замкнутая моноидальная sup-решётка $(Q, \otimes, \mathbf{1})$,
в которой тензорное произведение дистрибутивно над произвольными объединениями.
Если дополнительно $\otimes = \wedge$ (meet), quantale называется **фреймом** (frame).

> **Теорема.** $([0,1], \min, 1)$ является коммутативным идемпотентным quantale (фреймом),
> а значит — **complete Heyting algebra**.

*Доказательство.* Нужно проверить дистрибутивность:
$$\min\bigl(a,\, \sup_{i} b_i\bigr) = \sup_i \min(a, b_i) \quad \forall a \in [0,1], \forall \{b_i\}$$
Это верно для любых вещественных $a, b_i \geq 0$ — стандартный факт о вещественных числах. $\square$

**Внутренний hom (residuation)** quantale $[0,1]$:
$$a \multimap b \;=\; \sup\{c \mid \min(a,c) \leq b\} \;=\; \begin{cases} 1 & a \leq b \\ b & a > b \end{cases}$$

Это именно импликация Хейтинга $a \Rightarrow b$! Таким образом, quantale $[0,1]$ несёт в себе
интуиционистскую логику — и это не случайно.

## Две топологии Скотта: откуда берётся битопос

Каждый частично упорядоченный тип (poset) порождает **топологию Скотта**: открытые множества —
верхние множества (upset), замкнутые относительно супремумов направленных семейств.
На $[0,1]$ есть **два естественных порядка**, каждый даёт свою топологию:

| Порядок | Топология Скотта | Открытые базисные множества | Интерпретация |
|---------|-----------------|----------------------------|---------------|
| $\leq$ (стандартный) | $\mathcal{T}_{\uparrow}$ | $(t, 1]$ для $t \in [0,1]$ | «высокое правдоподобие» |
| $\geq$ (обратный) | $\mathcal{T}_{\downarrow}$ | $[0, t)$ для $t \in [0,1]$ | «низкое доверие» |

Таким образом $([0,1], \mathcal{T}_{\uparrow}, \mathcal{T}_{\downarrow})$ — битопологическое пространство.

## Как $\tau$ индуцирует битопос на $X$

![Топологии Скотта: tau индуцирует битопос](../diagrams/subj/scott_topologies.svg)

Функция $\tau: X \to [0,1]$ **функториально** порождает через прообразы две топологии на $X$:

$$\mathcal{T}_{\uparrow}^X = \{\tau^{-1}(U) \mid U \in \mathcal{T}_{\uparrow}\} = \bigl\{\{x : \tau(x) > t\} \mid t \in [0,1]\bigr\}$$

$$\mathcal{T}_{\downarrow}^X = \{\tau^{-1}(U) \mid U \in \mathcal{T}_{\downarrow}\} = \bigl\{\{x : \tau(x) < t\} \mid t \in [0,1]\bigr\}$$

**Конкретный пример:** $X = \{a,b,c\}$, $\tau(a)=1.0$, $\tau(b)=0.6$, $\tau(c)=0.3$.

Открытые в $\mathcal{T}_{\uparrow}^X$: $\emptyset$, $\{a\}$ (при $t=0.6$), $\{a,b\}$ (при $t=0.3$), $\{a,b,c\}$.

Открытые в $\mathcal{T}_{\downarrow}^X$: $\emptyset$, $\{c\}$ (при $t=0.3$), $\{b,c\}$ (при $t=0.6$), $\{a,b,c\}$.

## Теорема: Pl = Interior, Bel = Closure

> **Теорема.** Для дискретного $X$ и $E \subseteq X$:
> $$\mathrm{Pl}(E) = \sup_{x \in E} \tau(x) = \mathrm{Int}_{\mathcal{T}_{\uparrow}^X}(E)$$
> $$\mathrm{Bel}(E) = \inf_{x \notin E} \bar{\tau}(x) = \mathrm{Cl}_{\mathcal{T}_{\downarrow}^X}(E)$$

*Доказательство для Pl:*
$\mathrm{Int}_{\mathcal{T}_{\uparrow}}(E) = \sup\{t : (t,1] \cap X \subseteq E\}$.
Наибольшее такое $t$ — это $\sup_{x \notin E} \tau(x)$... нет, это
$\sup_{x \in E} \tau(x)$ потому что $(t,1] \cap X \subseteq E$ тогда и только тогда когда
все $x$ с $\tau(x) > t$ лежат в $E$, то есть $t \geq \sup_{x \notin E} \tau(x)$.
В итоге $\mathrm{Int}(E) = \sup_{x \in E} \tau(x)$. $\square$

**Пример:** $E = \{a,b\}$, $\tau(a)=1.0$, $\tau(b)=0.6$, $\tau(c)=0.3$:
$\mathrm{Pl}(\{a,b\}) = \sup(1.0, 0.6) = 1.0$, $\quad \mathrm{Bel}(\{a,b\}) = \inf(\bar{\tau}(c)) = \inf(0.7) = 0.7$.

## Связь с алгеброй Хейтинга: от quantale к логике

Поскольку $[0,1]$ — complete Heyting algebra, **каждая из двух топологий** несёт
свою Heyting/co-Heyting структуру:

$$\mathcal{T}_{\uparrow}^X \text{ (Heyting):} \quad a \Rightarrow b = \begin{cases} 1 & a \leq b \\ b & a > b \end{cases}$$

$$\mathcal{T}_{\downarrow}^X \text{ (co-Heyting/Brouwer):} \quad a \leftarrow b = \begin{cases} 0 & a \geq b \\ b & a < b \end{cases}$$

Вместе $(\mathcal{T}_{\uparrow}^X, \mathcal{T}_{\downarrow}^X)$ образуют **BiHeyting-алгебру** на $[0,1]$.
Это структура, специфичная именно для *битопологических* пространств — и она возникает здесь
не ad hoc, а как следствие того, что quantale $[0,1]$ сам является self-dual объектом
(его порядок изоморфен обратному через отображение $x \mapsto 1-x$).

**Границы.** Обогащение над $[0,1]$ — не элементарный топос (нет классификатора в строгом смысле); это родственная, но другая структура (см. BiTopos в T5/T9).


# 14. Двойственность Исбелла и Гипотеза Единства Подходов

> **Зачем.** Гипотеза единства: Pl/Bel как левое/правое расширения Кана и двойственность Исбелла — общая рамка, связывающая все подходы курса ([KanExtensions.ipynb](KanExtensions.ipynb)).

## Isbell duality: пресноп ↔ копресноп

![Isbell duality над [0,1]-Frm](../diagrams/subj/isbell_duality.svg)

Для малой $\mathcal{V}$-обогащённой категории $\mathcal{C}$ существует каноническое сопряжение
(adjunction) между пресноп и копресноп — **двойственность Исбелла** (Isbell duality, 1966):

$$\bigl(\mathcal{O} \dashv \mathrm{Spec}\bigr)\;:\;
[\mathcal{C},\,\mathcal{V}]^{\mathrm{op}}
\underset{\mathcal{O}}{\overset{\mathrm{Spec}}{\rightleftharpoons}}
[\mathcal{C}^{\mathrm{op}},\,\mathcal{V}]$$

где $\mathcal{O}(X)(c) = [\mathcal{C}^{\mathrm{op}}, \mathcal{V}]\bigl(X,\, \mathcal{C}(-,c)\bigr)$ и
$\mathrm{Spec}(A)(c) = [\mathcal{C}, \mathcal{V}]\bigl(A,\, \mathcal{C}(c,-)\bigr)$.

**Ключевой факт из nLab:** $\mathrm{Spec}$ — это *левое расширение Кана вдоль вложения Йонеды*,
$\mathcal{O}$ — *левое расширение контравариантного вложения Йонеды*.
Двойственность Исбелла — общий шаблон для дуальностей Гельфанда, Стоуна, теоремы Серра-Свана.

## Применение к теории Пытьева: три уровня

Выберем $\mathcal{V} = ([0,1], \min, 1)$ — quantale/фрейм. Три уровня конкретизации:

**Уровень 1 — объекты:**
- $\mathbf{X}$ — дискретная $[0,1]$-обогащённая категория: $\mathbf{X}(x,y) = [x=y]$
- $\tau: \mathbf{X} \to \mathbf{1}$ — $[0,1]$-обогащённый функтор, задающий **пресноп правдоподобия**
- $\bar{\tau}: \mathbf{X} \to \mathbf{1}$ — двойственный **копресноп доверия**
- $J: \mathbf{X} \hookrightarrow \mathcal{P}(X)$ — включение точек в категорию подмножеств

**Уровень 2 — расширения Кана как coend/end:**

Формула левого расширения Кана через coend:
$$\mathrm{Lan}_J\,\tau\,(E) = \int^{x \in \mathbf{X}} \mathcal{P}(X)(J(x), E) \otimes_{\min} \tau(x)$$

При дискретном $X$: $\mathcal{P}(X)(J(x), E) = [x \in E] \in \{0,1\}$, тензор $\otimes = \min$, coend = $\sup$:

$$\mathrm{Lan}_J\,\tau\,(E) = \sup_{x \in E} \min(1, \tau(x)) = \sup_{x \in E} \tau(x) = \mathrm{Pl}(E)$$

Аналогично, **правое расширение Кана** через end:
$$\mathrm{Ran}_J\,\bar{\tau}\,(E) = \int_{x \in \mathbf{X}} [0,1]\bigl(\mathcal{P}(X)(E, J(x)),\, \bar{\tau}(x)\bigr)$$

**Где была дыра.** Если брать Ran вдоль того же профунктора $[x \in E]$, end даёт $\inf_{x \in E} \bar{\tau}(x)$ — а это **не** $\mathrm{Bel}(E)$. Правильная конструкция: Bel — это правое расширение Кана вдоль **профунктора дополнения** $J^{\complement}(E, x) = [x \notin E]$:

$$\mathrm{Ran}_{J^{\complement}}\,\bar{\tau}\,(E) = \int_{x \in \mathbf{X}} [0,1]\bigl([x \notin E],\, \bar{\tau}(x)\bigr) = \inf_{x \notin E} \bar{\tau}(x) = \mathrm{Bel}(E)\;\square$$

(внутренний hom: $[0,1](1, t) = t$, $[0,1](0, t) = 1$, поэтому end пробегает ровно $x \notin E$).

**Почему именно дополнение.** Дуальная согласованность Пытьева $\mathrm{Bel}(E) = \theta(\mathrm{Pl}(X \setminus E))$ — это утверждение, что инволюция $\theta(t) = 1 - t$ есть **дуальный изоморфизм кванталей** $([0,1], \max, \min) \xrightarrow{\;\sim\;} ([0,1], \min, \max)^{op}$, переводящий sup в inf и $\tau \mapsto \bar{\tau}$. Профунктор дополнения — это в точности образ $J$ под действием $\theta$ на истинностных значениях: $[x \notin E] = \theta_{\{0,1\}}([x \in E])$. Итого пара (Pl, Bel) — это пара (Lan вдоль $J$, Ran вдоль $\theta J$), и «гипотеза единства» сводится к коммутированию $\theta$ с парой расширений Кана — что проверяется на конечных примерах ниже.

**Уровень 3 — числовой пример:**

$X = \{a,b,c\}$, $\tau: a \mapsto 1.0,\, b \mapsto 0.6,\, c \mapsto 0.3$, $E = \{a,b\}$:

| Метод | Формула | Pl(E) | Bel(E) |
|-------|---------|-------|--------|
| Пытьев прямо | $\sup_{x\in E}\tau(x)$, $\inf_{x\notin E}\bar\tau(x)$ | $1.0$ | $0.7$ |
| Топология Скотта | $\mathrm{Int}_{\mathcal{T}_{\uparrow}}(E)$, $\mathrm{Cl}_{\mathcal{T}_{\downarrow}}(E)$ | $1.0$ | $0.7$ |
| Расширения Кана | $\mathrm{Lan}_J\tau(E)$, $\mathrm{Ran}_J\bar\tau(E)$ | $1.0$ | $0.7$ |

## Центральная гипотеза

> **Гипотеза** (к проверке). Пусть $\mathcal{V} = ([0,1], \min, 1)$. Тогда существует
> $\mathcal{V}$-обогащённый функтор $\Phi: \mathbf{Frm}_{[0,1]} \to \mathbf{BiTop}$
> такой, что квадрат
> $$\tau \xrightarrow{\mathrm{Lan}_J} \mathrm{Pl}\; =\; \mathrm{Int}_{\mathcal{T}_{\uparrow}} \xleftarrow{\Phi} (X, \mathcal{T}_{\uparrow}, \mathcal{T}_{\downarrow})$$
> коммутирует, и пара $(\mathrm{Pl}, \mathrm{Bel})$ является образом Isbell adjunction
> $\mathcal{O} \dashv \mathrm{Spec}$ применённого к $\tau$ над $\mathcal{V}$.

## Таблица статусов: что доказано, что открыто

| Утверждение | Статус | Источник |
|-------------|--------|----------|
| $[0,1]$ с $\min$ — commutative quantale (фрейм) | ✅ Доказано | nLab: quantale |
| $[0,1]$ несёт топологии Скотта $\mathcal{T}_{\uparrow}$, $\mathcal{T}_{\downarrow}$ | ✅ Доказано | — |
| $\tau^{-1}(\mathcal{T}_{\uparrow})$ и $\tau^{-1}(\mathcal{T}_{\downarrow})$ — битопос на $X$ | ✅ Доказано | — |
| $\mathrm{Lan}_J\,\tau = \mathrm{Pl}$ на дискретном $X$ | ✅ Доказано | coend = sup |
| $\mathrm{Ran}_J\,\bar{\tau} = \mathrm{Bel}$ на дискретном $X$ | ✅ Доказано | end = inf |
| $\mathrm{Pl}(E) = \mathrm{Int}_{\mathcal{T}_{\uparrow}}(E)$ на дискретном $X$ | ✅ Доказано | теорема выше |
| $\mathrm{Bel}(E) = \mathrm{Cl}_{\mathcal{T}_{\downarrow}}(E)$ на дискретном $X$ | ✅ Доказано | теорема выше |
| Isbell duality существует для любых $\mathcal{V}$-категорий | ✅ | nLab: Isbell duality |
| BiHeyting из $\mathcal{V}$-enrichment, а не только из топоса | ✅ через $a \multimap b$ | quantale hom |
| $\Phi: \mathbf{Frm}_{[0,1]} \to \mathbf{BiTop}$ — полный функтор | ⚓ Гипотеза | — |
| Isbell monad над $[0,1]$ = пара $(\mathrm{Pl}, \mathrm{Bel})$ | ⚓ Гипотеза | — |
| Обобщение на непрерывный $X$ (не дискретный) | ⚓ Открыто | — |

## Почему это важно: место в большой картине

Двойственность Исбелла — это **общий шаблон** дуальностей «геометрия ↔ алгебра» в математике.
Если гипотеза верна, теория Пытьева вписывается в один ряд:

| Дуальность | Геометрическая сторона | Алгебраическая сторона |
|------------|----------------------|------------------------|
| Stone duality | топологические пространства | булевы алгебры |
| Gelfand duality | компактные хаусдорфовы пространства | $C^*$-алгебры |
| Locale duality | локали | фреймы |
| **Пытьев (гипотеза)** | **битопос $(X, \mathcal{T}_{\uparrow}, \mathcal{T}_{\downarrow})$** | **пресноп $\tau: X \to [0,1]$** |

Общий объект, унифицирующий все строчки: **Isbell adjunction над соответствующим $\mathcal{V}$**.

**Границы.** Это исследовательская гипотеза, а не теорема из Пытьева; статус утверждений различается (часть доказана в примерах раздела 15, часть — программа).

In [9]:
-- | Раздел 14+: Bel = Ran вдоль профунктора дополнения (библиотека: lanAlong/ranAlong)

demoBelKan :: IO ()
demoBelKan = do
  putStrLn "=== Razdel 14+: Bel cherez Ran vdol dopolneniya ==="
  let dom = "abc"
      tau c = case c of { 'a' -> 1.0; 'b' -> 0.6; _ -> 0.2 }
      tauBar = (1 -) . tau
      subs = filterM (const [True, False]) dom
      pl  e = unUI (plMeasure dom (ui . tau) e)
      bel e = unUI (belMeasure dom (ui . tauBar) e)
      chkDual = all (\e -> ui (bel e) =~ theta (ui (pl (filter (`notElem` e) dom)))) subs
  putStrLn $ "  Bel(E) = theta(Pl(X\\E)) na vseh 8 podmnozhestvah: " ++ show chkDual
  putStrLn $ "  theta - dualnyj izomorfizm (max<->min): " ++ show (isDualIso theta)
  mapM_ (\e -> putStrLn $ "    E=" ++ show e ++ "  Pl=" ++ show (pl e)
               ++ "  Bel=" ++ show (bel e)) subs

demoBelKan

=== Razdel 14+: Bel cherez Ran vdol dopolneniya ===
  Bel(E) = theta(Pl(X\E)) na vseh 8 podmnozhestvah: True
  theta - dualnyj izomorfizm (max<->min): True
    E="abc"  Pl=1.0  Bel=1.0
    E="ab"  Pl=1.0  Bel=0.8
    E="ac"  Pl=1.0  Bel=0.4
    E="a"  Pl=1.0  Bel=0.4
    E="bc"  Pl=0.6  Bel=0.0
    E="b"  Pl=0.6  Bel=0.0
    E="c"  Pl=0.2  Bel=0.0
    E=""  Pl=0.0  Bel=0.0

In [10]:
-- | Разделы 13-14: квантале [0,1], топологии Скотта, Lan/Ran — из библиотеки

demoS1314 :: IO ()
demoS1314 = do
  putStrLn "=== Razdely 13-14: quantale, Scott, Kan ==="
  -- законы квантале: одна полиморфная проверка для [0,1] и Bool
  let gridS = [ui 0, ui 0.25, ui 0.5, ui 0.75, ui 1]
  putStrLn $ "  Residuation adj [0,1]: " ++ show (checkResiduationAdj gridS)
  putStrLn $ "  Residuation adj Bool:  " ++ show (checkResiduationAdj [False, True])
  putStrLn $ "  Frame distributivity:  " ++ show (checkFrameDistributivity gridS)
  -- топологии Скотта, индуцированные tau
  let tau c = case c of { 'a' -> 1.0; 'b' -> 0.6; _ -> 0.2 }
  putStrLn $ "  T_up   (t=0.5): " ++ show (scottUpOnX tau 0.5 "abc")
  putStrLn $ "  T_down (t=0.5): " ++ show (scottDownOnX tau 0.5 "abc")
  -- Lan вдоль принадлежности = Pl; Ran вдоль дополнения = Bel
  let dom = "abc"
  putStrLn $ "  Lan{a,b} = Pl  = " ++ show (unUI (plMeasure dom (ui . tau) "ab"))
  putStrLn $ "  Ran{a,b} = Bel = " ++ show (unUI (belMeasure dom (ui . (1 -) . tau) "ab"))

demoS1314

=== Razdely 13-14: quantale, Scott, Kan ===
  Residuation adj [0,1]: True
  Residuation adj Bool:  True
  Frame distributivity:  True
  T_up   (t=0.5): "ab"
  T_down (t=0.5): "c"
  Lan{a,b} = Pl  = 1.0
  Ran{a,b} = Bel = 0.8

# 14.5 Теоремы: Состоятельность $\mathrm{Pl}/\mathrm{Bel}$ и Эквивалентность Конструкций

> **Зачем.** Раздел 14 предъявил $\mathrm{Pl} = \mathrm{Lan}_J\,\tau$ и $\mathrm{Bel} = \mathrm{Ran}_{J^{\complement}}\,\bar\tau$ и назвал единство подходов «гипотезой». Здесь — доказательства: обе конструкции характеризуются **универсальными свойствами** (состоятельность: аксиомы Пытьева выводятся, а не постулируются), а пара $(\mathrm{Pl}, \mathrm{Bel})$ — **одна** конструкция, $\theta$-сопряжённая сама себе (эквивалентность). Машинная верификация T1–T4 — в ячейке ниже.

**Постановка.** $(q, \le, \bigvee, \bigwedge, \otimes, 1, \multimap)$ — коммутативная единичная кванталь: полная решётка, $\otimes$ сохраняет joins поаргументно, residuation $a\otimes c \le b \iff c \le a\multimap b$ (класс `Quantale`; закон — `checkResiduationAdj`). $X$ конечно и дискретно, $\tau,\bar\tau: X \to q$. Профункторы $J(E,x) = [x\in E]$, $J^{\complement}(E,x) = [x\notin E]$ со значениями $\{\bot, 1\}$ (`memberProf`, `complementProf`).

**Лемма 0** (следствия residuation): $1\otimes a = a$; $\;\bot\otimes a = \bot$; $\;1\multimap b = b$; $\;\bot\multimap b = \top$; $\;(\bigvee_i a_i)\multimap b = \bigwedge_i (a_i \multimap b)$. $\square$

## Теорема 1 (состоятельность $\mathrm{Pl} = \mathrm{Lan}_J\,\tau$)

**(a) Вычисление.** $\mathrm{Lan}_J\tau(E) = \bigvee_{x\in E} 1\otimes\tau(x) \;\vee\; \bigvee_{x\notin E}\bot = \bigvee_{x\in E}\tau(x)$ — формула Пытьева (п. 1.1).

**(b) Универсальное свойство.** Для любого монотонного $m: \mathcal P(X)\to q$:
$$\mathrm{Lan}_J\tau \le m \;\iff\; \forall E,x:\ J(E,x)\otimes\tau(x)\le m(E) \;\iff\; \forall x:\ \tau(x)\le m(\{x\})$$
(определение join $\Rightarrow$ residuation $\Rightarrow$ монотонность: $\bigwedge_{E\ni x} m(E) = m(\{x\})$). То есть $\mathrm{Pl}$ — **наименьшее** монотонное продолжение $\tau$ с синглетонов; это сопряжение $\mathrm{Lan}_J \dashv (J\multimap -)$.

**(c) Аксиомы выводятся + единственность.** $J(\bigcup_j E_j, x) = \bigvee_j J(E_j,x)$ и дистрибутивность $\otimes$ дают **макситивность** $\mathrm{Pl}(\bigcup_j E_j) = \bigvee_j \mathrm{Pl}(E_j)$ — полная аддитивность в $L$ есть сохранение копределов левым сопряжённым. Обратно, любое join-сохраняющее $m$ с $m(\{x\}) = \tau(x)$ обязано совпасть: $m(E) = m(\bigcup_{x\in E}\{x\}) = \bigvee_{x\in E}\tau(x)$. Категорно: $(\mathcal P(X), \subseteq)$ — свободное sup-пополнение дискретного $X$, и $\mathrm{Pl}$ — единственное коконтинуальное продолжение. $\square$

## Теорема 2 (состоятельность $\mathrm{Bel} = \mathrm{Ran}_{J^{\complement}}\,\bar\tau$) — двойственно

**(a)** $\mathrm{Ran}_{J^{\complement}}\bar\tau(E) = \bigwedge_{x\notin E}(1\multimap\bar\tau(x)) \wedge \bigwedge_{x\in E}(\bot\multimap\bar\tau(x)) = \bigwedge_{x\notin E}\bar\tau(x)$ — профунктор дополнения работает именно из-за $\bot\multimap b = \top$: end пробегает ровно $x\notin E$.

**(b)** $m \le \mathrm{Ran}_{J^{\complement}}\bar\tau \iff \forall x: \bigvee_{E\not\ni x} m(E) \le \bar\tau(x) \iff m(X{\setminus}\{x\}) \le \bar\tau(x)$: $\mathrm{Bel}$ — **наибольшее** монотонное $m$ под $\bar\tau$ на ко-синглетонах, с равенством $\mathrm{Bel}(X{\setminus}\{x\}) = \bar\tau(x)$.

**(c)** **Минитивность** из Леммы 0: $J^{\complement}(\bigcap_j E_j, x) = \bigvee_j J^{\complement}(E_j,x)$ и $(\bigvee a)\multimap b = \bigwedge(a\multimap b)$. **Единственность:** $E = \bigcap_{x\notin E}(X{\setminus}\{x\})$, поэтому всякое meet-сохраняющее $m$ с $m(X{\setminus}\{x\}) = \bar\tau(x)$ равно $\mathrm{Bel}$ (и $\mathrm{Bel}(X) = \top$ — пустой meet). $\square$

## Теорема 3 (эквивалентность: $\theta$-сопряжение)

Пусть $\bar\tau = \theta\circ\tau$ (дуальная согласованность), $\theta$ — инволютивный дуальный изоморфизм ($\theta(\bigvee S) = \bigwedge\theta S$, $\theta\circ\theta = \mathrm{id}$; для $[0,1]$: $\theta(t) = 1-t$, `isDualIso`). Тогда
$$\mathrm{Ran}_{J^{\complement}}(\theta\circ\tau) \;=\; \theta\circ\mathrm{Lan}_J\tau\circ\complement.$$

*Доказательство — через единственность (Т2c).* Положим $m := \theta\circ\mathrm{Pl}\circ\complement$. Оно meet-сохраняющее: $m(\bigcap_j E_j) = \theta\bigl(\mathrm{Pl}(\bigcup_j \complement E_j)\bigr) = \theta\bigl(\bigvee_j \mathrm{Pl}(\complement E_j)\bigr) = \bigwedge_j m(E_j)$, и $m(X) = \theta(\mathrm{Pl}(\varnothing)) = \theta(\bot) = \top$. На ко-синглетонах: $m(X{\setminus}\{x\}) = \theta(\mathrm{Pl}(\{x\})) = \theta(\tau(x)) = \bar\tau(x)$. По Т2c такое $m$ единственно, значит $m = \mathrm{Bel}$. $\square$ Инволютивность даёт обратное: $\mathrm{Pl} = \theta\circ\mathrm{Bel}\circ\complement$.

![Квадрат theta-сопряжения Pl/Bel](../diagrams/subj/subj_theta_square.svg)

Пара $(\mathrm{Pl},\mathrm{Bel})$ — одна конструкция, увиденная в $\mathcal C$ и в $\mathcal C^{op}$: профунктор дополнения — это $\theta$-образ профунктора принадлежности на истинностных значениях, $[x\notin E] = \theta_{\{0,1\}}[x\in E]$. Если же $\bar\tau \ne \theta\circ\tau$ (Пытьев допускает такие пары), обе конструкции остаются состоятельными (Т1, Т2), а эквивалентность принимает форму $\mathrm{Bel}_{\bar\tau} = \theta\circ\mathrm{Pl}_{\theta^{-1}\circ\bar\tau}\circ\complement$: эквивалентны **схемы**, равенство для конкретной пары $\iff$ дуальная согласованность.

## Предложение 4 (битопос $\leftrightarrow$ Кан: точная граница)

**Дискретный $X$:** $t < \mathrm{Pl}(E) \iff E\cap U_t \ne \varnothing$, где $U_t = \{\tau > t\}$ — открытые верхней топологии Скотта; на конечном $X$: $\mathrm{Bel}(E)\ge t \iff \complement E \subseteq \{\bar\tau \ge t\}$. Значит $(\mathrm{Pl},\mathrm{Bel})$ — в точности hitting/containment-данные индуцированной битопологии, и обратно $\tau(x) = \mathrm{Pl}(\{x\})$: описания взаимно однозначны (пример 1 ниже — тождественное совпадение).

**Обогащённый $X$ (критерий расхождения):** Кан-сторона идёт вдоль Йонеды и сначала пресноп-ифицирует, $\hat\tau(x) = \bigvee_y \mathrm{hom}(x,y)\otimes\tau(y)$ (`yonedaHat`); битопологическая считает по сырому $\tau$. Всегда $\hat\tau \ge \tau$, и **совпадение $\iff$ $\tau$ — пресноп** ($\mathrm{hom}(x,y)\otimes\tau(y)\le\tau(x)$, `isPresheaf`) $\iff \hat\tau = \tau$. Это точный критерий расхождения из примера 3.

**Границы.** Доказательства выше — для конечного дискретного $X$ над произвольной коммутативной единичной кванталью (над `Bool` та же пара даёт $\mathrm{Pl} = \exists$, $\mathrm{Bel} = \forall$). Для бесконечного $X$ нужны joins/meets по произвольным семействам (в библиотеке — по спискам; см. «идеи расширения» в `Quantale.hs`); дорожная карта бесконечного случая — §14.7. Открытой остаётся функториальная часть гипотезы §14 ($\Phi: \mathbf{Frm}_{[0,1]}\to\mathbf{BiTop}$ на морфизмах).

In [11]:
-- | Раздел 14.5: машинная верификация теорем T1-T4 + два примера.
-- T1b/T2b перебирают ВСЕ монотонные m : P(X) -> {0, 1/2, 1} (фильтр из 3^8 = 6561),
-- T1c/T2c — все join-/meet-сохраняющие продолжения; конечность страхует формализацию.
import Data.List (subsequences, sort)
import Control.Monad (replicateM)

xsP :: String
xsP = "abc"

subsP :: [String]
subsP = map sort (subsequences xsP)

complP :: String -> String
complP e = [c | c <- xsP, c `notElem` e]

keyUP :: String -> String -> String
keyUP e f = sort (nub (e ++ f))

keyIP :: String -> String -> String
keyIP e f = sort [c | c <- e, c `elem` f]

grid3P :: [UnitInterval]
grid3P = map ui [0, 0.5, 1]

tausP :: [Char -> UnitInterval]
tausP = [ \c -> if c == 'a' then ta else if c == 'b' then tb else tc | ta <- grid3P, tb <- grid3P, tc <- grid3P ]

allMsP :: [[(String, UnitInterval)]]
allMsP = map (zip subsP) (replicateM (length subsP) grid3P)

appMP :: [(String, UnitInterval)] -> String -> UnitInterval
appMP m e = head [v | (k, v) <- m, k == sort e]

isMonoP :: [(String, UnitInterval)] -> Bool
isMonoP m = and [ appMP m e <= appMP m f | e <- subsP, f <- subsP, all (`elem` f) e ]

isJoinPresP :: [(String, UnitInterval)] -> Bool
isJoinPresP m = appMP m "" == lbot && and [ appMP m (keyUP e f) == ljoin (appMP m e) (appMP m f) | e <- subsP, f <- subsP ]

isMeetPresP :: [(String, UnitInterval)] -> Bool
isMeetPresP m = appMP m xsP == ltop && and [ appMP m (keyIP e f) == lmeet (appMP m e) (appMP m f) | e <- subsP, f <- subsP ]

monoMsP :: [[(String, UnitInterval)]]
monoMsP = filter isMonoP allMsP

t1a :: Bool
t1a = and [ plMeasure xsP tau e =~ joins [tau x | x <- e] | tau <- tausP, e <- subsP ]

t1b :: Bool
t1b = and [ all (\e -> plMeasure xsP tau e <= appMP m e) subsP == all (\x -> tau x <= appMP m [x]) xsP | tau <- tausP, m <- monoMsP ]

t1c :: Bool
t1c = and [ all (\e -> appMP m e =~ plMeasure xsP tau e) subsP | tau <- tausP, m <- filter isJoinPresP allMsP, all (\x -> appMP m [x] =~ tau x) xsP ]

t2a :: Bool
t2a = and [ belMeasure xsP tb e =~ meets [tb x | x <- complP e] | tb <- tausP, e <- subsP ]

t2b :: Bool
t2b = and [ all (\e -> appMP m e <= belMeasure xsP tb e) subsP == all (\x -> appMP m (complP [x]) <= tb x) xsP | tb <- tausP, m <- monoMsP ]

t2c :: Bool
t2c = and [ all (\e -> appMP m e =~ belMeasure xsP tb e) subsP | tb <- tausP, m <- filter isMeetPresP allMsP, all (\x -> appMP m (complP [x]) =~ tb x) xsP ]

t3P :: Bool
t3P = and [ belMeasure xsP (theta . tau) e =~ theta (plMeasure xsP tau (complP e)) | tau <- tausP, e <- subsP ]

t4P :: Bool
t4P = and [ plMeasure xsP p e == any p e && belMeasure xsP p e == all p (complP e) | p <- [(> 'a'), (< 'c'), const True, const False, (== 'b')], e <- subsP ]

-- Пример 1 (числовой): tau* = {a: 1.0, b: 0.6, c: 0.3}, tauBar = theta . tau*
tauStar :: Char -> UnitInterval
tauStar c = ui (if c == 'a' then 1.0 else if c == 'b' then 0.6 else 0.3)

-- Пример 2 (обогащённый критерий): цепь a <= b <= c, hom x y = [x <= y]
homCh :: Char -> Char -> UnitInterval
homCh x y = if x <= y then ltop else lbot

tauInc :: Char -> UnitInterval
tauInc c = ui (if c == 'a' then 0.2 else if c == 'b' then 0.6 else 1.0)

tauDec :: Char -> UnitInterval
tauDec c = ui (if c == 'a' then 1.0 else if c == 'b' then 0.6 else 0.2)

demoProofs :: IO ()
demoProofs = do
  putStrLn "=== Razdel 14.5: verifikaciya teorem (X = {a,b,c}) ==="
  putStrLn ("  T1a  Lan = sup po E (formula Pytjeva):            " ++ show t1a)
  putStrLn ("  T1b  UP: Pl <= m  <=>  tau <= m na singletonah:   " ++ show t1b)
  putStrLn ("  T1c  edinstvennost join-prodolzheniya:            " ++ show t1c)
  putStrLn ("  T2a  Ran vdol dopolneniya = inf vne E:            " ++ show t2a)
  putStrLn ("  T2b  UP: m <= Bel  <=>  m <= tauBar na ko-singl.: " ++ show t2b)
  putStrLn ("  T2c  edinstvennost meet-prodolzheniya:            " ++ show t2c)
  putStrLn ("  T3   ekvivalentnost: Bel = theta . Pl . compl:    " ++ show t3P)
  putStrLn ("  T4   Bool-kvantal: Pl = exists, Bel = forall:     " ++ show t4P)
  putStrLn ("  residuation (dvizhok T1b/T2b):                    " ++ show (checkResiduationAdj grid3P))
  putStrLn ("  theta - dualnyi izomorfizm:                       " ++ show (isDualIso theta))
  putStrLn ""
  putStrLn "--- Primer 1: tau* = {a:1.0, b:0.6, c:0.3}, tauBar = theta . tau* ---"
  putStrLn "  E      | Pl(E) | Bel(E) | theta(Pl(compl E))"
  mapM_ (\e -> putStrLn ("  " ++ (if null e then "{}" else e) ++ replicate (7 - max 2 (length e)) ' '
           ++ "| " ++ show (unUI (plMeasure xsP tauStar e))
           ++ "   | " ++ show (unUI (belMeasure xsP (theta . tauStar) e))
           ++ "    | " ++ show (unUI (theta (plMeasure xsP tauStar (complP e)))))) subsP
  putStrLn ""
  putStrLn "--- Primer 2: obogashchennyi kriterii (cep a <= b <= c, hom x y = [x <= y]) ---"
  let yInc = yonedaHat homCh xsP tauInc
  let yDec = yonedaHat homCh xsP tauDec
  putStrLn ("  tauInc (vozrastayushchii) presheaf? " ++ show (isPresheaf homCh xsP tauInc)
            ++ "  -> yonedaHat = " ++ show (map (unUI . yInc) xsP) ++ " (dostroen, != tau)")
  putStrLn ("  yonedaHat tauInc presheaf?          " ++ show (isPresheaf homCh xsP yInc))
  putStrLn ("  tauDec (ubyvayushchii) presheaf?    " ++ show (isPresheaf homCh xsP tauDec)
            ++ "  -> nepodvizhen: " ++ show (and [yDec x =~ tauDec x | x <- xsP]))
  putStrLn "  => bitopos i Kan sovpadayut rovno na presnopah (Predlozhenie 4)"

demoProofs

=== Razdel 14.5: verifikaciya teorem (X = {a,b,c}) ===
  T1a  Lan = sup po E (formula Pytjeva):            True
  T1b  UP: Pl <= m  <=>  tau <= m na singletonah:   True
  T1c  edinstvennost join-prodolzheniya:            True
  T2a  Ran vdol dopolneniya = inf vne E:            True
  T2b  UP: m <= Bel  <=>  m <= tauBar na ko-singl.: True
  T2c  edinstvennost meet-prodolzheniya:            True
  T3   ekvivalentnost: Bel = theta . Pl . compl:    True
  T4   Bool-kvantal: Pl = exists, Bel = forall:     True
  residuation (dvizhok T1b/T2b):                    True
  theta - dualnyi izomorfizm:                       True

--- Primer 1: tau* = {a:1.0, b:0.6, c:0.3}, tauBar = theta . tau* ---
  E      | Pl(E) | Bel(E) | theta(Pl(compl E))
  {}     | 0.0   | 0.0    | 0.0
  a     | 1.0   | 0.4    | 0.4
  b     | 0.6   | 0.0    | 0.0
  ab     | 1.0   | 0.7    | 0.7
  c     | 0.3   | 0.0    | 0.0
  ac     | 1.0   | 0.4    | 0.4
  bc     | 0.6   | 0.0    | 0.0
  abc    | 1.0   | 1.0    | 1.0

---

# 14.6 Изоморфизм Конструкций: Битопос ≅ Кан

> **Зачем.** Раздел 14.5 доказал, что обе конструкции состоятельны, а Предложение 4 дало *поточечную* биекцию описаний. Но в теории категорий «эквивалентность конструкций» значит больше: обе конструкции должны быть **функторами**, и между ними должен существовать **(натуральный) изоморфизм** — тогда совпадают не только значения на объектах, но и всё поведение относительно морфизмов. Здесь мы строим такой изоморфизм в сильнейшей форме: изоморфизм *категорий данных* $\Lambda$ с равенством $\Lambda\circ K = B$ **на носу**. Бонус: требование натуральности само вычисляет правильные формулы образов $\tau,\bar\tau$ (п. 1.4) — и вскрывает тонкость в их привычной записи (см. предупреждение в §3).

## Зачем нужны категории данных

Сравнивать конструкции «по значениям» недостаточно: две конструкции могут совпадать на каждом объекте, но по-разному вести себя при переходе вдоль отображений $\varphi: X \to Y$ — тогда это *разные* конструкции, случайно совпавшие на объектах. Категорный стандарт (Mac Lane, гл. I, IV): оформить каждую конструкцию как функтор и предъявить натуральный изоморфизм. Поэтому фиксируем три категории (конечный случай, $\mathcal V = ([0,1], \max, \min)$):

- $\mathbf{SubMod}$ — **субъективные модели**: объекты $(X, \tau)$, конечное $X$ и $\tau: X \to [0,1]$; морфизмы $\varphi: (X,\tau_X) \to (Y,\tau_Y)$ — отображения с $\tau_Y = \mathrm{Lan}_\varphi \tau_X$, где $(\mathrm{Lan}_\varphi\tau)(y) = \sup_{\varphi(x)=y} \tau(x)$ — **проталкивание по слоям**. Заметьте: само проталкивание — это *левое расширение Кана вдоль* $\varphi$ (дискретный случай формулы coend из §14); морфизмы теории тоже оказываются расширениями Кана.
- $\mathbf{MaxMeas}$ — **макситивные меры** (сторона Кана): объекты $(X, m)$, где $m: \mathcal P(X) \to [0,1]$ сохраняет произвольные sup (в терминологии Пытьева — вполне аддитивна в $L$; в терминологии Shilkret 1971 — maxitive); морфизмы $\varphi$ с $m_Y = m_X \circ \varphi^{-1}$ — **образ меры по прообразу**, как у мер возможности у Dubois–Prade.
- $\mathbf{ScottFilt}$ — **фильтрации уровней** (битопологическая сторона): объекты $(X, (U_t)_{t\in[0,1)})$, где семейство $U_t \subseteq X$ антитонно по $t$ и право-непрерывно, $U_t = \bigcup_{s>t} U_s$. Это в точности данные **начальной топологии** отображения $\tau: X \to ([0,1], \mathcal T_\uparrow)$ относительно верхней топологии Скотта (открытые $(t,1]$; Gierz et al., гл. II; Johnstone, гл. VII): $U_t = \tau^{-1}(t,1]$. Морфизмы: $\varphi$ с $V_t = \varphi(U_t)$ (образ множества).

Две конструкции из §14–15 — это два функтора из $\mathbf{SubMod}$:

$$K(X,\tau) = \bigl(X,\ \mathrm{Lan}_J\,\tau\bigr) = (X, \mathrm{Pl}_\tau) \in \mathbf{MaxMeas}, \qquad B(X,\tau) = \bigl(X,\ (\{\tau > t\})_t\bigr) \in \mathbf{ScottFilt}.$$

## Теорема (изоморфизм конструкций)

Существует **изоморфизм категорий** $\Lambda: \mathbf{MaxMeas} \xrightarrow{\ \cong\ } \mathbf{ScottFilt}$,

$$\Lambda(m) = \bigl(\{x : m\{x\} > t\}\bigr)_{t}, \qquad \Lambda^{-1}(U) = \Bigl(E \mapsto \sup\{t : E \cap U_t \neq \varnothing\}\Bigr),$$

причём $\Lambda \circ K = B$ — **равенство функторов** (сильнее натурального изоморфизма: сравнивающая 2-стрелка тождественна).

![Треугольник изоморфизма: Lambda . K = B](../diagrams/subj/subj_iso_triangle.svg)

**Доказательство.**

*Шаг 1: $\Lambda$ корректна и биективна на объектах.* Пусть $m$ макситивна. Уровни синглетонов $U_t = \{x: m\{x\}>t\}$ антитонны и право-непрерывны (строгие неравенства). Hitting-формула восстанавливает $m$: по Теореме 1(c) макситивная мера определяется значениями на синглетонах, поэтому $\sup\{t: E\cap U_t \neq \varnothing\} = \sup_{x\in E} m\{x\} = m(E)$ — это «послойное» (layer-cake) представление, классическое для макситивных мер (Shilkret 1971). Обратно, пусть $(U_t)$ — фильтрация. Hitting-мера $m_U(E) = \sup\{t: E\cap U_t\neq\varnothing\}$ макситивна ($E\cup F$ задевает $U_t$ ⟺ задевает $E$ или $F$, и sup объединения — max двух sup), а её уровни возвращают $(U_t)$ — здесь и только здесь нужна право-непрерывность: $\{x : \sup\{s: x\in U_s\} > t\} = \bigcup_{s>t}U_s = U_t$. Итого объектные части взаимно обратны. $\;\square_1$

*Шаг 2: $\Lambda$ биективна на морфизмах.* Надо: $m_Y = m_X\circ\varphi^{-1} \iff V_t = \varphi(U_t)$. В прямую сторону: $V_t = \{y: m_Y\{y\}>t\} = \{y: m_X(\varphi^{-1}\{y\})>t\}$; по макситивности на конечном $X$ sup по слою достигается, значит $m_X(\varphi^{-1}\{y\})>t \iff \exists x\in\varphi^{-1}\{y\}: m_X\{x\}>t \iff y \in \varphi(U_t)$. Обратная сторона симметрична. Значит $\Lambda$ — биекция на объектах и на hom-множествах, то есть **изоморфизм категорий** (не только эквивалентность). $\;\square_2$

*Шаг 3: $\Lambda\circ K = B$ на носу.* Уровни меры $\mathrm{Pl}_\tau$ на синглетонах: $\mathrm{Pl}_\tau\{x\} = \tau(x)$ (Теорема 1b), поэтому $\Lambda(K(X,\tau)) = (\{\tau>t\})_t = B(X,\tau)$ — буквально те же множества. А равенство $K = \Lambda^{-1}\circ B$ — это Предложение 4 (hitting = $\sup_{x\in E}\tau$). $\;\square_3$

*Шаг 4: функториальность обеих сторон = формула п. 1.4.* Для $\varphi: (X,\tau)\to(Y,\mathrm{Lan}_\varphi\tau)$:
$$\mathrm{Pl}_{\mathrm{Lan}_\varphi\tau}(A) = \sup_{y\in A}\,\sup_{\varphi(x)=y}\tau(x) = \sup_{x\in\varphi^{-1}A}\tau(x) = \mathrm{Pl}_\tau(\varphi^{-1}A),$$
то есть аксиома морфизмов $\mathbf{MaxMeas}$ выполняется автоматически — **формула Пытьева п. 1.4 и есть условие натуральности** конструкции. Аналогично на фильтрациях: $\varphi(\{\tau>t\}) = \{\mathrm{Lan}_\varphi\tau > t\}$. $\;\blacksquare$

## Следствия

**1. Bel-сторона и пара (битопология).** Транспорт по $\theta$ (Теорема 3) переносит всё на доверие: $\mathrm{Bel} = \theta\circ\mathrm{Pl}\circ\complement$, а нижняя фильтрация $\theta$-двойника — **та же самая** верхняя фильтрация, прочитанная через $\theta$:
$$\{\theta\circ\tau < t\} = \{\tau > \theta(t)\} = U_{\theta(t)}(\tau).$$
Битопология пары $(\tau, \theta\tau)$ — одна фильтрация с двух концов, ровно как пара $(\mathrm{Lan}, \mathrm{Ran})$ — одна конструкция в $\mathcal C$ и $\mathcal C^{op}$. Правильный образ ко-синглетонного распределения доверия — $\bar\tau_Y = \mathrm{Ran}_\varphi\bar\tau$, $(\mathrm{Ran}_\varphi\bar\tau)(y) = \inf_{\varphi(x)=y}\bar\tau(x)$ (inf по слою), и тогда $\mathrm{Bel}_Y = \mathrm{Bel}_X\circ\varphi^{-1}$. Пара пушфорвардов $(\mathrm{Lan}_\varphi, \mathrm{Ran}_\varphi)$ — ещё одно появление левого/правого расширений Кана, теперь вдоль $\varphi$, а не вдоль $J$.

**2. $[0,1]$ как амбиморфный (дуализирующий) объект.** Почему вообще существует $\Lambda$? Потому что $[0,1]$ несёт **две согласованные структуры**: sup-решётки (мишень Кан-стороны) и би-Скотт-пространства (мишень битопологической). Открытые верхней топологии $(t,1]$ — это в точности sup-характеры: $\mathrm{Sup}([0,1], \mathbf 2) \cong \{(t,1]\}$, — а значит обе конструкции суть $\mathrm{Hom}(-, [0,1])$, вычисленный через две ипостаси одного объекта. Это тот же механизм «дуализирующего объекта», что порождает двойственности Стоуна и Гельфанда и двойственность Исбелла из §14 (nLab: dualizing object; Johnstone, Stone Spaces).

**3. Свободность как источник всей картины.** $(\mathcal P(X), \subseteq)$ — свободная sup-решётка на дискретном $X$ (Joyal–Tierney, гл. I), поэтому sup-сохраняющие меры $\mathcal P(X)\to[0,1]$ ⟺ произвольные функции $\tau: X\to[0,1]$ (Теорема 1) ⟺ фильтрации уровней (Шаг 1) — три эквивалентных представления одного объекта $[0,1]^X$, и $\Lambda$ — каноническая сшивка второй и третьей биекций.

**Находка (проверка C7 ниже).** Требование натуральности *вычисляет* правильную формулу образа $\bar\tau$ — и показывает, что запись п. 1.4 в §3 (inf по $\varphi(x)\neq y$) отвечает *другому* объекту (доверию к событию $\tilde y = y$), а не ко-синглетонному распределению: подстановка её в $\mathrm{Bel}(A) = \inf_{y\notin A}\bar\tau(y)$ ломается уже при $\varphi = \mathrm{id}$. Подробное предупреждение добавлено в §3; корректный образ — $\mathrm{Ran}_\varphi\bar\tau$.

## Литература

- **Ю.П. Пытьев**, «Математические методы субъективного моделирования…», Вестн. МГУ, Физика. Астрономия, 2018, №1–2 — исходные определения $\tau, \bar\tau, \mathrm{Pl}, \mathrm{Bel}$, формулы образов (п. 1.4), полная аддитивность (п. 1.1).
- **S. Mac Lane**, *Categories for the Working Mathematician*, 2nd ed., Springer 1998 — натуральные преобразования (гл. I), сопряжения (гл. IV), расширения Кана и «все понятия суть расширения Кана» (гл. X).
- **G.M. Kelly**, *Basic Concepts of Enriched Category Theory*, LMS Lecture Notes 64, CUP 1982 — обогащённые категории, взвешенные (ко)пределы, свободные (ко)пополнения (гл. 3–4): формулы coend/end для $\mathrm{Lan}/\mathrm{Ran}$ из §14 — частный случай.
- **F.W. Lawvere**, «Metric spaces, generalized logic, and closed categories», Rend. Sem. Mat. Fis. Milano 43 (1973) — категории, обогащённые над кванталью; $[0,1]$-обогащение как «обобщённая логика».
- **I. Stubbe**, «An introduction to quantaloid-enriched categories», Fuzzy Sets and Systems 256 (2014) — преснопы и расширения Кана над кванталями, ближайшая к нашей библиотеке рамка.
- **K.I. Rosenthal**, *Quantales and Their Applications*, Longman 1990 — квантали, residuation ($a\otimes c \le b \iff c \le a\multimap b$).
- **A. Joyal, M. Tierney**, *An Extension of the Galois Theory of Grothendieck*, Mem. AMS 309 (1984), гл. I — категория $\mathbf{Sup}$; $\mathcal P(X)$ как свободная sup-решётка.
- **N. Shilkret**, «Maxitive measure and integration», Indag. Math. 33 (1971) — макситивные меры, определяемость плотностью/уровнями (наши Шаг 1 и Т1c).
- **D. Dubois, H. Prade**, *Possibility Theory*, Plenum 1988 — меры возможности/необходимости, двойственность $N(A) = 1 - \Pi(\complement A)$, образы мер: $\Pi_Y = \Pi_X\circ\varphi^{-1}$, распределение по слоям.
- **P.T. Johnstone**, *Stone Spaces*, CUP 1982 — двойственность Стоуна, дуализирующие объекты, топология Скотта (гл. II, VII).
- **G. Gierz et al.**, *Continuous Lattices and Domains*, CUP 2003 — топология Скотта на решётках (гл. II).
- **J.C. Kelly**, «Bitopological spaces», Proc. London Math. Soc. 13 (1963) — битопологические пространства как самостоятельный объект.
- **nLab**: статьи *Isbell duality*, *dualizing object*, *free cocompletion* — современная рамка для §14 и данного раздела.

In [12]:
-- | Раздел 14.6: верификация изоморфизма конструкций (C1-C7).
-- C1-C2: Lambda корректна и обратима; C3-C4: натуральность (= п.1.4);
-- C5: битопология theta-двойника; C6: Bel при Ran_phi; C7: контрпример к записи п.1.4.
xsI :: String
xsI = "abc"
ysI :: String
ysI = "uv"

phiI :: Char -> Char
phiI c = if c == 'c' then 'v' else 'u'

subsXI :: [String]
subsXI = map sort (subsequences xsI)
subsYI :: [String]
subsYI = map sort (subsequences ysI)

gridI :: [UnitInterval]
gridI = map ui [0, 0.5, 1]

tausI :: [Char -> UnitInterval]
tausI = [ \c -> if c == 'a' then ta else if c == 'b' then tb else tc | ta <- gridI, tb <- gridI, tc <- gridI ]

testTsI :: [Double]
testTsI = [0, 0.25, 0.5, 0.75]

epsI :: Double
epsI = 1e-9

levelUI :: String -> (Char -> UnitInterval) -> Double -> String
levelUI d tau t = [ x | x <- d, unUI (tau x) > t ]

hitFromI :: String -> (Char -> UnitInterval) -> String -> Double
hitFromI d tau e = maximum (0 : [ v | v <- map (unUI . tau) d, any (`elem` levelUI d tau (v - epsI)) e ])

pushLanI :: (Char -> UnitInterval) -> Char -> UnitInterval
pushLanI tau y = joins [ tau x | x <- xsI, phiI x == y ]

pushRanI :: (Char -> UnitInterval) -> Char -> UnitInterval
pushRanI tb y = meets [ tb x | x <- xsI, phiI x == y ]

preimgI :: String -> String
preimgI a = [ x | x <- xsI, phiI x `elem` a ]

cI1 :: Bool
cI1 = and [ sort (levelUI xsI tau t) == sort [ x | x <- xsI, unUI (plMeasure xsI tau [x]) > t ] | tau <- tausI, t <- testTsI ]

cI2 :: Bool
cI2 = and [ abs (hitFromI xsI tau e - unUI (plMeasure xsI tau e)) < epsI | tau <- tausI, e <- subsXI ]

cI3 :: Bool
cI3 = and [ plMeasure ysI (pushLanI tau) a =~ plMeasure xsI tau (preimgI a) | tau <- tausI, a <- subsYI ]

cI4 :: Bool
cI4 = and [ sort (nub (map phiI (levelUI xsI tau t))) == sort (levelUI ysI (pushLanI tau) t) | tau <- tausI, t <- testTsI ]

cI5 :: Bool
cI5 = and [ sort [ x | x <- xsI, unUI (theta (tau x)) < t ] == sort (levelUI xsI tau (1 - t)) | tau <- tausI, t <- testTsI, t > 0 ]

cI6 :: Bool
cI6 = and [ belMeasure ysI (pushRanI tb) a =~ belMeasure xsI tb (preimgI a) | tb <- tausI, a <- subsYI ]

tbIdI :: Char -> UnitInterval
tbIdI c = ui (if c == 'a' then 0.9 else if c == 'b' then 0.5 else 0.1)

pushP14I :: (Char -> UnitInterval) -> Char -> UnitInterval
pushP14I tb y = meets [ tb x | x <- xsI, x /= y ]

cI7 :: Bool
cI7 = not (and [ belMeasure xsI (pushP14I tbIdI) e =~ belMeasure xsI tbIdI e | e <- subsXI ])

demoIso :: IO ()
demoIso = do
  putStrLn "=== Razdel 14.6: izomorfizm konstruktsii (X={a,b,c}, Y={u,v}, phi: a,b->u, c->v) ==="
  putStrLn ("  C1 urovni Pl na singletonah = filtratsiya tau:        " ++ show cI1)
  putStrLn ("  C2 hitting . levels = id  (Lambda obratima):          " ++ show cI2)
  putStrLn ("  C3 Pl_{Lan_phi tau} = Pl_tau . phi_inv  (natural.):   " ++ show cI3)
  putStrLn ("  C4 phi(U_t(tau)) = U_t(Lan_phi tau)     (natural.):   " ++ show cI4)
  putStrLn ("  C5 nizhnyaya filtratsiya theta.tau = U_{theta t}(tau):" ++ show cI5)
  putStrLn ("  C6 Bel_{Ran_phi tb} = Bel_tb . phi_inv  (natural.):   " ++ show cI6)
  putStrLn ("  C7 zapis p.1.4 (inf po phi(x)/=y) lomaetsya na id:    " ++ show cI7)
  putStrLn ""
  putStrLn "--- Lambda na primere tau* = {a:1.0, b:0.6, c:0.3} ---"
  let tauS c = ui (if c == 'a' then 1.0 else if c == 'b' then 0.6 else 0.3)
  mapM_ (\t -> putStrLn ("  U_" ++ show t ++ " = {tau > " ++ show t ++ "} = " ++ show (levelUI xsI tauS t))) testTsI
  mapM_ (\e -> putStrLn ("  hitting(" ++ (if null e then "{}" else e) ++ ") = "
           ++ show (hitFromI xsI tauS e) ++ "  = Pl = " ++ show (unUI (plMeasure xsI tauS e)))) ["", "b", "bc", "abc"]
  putStrLn ""
  putStrLn "--- Kontrprimer C7 (phi = id, tauBar = {a:0.9, b:0.5, c:0.1}) ---"
  let tb14 = pushP14I tbIdI
  putStrLn ("  zapis p.1.4 dala by tauBar'(a) = min(0.5,0.1) = " ++ show (unUI (tb14 'a')))
  putStrLn ("  Bel_{tauBar'}(bc) = " ++ show (unUI (belMeasure xsI tb14 "bc"))
            ++ "  no dolzhno byt Bel(bc) = tauBar(a) = " ++ show (unUI (belMeasure xsI tbIdI "bc")))
  putStrLn "  korrektnyi obraz: tauBar_Y = Ran_phi tauBar (inf po sloyu), togda Bel_Y = Bel_X . phi_inv (C6)"

demoIso

=== Razdel 14.6: izomorfizm konstruktsii (X={a,b,c}, Y={u,v}, phi: a,b->u, c->v) ===
  C1 urovni Pl na singletonah = filtratsiya tau:        True
  C2 hitting . levels = id  (Lambda obratima):          True
  C3 Pl_{Lan_phi tau} = Pl_tau . phi_inv  (natural.):   True
  C4 phi(U_t(tau)) = U_t(Lan_phi tau)     (natural.):   True
  C5 nizhnyaya filtratsiya theta.tau = U_{theta t}(tau):True
  C6 Bel_{Ran_phi tb} = Bel_tb . phi_inv  (natural.):   True
  C7 zapis p.1.4 (inf po phi(x)/=y) lomaetsya na id:    True

--- Lambda na primere tau* = {a:1.0, b:0.6, c:0.3} ---
  U_0.0 = {tau > 0.0} = "abc"
  U_0.25 = {tau > 0.25} = "abc"
  U_0.5 = {tau > 0.5} = "ab"
  U_0.75 = {tau > 0.75} = "a"
  hitting({}) = 0.0  = Pl = 0.0
  hitting(b) = 0.6  = Pl = 0.6
  hitting(bc) = 0.6  = Pl = 0.6
  hitting(abc) = 1.0  = Pl = 1.0

--- Kontrprimer C7 (phi = id, tauBar = {a:0.9, b:0.5, c:0.1}) ---
  zapis p.1.4 dala by tauBar'(a) = min(0.5,0.1) = 0.1
  Bel_{tauBar'}(bc) = 0.1  no dolzhno byt Bel(bc) = tauBar(a) 

# 14.7 Бесконечный Случай: Заметки и Тропы

> **Статус: рабочие заметки (не теоремы).** Разделы 14.5–14.6 доказаны для конечного дискретного $X$. Здесь — собранная фактура о бесконечном случае и дорожная карта: что переносится дословно, где проходит настоящая граница, и шесть направлений («троп») для развития. Цель — не потерять контекст к моменту, когда возьмёмся всерьёз.

## Как у Пытьева устроен бесконечный случай

В монографии «Возможность как альтернатива вероятности» (Физматлит, 2007/2016) и статьях 2018 г. бесконечность обрабатывается **аксиоматически, а не предельным переходом**:

- $X$ — произвольное множество; меры определены на **всём** $\mathcal P(X)$ — без σ-алгебр и продолжения по Каратеодори. Это возможно потому, что шкала $[0,1]$ — полная решётка: sup/inf существуют для любых семейств.
- **Полная аддитивность** (п. 1.1; §3 выше): $\mathrm{Pl}(\bigcup_{j\in J}E_j) = \sup_{j\in J}\mathrm{Pl}(E_j)$ по **произвольным** (не только счётным!) семействам. Из неё немедленно следует представимость плотностью: $\mathrm{Pl}(E) = \sup_{x\in E}\tau(x)$, где $\tau(x) = \mathrm{Pl}\{x\}$ — у Пытьева плотность существует **по построению**.
- Шкала порядковая (принцип относительности, §4): пределы и арифметика не нужны, только порядок — интегралы §5 (sup-min) переносятся на произвольный $X$ без изменений.

**Чем оплачено** (честные разломы, не баги):
- **Нет непрерывности сверху:** $X=\mathbb N$, $\tau\equiv 1$, $E_n = \{n, n{+}1,\dots\}$: $\mathrm{Pl}(E_n)=1$ для всех $n$, но $\mathrm{Pl}(\bigcap E_n) = \mathrm{Pl}(\varnothing)=0$. Двойственно $\mathrm{Bel}$ не непрерывна снизу. Вероятностная интуиция предельных переходов не работает.
- **Sup может не достигаться:** «наиболее правдоподобной точки» может не быть; задачи идентификации (§12, argmax) требуют $\varepsilon$-оптимизаторов.

## Ключ из литературы: макситивные меры и плотности

Обзор Poncet «Representation of maxitive measures» (Math. Slovaca 2016; arXiv:1405.2238) и работы Akian по идемпотентному анализу дают точную карту:

1. **Вполне макситивная ⟺ существует (кардинальная) плотность.** Аксиома Пытьева — это *в точности* условие представимости плотностью.
2. **σ-макситивная мера может плотности не иметь.** Канонические примеры: косчётная мера на несчётном $X$ ($m(E)=0$ для счётных $E$, иначе $1$ — все точки нулевые, мера нет) и $\operatorname{ess\,sup}$-мера относительно меры Лебега.
3. **Теорема разложения:** всякая макситивная мера $=$ (часть с плотностью) $\vee$ (остаток, нулевой на компактах).
4. **Akian:** плотности для мер на открытых существуют ⟺ решётка значений **непрерывна** (continuous lattice) — доменная теория входит сама.

**Пуанта для Теоремы 14.6.** Кан-конструкция $\mathrm{Lan}_J\tau$ по построению выдаёт меры с плотностью. Значит на бесконечном $X$:

> $\Lambda$ остаётся изоморфизмом — но между $\mathbf{CompMaxMeas}$ (вполне макситивные меры) и фильтрациями. σ-макситивные меры **без** плотности — ровно то, чего Кан-сторона не видит. Аксиома полной аддитивности Пытьева $=$ условие сюръективности Кан-конструкции. Граница эквивалентности найдена, и она содержательная.

При этом большинство доказательств 14.5–14.6 переносится **дословно**: Т1/Т2 используют лишь полноту $[0,1]$ и residuation; натуральность спасают строгие неравенства ($\sup_i a_i > t \iff \exists i:\, a_i > t$ — достижимость sup не нужна); право-непрерывность уровней $\{\tau>t\} = \bigcup_{s>t}\{\tau>s\}$ автоматична. Конечность реально использовалась только в переборных проверках (C1–C7, T1b–T2c).

## Что говорит теория возможностей (математика та же: $\Pi = \mathrm{Pl}$, $N = \mathrm{Bel}$)

Поскольку возможность и правдоподобие построены одинаково, бесконечный случай можно сверять по литературе теории возможностей — а там он проработан детально.

- **Zadeh (1978)** — как у Пытьева: возможность вводится «от плотности» $\pi: X \to [0,1]$ на произвольном универсуме, $\Pi(A) = \sup_{x\in A}\pi(x)$; бесконечность бесплатна по построению.
- **De Cooman (1997–99)** — строгая общая теория: меры возможности = **sup-сохраняющие функции множеств** на **ample fields** — полях, замкнутых относительно *произвольных* объединений и дополнений, над произвольным $X$. Там же: строгое кондиционирование, возможностные переменные и процессы, возможностный аналог теоремы **Даниэля–Колмогорова** (построение процесса из согласованных конечномерных распределений) — но только при **компактных** пространствах состояний и **регулярных** мерах. С Aeyels — семантика верхних вероятностей (imprecise probabilities): мост к «субъективности» Пытьева.
- 💎 **Ample field = CABA.** Замкнутость относительно произвольных $\bigcup$ и дополнений — это определение полной атомарной булевой алгебры. «Ample-пространства» Де Кумана — объекты $\mathbf{Set}^{op} \simeq \mathbf{CABA}$ (см. этюд [SetOp.ipynb](SetOp.ipynb)); плотность живёт на атомах, и обобщение Теоремы 14.6 на ample fields — та же теорема на факторе $X/{\approx}$ по атомам. **Тропа 1 получает готовый классический каркас, состыкованный с SetOp.**
- **Vervaat / Norberg / O'Brien (sup-меры, 1980–90-е)** — тропы 3–4 с аналитическим фундаментом: sup-меры определяются на **открытых** множествах (sup-сохранение по произвольным объединениям открытых), и у **всякой** sup-меры на открытых есть плотность через **sup-производную** $d^\vee m(t) = \inf_{G \ni t} m(G)$ — автоматически полунепрерывную сверху (usc), с восстановлением $m(G) = \sup_{t\in G} d^\vee m(t)$. Значит патологии «мер без плотности» живут только на полном $\mathcal P(X)$; **на фрейме открытых плотность есть всегда** — sup-мера на $\mathcal O(X)$ — это буквально $\mathbf{Sup}$-морфизм из фрейма (бесточечная тропа 3 обоснована классикой).
- **Слияние троп 3 и 5.** Пара «sup-интеграл $i^\vee$ ⊣ sup-производная $d^\vee$» — **сопряжение Галуа** между usc-функциями и sup-мерами, с неподвижными точками = эквивалентность; usc-регуляризация Верваата — то же явление, что `yonedaHat`-оболочка из §14.6: гипотеза «пресноповость = полунепрерывность» из таблицы ниже подтверждается классикой (узор Исбелла $\mathcal O \dashv \mathrm{Spec}$, §14).
- **Цена «процессов»:** компактность + регулярность в возможностном Даниэле–Колмогорове согласуются с ролью непрерывных решёток у Akian (тропа 4): доменная теория — не украшение, а рабочее условие.

### Врезка: sup-производная и сопряжение $i^\vee \dashv d^\vee$

Sup-мера Верваата задана **только на открытых**: $m: \mathcal O(X) \to [0,1]$, $m(\varnothing)=0$, $m(\bigcup_\alpha G_\alpha) = \sup_\alpha m(G_\alpha)$ по произвольным семействам. **Sup-производная** —

$d^\vee m(x) = \inf_{G \ni x,\ G\ \text{откр.}} m(G)$

— инфимум меры по стягивающимся окрестностям точки: идемпотентный аналог производной Радона–Никодима (классически плотность — предел *отношений* по шарам; при $\otimes = \min$ деление вырождается в residuation, и «локальное значение» — просто inf по окрестностям). Двойник — **sup-интеграл** $i^\vee f(G) = \sup_{x\in G} f(x)$, и теорема Верваата $m = i^\vee d^\vee m$ (на открытых) — аналог $\mu(A) = \int_A \frac{d\mu}{d\lambda}\,d\lambda$: у всякой sup-меры на открытых есть плотность.

**Ключ — сопряжение Галуа, доказываемое в две строки** (тем же приёмом, что Т1b в §14.5):

$i^\vee f \le m \iff \forall G\,\forall x \in G: f(x) \le m(G) \iff \forall x: f(x) \le \inf_{G\ni x} m(G) = d^\vee m(x),$

то есть $i^\vee \dashv d^\vee$. Следствия даром: $i^\vee d^\vee m = m$ на открытых (восстановление), а $d^\vee i^\vee f = f^\ast$ — **usc-оболочка** (полунепрерывная сверху регуляризация). Неподвижные точки: usc-функции $\leftrightarrow$ sup-меры — идемпотентное замыкание того же вида, что `yonedaHat` (наименьший пресноп $\ge \tau$) и $\mathcal O \dashv \mathrm{Spec}$ Исбелла из §14.

**Пример руками.** $X = \mathbb R$, кандидат в плотности $f = \chi_{(0,1)}$ (индикатор открытого интервала, lsc). Sup-мера: $m(G)=1 \iff G$ задевает $(0,1)$. Тогда $d^\vee m(x) = 1$ для $x \in [0,1]$ (**замыкание**: любая окрестность задевает интервал) и $0$ вне — производная «дожала» плотность до usc-оболочки $\chi_{[0,1]}$. Обе плотности дают одну и ту же sup-меру на открытых, а расходятся ровно на **границе** $\partial = \{0,1\}$ — той самой ко-Гейтинговой границе $\partial a = a \wedge {\sim}a$ (см. [SetOp.ipynb](SetOp.ipynb)): место, где информация о плотности теряется на уровне открытых. Канонический (наибольший) представитель — usc, его и выбирает $d^\vee$.

![sup-производная: lsc-плотность и её usc-оболочка](../diagrams/subj/subj_sup_derivative.svg)

**Почему это не противоречит «мерам без плотности».** Косчётная и $\operatorname{ess\,sup}$-меры не имели плотности на **полном** $\mathcal P(X)$. На открытых плотность есть всегда: например, для косчётной меры на $\mathbb R$ всякое непустое открытое несчётно, $m(G) = 1$, и $d^\vee m \equiv 1$ честно восстанавливает её *на открытых* — но врёт на синглетонах ($m\{x\} = 0 \ne 1$). Точная формулировка: **на фрейме $\mathcal O(X)$ плотность бесплатна (Верваат); дорого — согласованно продолжить меру с открытых на все подмножества, и это ровно полная макситивность (аксиома Пытьева).** Патология живёт в зазоре между $\mathcal O(X)$ и $\mathcal P(X)$ — ещё один аргумент за бесточечную тропу 3: если носитель информации — фрейм, «мер без плотности» не бывает в принципе.

## Шесть троп

| # | Тропа | Идея | Риск / приз |
|---|-------|------|-------------|
| 1 | **Консервативная** | Теорема 14.6 для произвольного $X$: $\Lambda: \mathbf{CompMaxMeas}\cong\mathbf{Filt}$, «аксиома Пытьева = существование плотности = образ Кана» (опора — Poncet) | почти готова; замыкает «Границы» 14.5–14.6 |
| 2 | **σ-зазор** | Меры без плотности как «диффузная возможность»: $\operatorname{ess\,sup}$ — это sup в решётке классов mod-нуль ⇒ гипотеза: σ-меры = Кан-конструкция в **фактор-топосе по идеалу нулевых множеств**; категорный смысл «остатка» из теоремы разложения? | исследовательская; приз — новая математика |
| 3 | **Бесточечная** | Первичен не $X$, а фильтрация $U: [0,1) \to \mathbf{Frm}$: hitting-формула бесточечна. Правильный объект — **d-frames** (Jung–Moshier, битопологические локали): «битопос» наконец оправдает имя | средняя; стыкуется с Locale-строкой парада двойственностей; **ядро выполнено — §14.9; 3b (дискретный этаж) — §14.10** |
| 4 | **Топологическая / доменная** | $X$ Polish/domain: $\{\tau>t\}$ открыты ⟺ $\tau$ полунепрерывна снизу; битопология = пара (lsc, usc); мост к ёмкостям Шоке, random sup-measures (Vervaat, Norberg), большим уклонениям и **идемпотентному анализу Маслова** | приз — связь со «взрослой» аналитикой |
| 5 | **Обогащённая** | На метрическом $X$ (Ловер) $\widehat{\tau} = $ `yonedaHat` — липшицева регуляризация (оболочка Паша–Хаусдорфа / МакШейна) ⇒ гипотеза: **пресноповость = полунепрерывность**, расхождение битопос/Кан = регуляризация | соединяет 3+4 с критерием преснопа из 14.6 |
| 6 | **Монадная** | §17 на бесконечном $X$: монада возможности = fuzzy powerset / free $\mathbf{Sup}$-module; идемпотентный двойник монады Гири, измеримость не нужна | §17 дорастает до бесконечности |

**Вычислительная заметка (к тропе 1).** Бесконечный случай проверяем в Haskell: представить $\tau$ на счётном $X$ как «конечное возмущение константы» (конечная карта + default-значение хвоста) — тогда все sup/inf по бесконечному носителю вычисляются точно, и $\Lambda$/натуральность верифицируемы на настоящем бесконечном $X$. В библиотеке для этого нужен класс полных решёток с sup по представимым семействам (см. «идеи расширения» в `Quantale.hs`).

**Первые шаги:** (а) сверить по монографии 2016 г., как именно введена полная аддитивность на бесконечном $X$ — аксиома или следствие, и есть ли у Пытьева примеры мер без плотности; (б) мини-этюд «finite support + default» — **выполнено (§14.8)**; (в) черновик доказательства тропы 1 — **выполнено, теорема в §14.8**.

## Литература к разделу

- **Ю.П. Пытьев**, *Возможность как альтернатива вероятности. Математические и эмпирические основы, применение*, Физматлит, 2-е изд. 2016 — бесконечный $X$, полная аддитивность, интегралы.
- **P. Poncet**, «Representation of maxitive measures: an overview», Mathematica Slovaca 66 (2016); arXiv:1405.2238 — плотности, разложение, полный обзор.
- **N. Shilkret**, «Maxitive measure and integration», Indag. Math. 33 (1971) — исходное определение.
- **M. Akian**, «Densities of idempotent measures and large deviations», Trans. AMS 351 (1999) — плотности и непрерывные решётки; идемпотентный анализ.
- **W. Vervaat, T. Norberg** — random sup-measures (extreme value theory); **А. Puhalskii** — большие уклонения как идемпотентная вероятность; **В.П. Маслов, Г.Л. Литвинов** — идемпотентная математика.
- **A. Jung, M.A. Moshier**, «On the bitopological nature of Stone duality» (TR, Birmingham 2006) — d-frames, битопологические локали (тропа 3).
- **L.A. Zadeh**, «Fuzzy sets as a basis for a theory of possibility», Fuzzy Sets and Systems 1 (1978) — возможность «от плотности» на произвольном универсуме.
- **G. de Cooman**, «Possibility theory I–III», Int. J. General Systems 25 (1997); **G. de Cooman, D. Aeyels**, «Supremum preserving upper probabilities», Information Sciences 118 (1999); «A Daniell–Kolmogorov theorem for supremum preserving upper probabilities», Fuzzy Sets and Systems (1999); «Ample fields as a basis for possibilistic processes», Fuzzy Sets and Systems (1999) — sup-сохраняющие меры на ample fields (= CABA), процессы, верхние вероятности.
- **F.W. Lawvere** (1973) — метрические пространства как $[0,1]$-категории (тропа 5).

# 14.8 Бесконечный Случай: Теорема (тропа 1 выполнена)

> **Зачем.** Закрываем консервативную тропу из §14.7: изоморфизм конструкций (§14.6) обобщается на **произвольный** $X$ — конечность нигде не была нужна по существу. Весь бесконечный случай держится на трёх маленьких леммах; остальное переносится из §14.5–14.6 дословно. Плюс следствие про ample fields Де Кумана (= CABA) и исполнимый этюд на бесконечном $X = \mathbb N$ с *точной* арифметикой sup/inf.

**Категории** — те же, что в §14.6, но без конечности:
- $\mathbf{SubMod}$: $(X, \tau)$, $\tau: X \to [0,1]$, произвольное $X$; морфизмы $\varphi$ с $\tau_Y = \mathrm{Lan}_\varphi\tau_X = \bigl(y \mapsto \sup_{\varphi(x)=y}\tau(x)\bigr)$.
- $\mathbf{CompMaxMeas}$: $(X, m)$, $m: \mathcal P(X)\to[0,1]$ **вполне макситивна** — $m(\bigcup_{j\in J}E_j) = \sup_{j\in J}m(E_j)$ по *произвольным* семействам (аксиома Пытьева, п. 1.1); морфизмы $\varphi$ с $m_Y = m_X\circ\varphi^{-1}$.
- $\mathbf{Filt}$: $(X, (U_t)_{t\in[0,1)})$, $U_t$ антитонна и право-непрерывна ($U_t = \bigcup_{s>t}U_s$); морфизмы $\varphi$ с $V_t = \varphi(U_t)$.

## Три леммы, на которых всё держится

**Лемма A (полная макситивность = плотность).** Если $m$ вполне макситивна, то $m(E) = m\bigl(\bigcup_{x\in E}\{x\}\bigr) = \sup_{x\in E} m\{x\}$ — плотность $\tau(x) = m\{x\}$ существует **в одну строку**, для любого $X$. $\square$
*Точность границы:* для σ-макситивности это неверно — косчётная мера ($m(E)=0$ для счётных $E$, иначе $1$) σ-макситивна, но $\sup_{x\in E}m\{x\} = 0 \ne 1$: плотности нет. Полная аддитивность Пытьева — не перестраховка, а ровно та сила аксиомы, которая делает Кан-сторону сюръективной.

**Лемма B (принцип строгого уровня).** В $[0,1]$: $\sup S > t \iff \exists s \in S: s > t$ — для *произвольного* $S$, безо всякой достижимости супремума. Следствия:
1. уровни $U_t = \{\tau > t\}$ право-непрерывны: $\{\tau>t\} = \bigcup_{s>t}\{\tau>s\}$;
2. hitting-обращение: $\sup\{t : E \cap U_t \ne \varnothing\} = \sup_{x\in E}\tau(x)$ (обе части — sup по одному и тому же множеству пар $(t,x)$ с $\tau(x)>t$);
3. натуральность на бесконечных слоях: $\{y : \sup_{\varphi(x)=y}\tau(x) > t\} = \varphi(\{\tau > t\})$ — sup по слою $>t$ ⟺ в слое есть точка $>t$. $\square$

**Лемма C ($\theta$ на произвольных семействах).** $\theta(t) = 1-t$ — антиизоморфизм порядка, поэтому $\theta(\sup S) = \inf \theta S$ для произвольных $S$ — транспорт Bel-стороны (Т3) не чувствует бесконечности. $\square$

## Теорема (изоморфизм конструкций, произвольный $X$)

Функторы $K$, $B$, $\Lambda$, $\Lambda^{-1}$ из §14.6 корректно определены на категориях выше, и:
$$\mathbf{SubMod} \;\xrightarrow[\;\cong\;]{K}\; \mathbf{CompMaxMeas} \;\xrightarrow[\;\cong\;]{\Lambda}\; \mathbf{Filt}, \qquad \Lambda\circ K = B\ \text{(на носу)}.$$

**Доказательство.** По шагам §14.6, с точечными заменами:
- *$K$ биективен на объектах*: инъективность — $\mathrm{Pl}_\tau\{x\} = \tau(x)$; сюръективность — **Лемма A** (вот где нужна полная, а не σ-, макситивность).
- *$\Lambda$ корректна и обратима*: право-непрерывность уровней и hitting-обращение — **Лемма B(1,2)**; уровни hitting-меры возвращают фильтрацию — как Шаг 1 §14.6, с B(1) вместо конечности.
- *Соответствие морфизмов и $\Lambda\circ K = B$*: Шаги 2–3 §14.6 дословно, с **Леммой B(3)** вместо «sup по слою достигается».
- *Функториальность = п. 1.4*: выкладка Шага 4 §14.6 — это равенство sup по двум разбиениям одного множества, конечность не использовалась.
- *Универсальные свойства и единственность* (Т1b–c, Т2b–c из §14.5): доказательства используют только полноту решётки $[0,1]$ и residuation; $E = \bigcup_{x\in E}\{x\}$ — произвольное объединение. Дословно. $\blacksquare$

**Bel-сторона.** По **Лемме C** транспорт Т3 работает без изменений: $\mathrm{Bel} = \theta\circ\mathrm{Pl}\circ\complement$, нижняя фильтрация $\{\theta\tau < t\} = U_{\theta(t)}(\tau)$, образы $\bar\tau_Y = \mathrm{Ran}_\varphi\bar\tau$ (inf по слою) дают $\mathrm{Bel}_Y = \mathrm{Bel}_X\circ\varphi^{-1}$.

**Следствие (ample fields Де Кумана).** Пусть $\mathfrak A \subseteq \mathcal P(X)$ — ample field (замкнуто относительно произвольных $\bigcup$ и дополнений). Положим $x \sim y \iff \forall A\in\mathfrak A\,(x\in A \Leftrightarrow y\in A)$; атомы $\mathfrak A$ — классы $\sim$, каждый $A \in\mathfrak A$ — объединение атомов, т.е. $\mathfrak A \cong \mathcal P(X/{\sim})$ — CABA (ср. [SetOp.ipynb](SetOp.ipynb)). Sup-сохраняющие меры на $(X,\mathfrak A)$ — в точности $\mathbf{CompMaxMeas}$ на $X/{\sim}$, и теорема применяется к атомам: **вся теория возможностей Де Кумана на ample-пространствах — частный случай теоремы через фактор по атомам.**

**Что по-прежнему не выполняется (и не нужно):** непрерывности сверху нет ($E_n = \{n, n{+}1, \dots\}$: $\mathrm{Pl}(E_n) = 1$, $\mathrm{Pl}(\bigcap E_n) = 0$) — ни один шаг доказательства её не требует. σ-зазор (меры без плотности) остаётся содержанием тропы 2.

## Этюд: бесконечный $X = \mathbb N$ с точной арифметикой

Идея из §14.7: чтобы **вычислять** sup/inf по бесконечному носителю точно, берём представления с конечным описанием:
- **распределения** — «конечные поправки + default»: $\tau = (d, \{(n_i, v_i)\})$, вне поправок $\tau \equiv d$;
- **события** — конечно-коконечная алгебра $\mathrm{FC}(\mathbb N)$ (конечные множества и дополнения конечных): она замкнута относительно $\cap, \cup, \complement$ — дополнение **точное**, что делает вычислимой и Bel-сторону;
- **уровни** $\{\tau > t\}$ снова конечны или коконечны — $\Lambda$ вычислима в обе стороны;
- вторая семья $\tau_n = 1 - \frac{1}{n+2}$ — **sup не достигается**: проверяем, что строгие уровни (Лемма B) всё равно дают $\mathrm{Pl}(\mathbb N) = 1$;
- морфизм $\varphi$ с бесконечным слоем ($\varphi(n) = [n \ge 5]$: слой $\{5,6,\dots\}$ коконечен) — натуральность п. 1.4 на бесконечных слоях.

Проверки I1–I6 в ячейке ниже — все на *настоящем* бесконечном $X$, без усечений.

In [13]:
-- | Раздел 14.8: этюд — бесконечный X = N, точные sup/inf через конечные описания.
-- Распределение: default d + конечные поправки; события: конечно-коконечная алгебра.
data FSD = FSD Double [(Int, Double)]

tauAtF :: FSD -> Int -> Double
tauAtF (FSD d o) n = fromMaybe d (lookup n o)

thetaF :: FSD -> FSD
thetaF (FSD d o) = FSD (1 - d) [ (n, 1 - v) | (n, v) <- o ]

data FCSet = Fin [Int] | Cofin [Int] deriving Show

memF :: Int -> FCSet -> Bool
memF n (Fin xs) = n `elem` xs
memF n (Cofin xs) = n `notElem` xs

complFC :: FCSet -> FCSet
complFC (Fin xs) = Cofin xs
complFC (Cofin xs) = Fin xs

emptyFC :: FCSet -> Bool
emptyFC (Fin xs) = null xs
emptyFC (Cofin _) = False

meetsFC :: FCSet -> FCSet -> Bool
meetsFC (Fin xs) s = any (`memF` s) xs
meetsFC (Cofin _) (Cofin _) = True
meetsFC s (Fin xs) = any (`memF` s) xs

plFC :: FSD -> FCSet -> Double
plFC t (Fin xs) = maximum (0 : map (tauAtF t) xs)
plFC (FSD d o) (Cofin xs) = maximum (d : [ v | (n, v) <- o, n `notElem` xs ])

infFC :: FSD -> FCSet -> Double
infFC t (Fin xs) = minimum (1 : map (tauAtF t) xs)
infFC (FSD d o) (Cofin xs) = minimum (d : [ v | (n, v) <- o, n `notElem` xs ])

belFC :: FSD -> FCSet -> Double
belFC tb e = infFC tb (complFC e)

levelFC :: FSD -> Double -> FCSet
levelFC (FSD d o) t = if d > t then Cofin [ n | (n, v) <- o, v <= t ] else Fin [ n | (n, v) <- o, v > t ]

hitFC :: FSD -> FCSet -> Double
hitFC t@(FSD d o) e = maximum (0 : [ v | v <- d : map snd o, meetsFC e (levelFC t (v - 1e-9)) ])

tau1F :: FSD
tau1F = FSD 0.4 [(0, 1.0), (1, 0.7), (7, 0.1)]

samplesF :: [FCSet]
samplesF = [Fin [], Fin [1, 7], Fin [2, 3], Cofin [], Cofin [0], Cofin [0, 1, 2, 7]]

iA :: Bool
iA = and [ abs (plFC tau1F (Fin [n]) - tauAtF tau1F n) < 1e-9 | n <- [0, 1, 2, 7, 100] ]

iB :: Bool
iB = and [ abs (hitFC tau1F e - plFC tau1F e) < 1e-9 | e <- samplesF ]

iC :: Bool
iC = and [ abs (belFC (thetaF tau1F) e - (1 - plFC tau1F (complFC e))) < 1e-9 | e <- samplesF ]

preimF :: [Int] -> FCSet
preimF a = case (0 `elem` a, 1 `elem` a) of
  (False, False) -> Fin []
  (True, False) -> Fin [0 .. 4]
  (False, True) -> Cofin [0 .. 4]
  (True, True) -> Cofin []

pushLanF :: FSD -> Int -> Double
pushLanF t y = if y == 0 then plFC t (Fin [0 .. 4]) else plFC t (Cofin [0 .. 4])

iD :: Bool
iD = and [ abs (maximum (0 : map (pushLanF tau1F) a) - plFC tau1F (preimF a)) < 1e-9 | a <- [[], [0], [1], [0, 1]] ]

iE :: Bool
iE = and [ (pushLanF tau1F y > t) == meetsFC (preimF [y]) (levelFC tau1F t) | y <- [0, 1], t <- [0, 0.25, 0.5, 0.75, 0.95] ]

tau2At :: Int -> Double
tau2At n = 1 - 1 / fromIntegral (n + 2)

level2 :: Double -> FCSet
level2 t = if t >= 1 then Fin [] else Cofin [ n | n <- [0 .. bnd], tau2At n <= t ]
  where bnd = max (0 :: Int) (ceiling (1 / (1 - t)))

iF :: Bool
iF = all ((< 1) . tau2At) [0 .. 999] && not (any (emptyFC . level2) [0, 0.9, 0.99, 0.999]) && emptyFC (level2 1)

demoInf :: IO ()
demoInf = do
  putStrLn "=== Razdel 14.8: beskonechnyi X = N, tochnye sup/inf ==="
  putStrLn ("  iA plotnost = singletony (Lemma A):               " ++ show iA)
  putStrLn ("  iB hitting . levels = Pl na beskonechnyh E:       " ++ show iB)
  putStrLn ("  iC Bel_{theta tau} = theta . Pl . compl:          " ++ show iC)
  putStrLn ("  iD naturalnost p.1.4 (sloi {5,6,..} beskonechen): " ++ show iD)
  putStrLn ("  iE phi(U_t) = U_t(Lan_phi tau)  (Lemma B3):       " ++ show iE)
  putStrLn ("  iF sup ne dostigaetsya, urovni nepusty pri t<1:   " ++ show iF)
  putStrLn ""
  putStrLn "--- tau1: default 0.4, popravki {0: 1.0, 1: 0.7, 7: 0.1} ---"
  mapM_ (\e -> putStrLn ("  " ++ show e ++ "   Pl=" ++ show (plFC tau1F e)
           ++ "  Bel=" ++ show (belFC (thetaF tau1F) e)
           ++ "  hit=" ++ show (hitFC tau1F e))) samplesF
  putStrLn ""
  putStrLn "--- tau2(n) = 1 - 1/(n+2): sup = 1 NE dostigaetsya ---"
  putStrLn ("  uroven {tau2 > 0.9} = " ++ show (level2 0.9) ++ " (kokonechen)")
  putStrLn "  strogie urovni nepusty dlya vseh t < 1  =>  hitting-sup = 1 = Pl(N), bez dostizhimosti"

demoInf

=== Razdel 14.8: beskonechnyi X = N, tochnye sup/inf ===
  iA plotnost = singletony (Lemma A):               True
  iB hitting . levels = Pl na beskonechnyh E:       True
  iC Bel_{theta tau} = theta . Pl . compl:          True
  iD naturalnost p.1.4 (sloi {5,6,..} beskonechen): True
  iE phi(U_t) = U_t(Lan_phi tau)  (Lemma B3):       True
  iF sup ne dostigaetsya, urovni nepusty pri t<1:   True

--- tau1: default 0.4, popravki {0: 1.0, 1: 0.7, 7: 0.1} ---
  Fin []   Pl=0.0  Bel=0.0  hit=0.0
  Fin [1,7]   Pl=0.7  Bel=0.0  hit=0.7
  Fin [2,3]   Pl=0.4  Bel=0.0  hit=0.4
  Cofin []   Pl=1.0  Bel=1.0  hit=1.0
  Cofin [0]   Pl=0.7  Bel=0.0  hit=0.7
  Cofin [0,1,2,7]   Pl=0.4  Bel=0.0  hit=0.4

--- tau2(n) = 1 - 1/(n+2): sup = 1 NE dostigaetsya ---
  uroven {tau2 > 0.9} = Cofin [0,1,2,3,4,5,6,7,8] (kokonechen)
  strogie urovni nepusty dlya vseh t < 1  =>  hitting-sup = 1 = Pl(N), bez dostizhimosti

# 14.9 Бесточечная Тропа: Мера = Сопряжение (тропа 3, ядро)

> **Зачем.** Шаг лестницы: §14.6 — дискретный конечный $X$, §14.8 — произвольный $X$, здесь — **точек нет вовсе**. Носитель информации — фрейм (решётка «открытых») $L$, и оказывается, что бесточечная версия $\Lambda$ — это не новая конструкция, а **само сопряжение Галуа**: «у каждого левого сопряжённого есть правый». Плотность без точек — это правый сопряжённый меры. Врезка §14.7 про $i^\vee \dashv d^\vee$ была точечной тенью этой теоремы.

## Теорема (бесточечная $\Lambda$)

Пусть $L$ — полная решётка (в приложениях — фрейм). Существует биекция между:

- **sup-мерами**: $m: L \to [0,1]$, сохраняющими *произвольные* joins (включая пустой: $m(0_L) = 0$);
- **ко-фильтрациями**: $g: [0,1] \to L$, монотонными и сохраняющими *произвольные* meets (включая пустой: $g(1) = 1_L$);

заданная взаимно обратными формулами

$$g(t) = \bigvee\{a \in L : m(a) \le t\}, \qquad m(a) = \inf\{t : a \le g(t)\},$$

и связанная сопряжением $\;m(a) \le t \iff a \le g(t)$.

**Доказательство** (четыре residuation-шага, стиль Т1b/§14.7).

1. *Из $m$ — сопряжение.* $a \le g(t) \Rightarrow m(a) \le m(g(t)) = \sup\{m(b): m(b)\le t\} \le t$; обратно $m(a)\le t \Rightarrow a$ входит в join, определяющий $g(t)$. Итого $m(a)\le t \iff a\le g(t)$. $\square_1$
2. *Восстановление $m$.* По сопряжению $\{t : a \le g(t)\} = [m(a), 1]$ — луч, его inf равен $m(a)$ и **достигается**. $\square_2$
3. *Из $g$ — мера.* Пусть $g$ монотонна и meet-непрерывна; положим $m(a) = \inf\{t: a\le g(t)\}$. Множество $T_a = \{t: a\le g(t)\}$ ↑-замкнуто (монотонность) и замкнуто относительно inf (**meet-непрерывность — бесточечная Лемма B**: она делает inf достижимым), значит $T_a = [m(a),1]$ и сопряжение выполняется. Тогда $m(\bigvee_i a_i) \le s \iff \forall i\, a_i \le g(s) \iff \sup_i m(a_i) \le s$ для всех $s$ — $m$ сохраняет joins. $\square_3$
4. *Биекция* — единственность сопряжённых. $\blacksquare$

**Натуральность бесплатно.** В бесточечном мире «мера по прообразу» — буквально композиция: для морфизма фреймов $h: L' \to L$ (это и есть бесточечный $\varphi^{-1}$) пушфорвард меры — $m \mapsto m \circ h$, а $g$ переносится правым сопряжённым $h_*$: никакой возни со слоями, вся содержательность п. 1.4 растворяется в композиции.

## Лестница: одна конструкция, три этажа

| Этаж | $L$ | Правый сопряжённый $g(t)$ | Точечная тень |
|------|-----|---------------------------|----------------|
| §14.6, 14.8 | $\mathcal P(X)$ | $\{x : \tau(x) \le t\}$ | есть дополнение: $U_t = \neg g(t) = \{\tau > t\}$ — фильтрация $\Lambda$ |
| Верваат (§14.7) | $\mathcal O(X)$ | наибольшее открытое меры $\le t$ | $d^\vee m(x) = \inf\{t : x \in g(t)\}$ — usc-плотность |
| §14.9 | любой фрейм | $\bigvee\{a : m(a)\le t\}$ | точек может не быть; $g$ — «бесточечная плотность» |

В $\mathcal P(X)$ у $g(t)$ есть дополнение — поэтому в §14.6/14.8 мы видели фильтрацию строгих уровней. В общем фрейме дополнений нет, и **носителем плотности становится сам $g$**. Меры «без плотности» (§14.7) — это меры, у которых нет *точечной* плотности; бесточечная ($g$) есть всегда — теорема выше безусловна.

## Почему для пары (Pl, Bel) нужен d-frame (набросок, тропа 3b)

В §14.6 Bel строился как $\theta\circ\mathrm{Pl}\circ\complement$ — через **дополнение**. В общем фрейме дополнения нет (этюд ниже показывает минимальный контрпример), значит внутри одного $L$ пару (Pl, Bel) не собрать. Правильный носитель — **d-frame** Юнга–Мошье: пара фреймов $(L_+, L_-)$ (для двух топологий битопологического пространства) с отношениями $\mathrm{con}$ (совместность) и $\mathrm{tot}$ (тотальность). Гипотеза 3b: пара $(\mathrm{Pl}: L_+ \to [0,1],\ \mathrm{Bel}: L_-\to[0,1]^{op})$ с условием $\mathrm{con} \Rightarrow \mathrm{Bel} \le \mathrm{Pl}$ — это в точности $[\mathrm{Bel}, \mathrm{Pl}]$-интервал билатиса `IV` из библиотеки (`Bitopos.hs`), а $\mathrm{tot}$ отвечает нормировке; базовый пример d-frame — логика Белнапа FOUR (`bTrue/bFalse/bUnknown/bContra`). Проработка — отдельным шагом.

## Этюд: сопряжение перечислением на не-булевом фрейме

Минимальный не-булев фрейм — **цепь Серпинского** $L = \{0 < m < 1\}$ (открытые пространства Серпинского). На нём (и на булевом $\mathcal P(\{x,y\})$ для сравнения) перечислением проверяем:

- **q1**: $m \mapsto g \mapsto m$ — тождество (восстановление по Шагу 2);
- **q2**: биекция — множество всех $\mathrm{radj}(m)$ совпадает со множеством всех монотонных $g$ с $g(1) = 1_L$ (счёт: $10 = 10$);
- **q3**: у среднего элемента **нет дополнения** — внутри $L$ Bel через $\complement$ не определить (мотивация d-frame);
- **q4**: точечная тень $d(x) = \inf\{t : x \in g(t)\}$ спатиально восстанавливает меру: $m(a) = \sup_{x\in a} d(x)$ (Серпинский — пространственный фрейм);
- **q5**: на булевом $\mathcal P(\{x,y\})$ дополнение $\neg g(t)$ — в точности строгий уровень $\{\tau > t\}$: этаж §14.8 — частный случай.

## Литература к разделу

- **A. Jung, M.A. Moshier**, «On the bitopological nature of Stone duality», Tech. Report CSR-06-13, Birmingham 2006 — d-frames, битопологические локали, Белнап.
- **J. Picado, A. Pultr**, *Frames and Locales: Topology without Points*, Birkhäuser 2012 — фреймы, сопряжения, бесточечная топология.
- **P.T. Johnstone**, *Stone Spaces*, CUP 1982, гл. II — фреймы и пространственность.
- **W. Vervaat** — sup-меры и sup-производная (см. §14.7): точечная тень этажа $\mathcal O(X)$.
- **N.D. Belnap**, «A useful four-valued logic» (1977) — FOUR; **M.L. Ginsberg** (1988) — билатисы.

In [14]:
-- | Раздел 14.9: этюд — бесточечная Lambda перечислением (цепь Серпинского + булев P(2)).
gridQ :: [Double]
gridQ = [0, 1/3, 2/3, 1]

carrS :: [Int]
carrS = [0, 1, 2]

measS :: [[(Int, Double)]]
measS = [ [(0, 0), (1, v1), (2, v2)] | v1 <- gridQ, v2 <- gridQ, v1 <= v2 ]

appQ :: [(Int, Double)] -> Int -> Double
appQ mm a = fromMaybe 0 (lookup a mm)

radjS :: [(Int, Double)] -> Double -> Int
radjS mm t = maximum (0 : [ a | a <- carrS, appQ mm a <= t ])

recovS :: [(Int, Double)] -> Int -> Double
recovS mm a = minimum [ t | t <- gridQ, a <= radjS mm t ]

q1 :: Bool
q1 = and [ abs (recovS mm a - appQ mm a) < 1e-9 | mm <- measS, a <- carrS ]

gTabS :: [(Int, Double)] -> [Int]
gTabS mm = map (radjS mm) gridQ

allGS :: [[Int]]
allGS = [ [a0, a1, a2, 2] | a0 <- carrS, a1 <- carrS, a0 <= a1, a2 <- carrS, a1 <= a2 ]

q2 :: Bool
q2 = sort (map gTabS measS) == sort allGS

q3 :: Bool
q3 = null [ b | b <- carrS, min 1 b == 0, max 1 b == 2 ]

memPtS :: Int -> Int -> Bool
memPtS x a = if x == 1 then a >= 1 else a >= 2

dShad :: [(Int, Double)] -> Int -> Double
dShad mm x = minimum [ t | t <- gridQ, memPtS x (radjS mm t) ]

q4 :: Bool
q4 = and [ abs (appQ mm a - maximum (0 : [ dShad mm x | x <- [0, 1], memPtS x a ])) < 1e-9 | mm <- measS, a <- carrS ]

ptsP :: [Int]
ptsP = [0, 1]

subsetsP :: [[Int]]
subsetsP = [[], [0], [1], [0, 1]]

plP :: (Int -> Double) -> [Int] -> Double
plP tf e = maximum (0 : map tf e)

radjP :: (Int -> Double) -> Double -> [Int]
radjP tf t = foldr unionP [] [ e | e <- subsetsP, plP tf e <= t ]
  where unionP xs ys = sort (nub (xs ++ ys))

q5 :: Bool
q5 = and [ sort [ x | x <- ptsP, x `notElem` radjP tf t ] == sort [ x | x <- ptsP, tf x > t ] | v0 <- gridQ, v1 <- gridQ, let tf x = if x == 0 then v0 else v1, t <- gridQ ]

demoQ :: IO ()
demoQ = do
  putStrLn "=== Razdel 14.9: bestochechnaya Lambda = sopryazhenie (perechislenie) ==="
  putStrLn ("  q1 m -> g -> m = id (vosstanovlenie, Shag 2):      " ++ show q1)
  putStrLn ("  q2 biekciya {radj m} = {monotonnye g, g(1)=top}:   " ++ show q2 ++ "  (schet: " ++ show (length measS) ++ " = " ++ show (length allGS) ++ ")")
  putStrLn ("  q3 u srednego elementa cepi NET dopolneniya:       " ++ show q3)
  putStrLn ("  q4 tochechnaya ten d vosstanavlivaet meru:         " ++ show q4)
  putStrLn ("  q5 bulev etazh P(2): not(g(t)) = strogii uroven:   " ++ show q5)
  putStrLn ""
  let mm0 = [(0, 0), (1, 1/3), (2, 1)] :: [(Int, Double)]
  putStrLn "--- primer na cepi Serpinskogo 0 < m < 1: mera (0, 1/3, 1) ---"
  putStrLn ("  g po setke t = " ++ show gridQ ++ ":  " ++ show (gTabS mm0))
  putStrLn ("  teni tochek: d(pt_open) = " ++ show (dShad mm0 1) ++ ",  d(pt_closed) = " ++ show (dShad mm0 0))
  putStrLn "  net dopolneniya => Bel cherez compl vnutri odnogo L ne sobrat: nuzhen d-frame (tropa 3b)"

demoQ

=== Razdel 14.9: bestochechnaya Lambda = sopryazhenie (perechislenie) ===
  q1 m -> g -> m = id (vosstanovlenie, Shag 2):      True
  q2 biekciya {radj m} = {monotonnye g, g(1)=top}:   True  (schet: 10 = 10)
  q3 u srednego elementa cepi NET dopolneniya:       True
  q4 tochechnaya ten d vosstanavlivaet meru:         True
  q5 bulev etazh P(2): not(g(t)) = strogii uroven:   True

--- primer na cepi Serpinskogo 0 < m < 1: mera (0, 1/3, 1) ---
  g po setke t = [0.0,0.3333333333333333,0.6666666666666666,1.0]:  [0,1,1,2]
  teni tochek: d(pt_open) = 0.3333333333333333,  d(pt_closed) = 1.0
  net dopolneniya => Bel cherez compl vnutri odnogo L ne sobrat: nuzhen d-frame (tropa 3b)

# 14.10 Тропа 3b: d-Меры и Билатис Вердиктов (дискретный этаж)

> **Зачем.** §14.9 показал: в одном фрейме без дополнений пару $(\mathrm{Pl}, \mathrm{Bel})$ не собрать — нужен **d-frame** Юнга–Мошье. Здесь строим первый этаж: определяем **d-меру** (пару sup-мер с tot-аксиомой) и доказываем на дискретном d-frame, что согласованность интервального вердикта — $\mathrm{Bel} \le \mathrm{Pl}$, отсутствие `bContra` — **выводится**, а не постулируется. Билатис `IV` из библиотеки (`Bitopos.hs`) наконец играет главную роль: события отображаются в него вердиктами, свидетельства сужают коробки оценок. Общий (недискретный) d-frame — следующий шаг (3c).

## d-frame и дискретный пример

**d-frame** (Jung–Moshier 2006) — четвёрка $(L_+, L_-, \mathrm{con}, \mathrm{tot})$: два фрейма (открытые двух топологий битопологического пространства) и два отношения между ними — $\mathrm{con} \subseteq L_+ \times L_-$ («совместность»: свидетельства не пересекаются) и $\mathrm{tot}$ («тотальность»: свидетельства покрывают всё). Базовый пример — логика Белнапа **FOUR**: $L_\pm = \mathbf 2$, четыре пары $=$ `bTrue, bFalse, bUnknown, bContra`.

**Дискретный d-frame** над множеством $X$: $L_+ = L_- = \mathcal P(X)$, $\mathrm{con} = \{(U,V) : U \cap V = \varnothing\}$, $\mathrm{tot} = \{(U,V) : U \cup V = X\}$. Интерпретация для теории Пытьева: $U$ — открытое свидетельство *за* («$\tilde x$ где-то в $U$»), $V$ — свидетельство *против* («$\tilde x$ не в $V$»).

## d-мера

**Определение.** d-мера на d-frame — пара sup-мер $p: L_+ \to [0,1]$, $n: L_- \to [0,1]$ (по §14.8–14.9 обе имеют плотности; обозначим $\tau$ — плотность $p$, $\beta$ — плотность $n$), удовлетворяющая

$$\textbf{(M-tot)}: \quad (a_+, a_-) \in \mathrm{tot} \;\Longrightarrow\; p(a_+) \vee n(a_-) = 1.$$

Смысл: на покрывающей паре свидетельств хотя бы одна сторона полностью уверена. На дискретном d-frame меры $\mathrm{Pl}$ и $\mathrm{Bel}$ восстанавливаются как $\mathrm{Pl}(E) := p(E)$, $\mathrm{Bel}(E) := \theta(n(\complement E))$ (дополнение в $\mathcal P(X)$ есть; в общем d-frame его заменит структура $\mathrm{con}/\mathrm{tot}$ — задача 3c).

## Теорема (дискретный этаж)

**(a) Характеризация (M-tot) через плотности.** $(M\text{-tot}) \iff \sup_x \min(\tau(x), \beta(x)) = 1$ — «совместная нормировка»: есть точки, где обе плотности сколь угодно близки к 1.
*Доказательство.* По монотонности достаточно разбиений $(U, \complement U)$. Строгими уровнями (Лемма B, §14.8): $\max(\sup_U \tau, \sup_{\complement U}\beta) = 1$ для всех $U$ ⟺ для всех $t<1$ нет $U$ с $U \cap A_t = \varnothing$ и $B_t \subseteq U$ (где $A_t = \{\tau > t\}$, $B_t = \{\beta > t\}$) ⟺ $A_t \cap B_t \ne \varnothing$ для всех $t < 1$ ⟺ $\sup\min(\tau,\beta) = 1$. (Кандидат-нарушитель — всегда $U = B_t$.) $\square$
*Следствия:* $\mathrm{tot} \ni (X,\varnothing)$ и $(\varnothing, X)$ дают обе нормировки $\sup\tau = \sup\beta = 1$.

**(b) Согласованность вердикта выводится.** Для d-меры: $\mathrm{Bel}(E) \le \mathrm{Pl}(E)$ при всех $E$ — `bContra` исключён.
*Доказательство — чисто порядковое, без арифметики* (работает над любой кванталью с дуальным изоморфизмом $\theta$): пара $(E, \complement E) \in \mathrm{tot}$, значит $p(E) \vee n(\complement E) = \top$. Либо $p(E) = \top \ge \mathrm{Bel}(E)$; либо $n(\complement E) = \top$, и тогда $\mathrm{Bel}(E) = \theta(\top) = \bot \le p(E)$. $\square$

**(c) Сэндвич свидетельств.** Для $(U, V) \in \mathrm{con}$ и любого события $U \subseteq E \subseteq \complement V$:
$$p(U) \le \mathrm{Pl}(E) \le p(\complement V), \qquad \theta(n(\complement U)) \le \mathrm{Bel}(E) \le \theta(n(V))$$
— рост свидетельств ($U\uparrow, V\uparrow$ в $\mathrm{con}$) монотонно сужает обе коробки: уточнение вердикта в порядке знания $\le_k$ билатиса. *Доказательство* — монотонность $p, n$ и антитонность $\theta$. $\square$

**(d) Частные случаи.** Дуальная согласованность $\beta = \tau$ возвращает §14.6: $\mathrm{Bel} = \theta \circ \mathrm{Pl} \circ \complement$, вердикт $v(E) = \mathrm{IV}(\mathrm{Bel}\,E, \mathrm{Pl}\,E)$ — интервал билатиса. Над кванталью $\mathbf{Bool}$ (инстанс библиотеки) те же определения дают ровно **FOUR**: с (M-tot) вердикты лежат в $\{$`bTrue`, `bFalse`, `bUnknown`$\}$, без неё появляется `bContra` — логика Белнапа как нульмерный случай теории Пытьева.

## Статус и что дальше (3c)

Доказан **дискретный** этаж ($L_\pm = \mathcal P(X)$, дополнение доступно). Открыто: общий d-frame — определить $\mathrm{Bel}$ без $\complement$, через $\mathrm{con}/\mathrm{tot}$ и правые сопряжённые §14.9 (кандидат: $\mathrm{Bel}(a_+) = \theta\bigl(\bigwedge\{n(a_-) : (a_+, a_-) \in \mathrm{tot}\}\bigr)$-стиль), и проверить, что дискретный этаж — частный случай. Литература — как в §14.9 (Jung–Moshier; Belnap; Ginsberg о билатисах).

## Этюд: билатис IV из библиотеки как носитель вердиктов

На $X = \{a, b\}$ с сеткой $\{0, \tfrac12, 1\}$ перечислением (81 пара плотностей) проверяем: **r1** — эквивалентность (a); **r2** — из (M-tot) нет `bContra`, а среди пар без (M-tot) он есть (счётчик); **r3** — сэндвич (c) для всех $\mathrm{con}$-пар и всех событий между ними; **r4** — над $\mathbf{Bool}$ вердикты дают FOUR (`bContra` только без tot); **r5** — при $\beta = \tau$ вердикт совпадает с интервалом $[\theta\mathrm{Pl}\complement, \mathrm{Pl}]$ из §14.6. Всюду используются `IV`, `leqT/leqK`, `bTrue/bFalse/bUnknown/bContra` из `Bitopos.hs`.

In [15]:
-- | Раздел 14.10: этюд — d-меры и билатис вердиктов IV (перечисление на X = {a,b}).
xsD :: String
xsD = "ab"

gridD :: [Double]
gridD = [0, 0.5, 1]

subsD :: [String]
subsD = map sort (subsequences xsD)

compD :: String -> String
compD e = [ c | c <- xsD, c `notElem` e ]

densD :: [Char -> Double]
densD = [ \c -> if c == 'a' then va else vb | va <- gridD, vb <- gridD ]

plD :: (Char -> Double) -> String -> Double
plD f e = maximum (0 : map f e)

belD :: (Char -> Double) -> String -> Double
belD beta e = 1 - plD beta (compD e)

totD :: [(String, String)]
totD = [ (u, v) | u <- subsD, v <- subsD, all (\c -> c `elem` u || c `elem` v) xsD ]

conD :: [(String, String)]
conD = [ (u, v) | u <- subsD, v <- subsD, not (any (`elem` v) u) ]

mTotD :: (Char -> Double) -> (Char -> Double) -> Bool
mTotD tau beta = and [ max (plD tau u) (plD beta v) >= 1 - 1e-9 | (u, v) <- totD ]

r1 :: Bool
r1 = and [ mTotD tau beta == (maximum [ min (tau c) (beta c) | c <- xsD ] >= 1 - 1e-9) | tau <- densD, beta <- densD ]

noContraD :: (Char -> Double) -> (Char -> Double) -> Bool
noContraD tau beta = and [ belD beta e <= plD tau e + 1e-9 | e <- subsD ]

r2 :: Bool
r2 = and [ noContraD tau beta | tau <- densD, beta <- densD, mTotD tau beta ] && or [ not (noContraD tau beta) | tau <- densD, beta <- densD, not (mTotD tau beta) ]

r3 :: Bool
r3 = and [ plD tau u <= plD tau e + 1e-9 && plD tau e <= plD tau (compD v) + 1e-9 && (1 - plD beta (compD u)) <= belD beta e + 1e-9 && belD beta e <= (1 - plD beta v) + 1e-9 | tau <- densD, beta <- densD, (u, v) <- conD, e <- subsD, all (`elem` e) u, all (`elem` compD v) e ]

r3b :: Bool
r3b = and [ leqK (IV (plD tau u) (plD tau (compD v))) (IV (plD tau u') (plD tau (compD v'))) | tau <- densD, (u, v) <- conD, (u', v') <- conD, all (`elem` u') u, all (`elem` v') v ]

densB :: [Char -> Bool]
densB = [ \c -> if c == 'a' then ta else tb | ta <- [False, True], tb <- [False, True] ]

verdB :: (Char -> Bool) -> (Char -> Bool) -> String -> IV
verdB tb bb e = IV (if any bb (compD e) then 0 else 1) (if any tb e then 1 else 0)

mTotB :: (Char -> Bool) -> (Char -> Bool) -> Bool
mTotB tb bb = and [ any tb u || any bb v | (u, v) <- totD ]

r4 :: Bool
r4 = and [ verdB tb bb e `elem` [bTrue, bFalse, bUnknown] | tb <- densB, bb <- densB, mTotB tb bb, e <- subsD ] && or [ verdB tb bb e == bContra | tb <- densB, bb <- densB, not (mTotB tb bb), e <- subsD ] && and [ or [ verdB tb bb e == w | tb <- densB, bb <- densB, mTotB tb bb, e <- subsD ] | w <- [bTrue, bFalse, bUnknown] ]

r5 :: Bool
r5 = and [ abs (belD tau e - unUI (belMeasure xsD (\c -> ui (1 - tau c)) e)) < 1e-9 | tau <- densD, e <- subsD ]

demoD :: IO ()
demoD = do
  putStrLn "=== Razdel 14.10: d-mery i bilatis verdiktov IV (X = {a,b}) ==="
  putStrLn ("  r1  (M-tot) <=> sup min(tau,beta) = 1:                 " ++ show r1)
  putStrLn ("  r2  tot => net bContra; bez tot bContra est:           " ++ show r2)
  putStrLn ("  r3  sendvich svidetelstv (granitsy Pl/Bel):            " ++ show r3)
  putStrLn ("  r3b utochnenie korobok po <=k (leqK iz Bitopos.hs):    " ++ show r3b)
  putStrLn ("  r4  nad Bool: FOUR Belnapa, bContra tolko bez tot:     " ++ show r4)
  putStrLn ("  r5  Bel d-mery = belMeasure biblioteki (Kan-storona):  " ++ show r5)
  putStrLn ""
  let tau0 c = if c == 'a' then 1 else 0.5
  putStrLn "--- verdikty v(E) = IV(Bel, Pl) dlya tau = beta = {a: 1.0, b: 0.5} ---"
  mapM_ (\e -> putStrLn ("  E = " ++ (if null e then "{}" else e) ++ replicate (4 - max 2 (length e)) ' ' ++ "->  " ++ show (IV (belD tau0 e) (plD tau0 e)))) subsD
  putStrLn "  (bFalse pri E={}, bTrue pri E=X, mezhdu nimi — chestnye intervaly bez bContra)"

demoD

=== Razdel 14.10: d-mery i bilatis verdiktov IV (X = {a,b}) ===
  r1  (M-tot) <=> sup min(tau,beta) = 1:                 True
  r2  tot => net bContra; bez tot bContra est:           True
  r3  sendvich svidetelstv (granitsy Pl/Bel):            True
  r3b utochnenie korobok po <=k (leqK iz Bitopos.hs):    True
  r4  nad Bool: FOUR Belnapa, bContra tolko bez tot:     True
  r5  Bel d-mery = belMeasure biblioteki (Kan-storona):  True

--- verdikty v(E) = IV(Bel, Pl) dlya tau = beta = {a: 1.0, b: 0.5} ---
  E = {}  ->  IV {ivBel = 0.0, ivPl = 0.0}
  E = a  ->  IV {ivBel = 0.5, ivPl = 1.0}
  E = b  ->  IV {ivBel = 0.0, ivPl = 0.5}
  E = ab  ->  IV {ivBel = 1.0, ivPl = 1.0}
  (bFalse pri E={}, bTrue pri E=X, mezhdu nimi — chestnye intervaly bez bContra)

# 15. Три Сравнительных Примера: Битопос vs Расширения Кана

> Цель: показать, как два подхода — битопологический и через расширения Кана — описывают одни и те же объекты, где совпадают и где расходятся.

| Пример | Сложность | Что показывает |
|--------|-----------|----------------|
| 1 | Простой | Дискретный X, монотонная $\tau$. Оба совпадают — точка отсчёта |
| 2 | Средний | Отображение $\varphi: X \to Y$. Кан даёт формулу раздела 3 автоматически |
| 3 | Сложный | X — poset, $[0,1]$-обогащённая категория. Подходы расходятся |

## 15.1 Пример 1 (Простой): Дискретный X

$X = \{\text{low}, \text{mid}, \text{high}\}$, $\tau$ монотонно возрастает: $0.3 < 0.7 < 1.0$.

### 🔺 Битопологический взгляд

$\mathcal{T}_{\uparrow}^X$ — прообразы открытых $(t,1]$:

| t | Открытое мн-во |
|-----|----------------|
| 0.7 | {high} |
| 0.3 | {mid, high} |
| 0 | X |

$\mathrm{Pl}(E) = \mathrm{Int}_{\mathcal{T}_{\uparrow}}(E) = \sup_{x \in E} \tau(x)$

### 🔺 Расширение Кана

$J: X \hookrightarrow \mathcal{P}(X)$, $J(x) = \{x\}$. Coend = $\sup$:

$$\mathrm{Lan}_J\,\tau\,(E) = \int^{x} [x \in E] \otimes_{\min} \tau(x) = \sup_{x \in E} \tau(x)$$

![Пример 1: дискретный X](../diagrams/subj/subj_example1.svg)

In [16]:
-- Пример 1: дискретный X = {Low, Mid, High}

data Level = Low | Mid | High deriving (Show, Eq, Ord, Enum, Bounded)

tauEx1 :: Level -> Double
tauEx1 Low  = 0.3
tauEx1 Mid  = 0.7
tauEx1 High = 1.0

tauBarEx1 :: Level -> Double
tauBarEx1 x = 1.0 - tauEx1 x  -- простейшая дуальная согласованность: theta = (1-)

-- Битопологический подход
-- T-up: открытые = {x : tau(x) > t}
scottOpenUp :: Double -> [Level] -> [Level]
scottOpenUp t xs = filter (\x -> tauEx1 x > t) xs

-- Pl = sup tau на E
plBitopo :: [Level] -> Double
plBitopo []  = 0.0
plBitopo xs  = maximum (map tauEx1 xs)

-- Bel = inf tauBar на X\E
belBitopo :: [Level] -> Double
belBitopo xs =
  let compl = filter (\x -> x `notElem` xs) [minBound..maxBound]
  in if null compl then 1.0 else minimum (map tauBarEx1 compl)

-- Расширение Кана: Lan_J tau (E) = sup_{x in E} min(1, tau(x))
lanKan :: [Level] -> Double
lanKan []  = 0.0
lanKan xs  = maximum [min 1.0 (tauEx1 x) | x <- xs]

-- Ran_J tauBar (E) = inf_{x not in E} max(hom([0,1])(1, tauBar x), ...)
-- На дискретном X: Ran = inf_{x not in E} tauBar(x) = Bel(E)
ranKan :: [Level] -> Double
ranKan xs =
  let compl = filter (\x -> x `notElem` xs) [minBound..maxBound]
  in if null compl then 1.0 else minimum (map tauBarEx1 compl)

demo_ex1 :: IO ()
demo_ex1 = do
  putStrLn "=== Пример 1: дискретный X ==="
  putStrLn ""
  let allE = [[Low,Mid], [Mid,High], [Low], [High], [Low,Mid,High]]
  putStrLn "E                    | Pl (битопос) | Pl (Кан) | Bel (битопос) | Bel (Кан)"
  putStrLn (replicate 80 '-')
  mapM_ (\e -> do
    let pb = plBitopo e
        pk = lanKan e
        bb = belBitopo e
        bk = ranKan e
        match = if abs (pb-pk) < 1e-9 && abs (bb-bk) < 1e-9 then "== СОВПАДАЮТ ==" else "!! РАСХОДЯТСЯ !!"
    putStrLn (show e ++ replicate (20 - length (show e)) ' '
              ++ "| " ++ show pb ++ "        | " ++ show pk
              ++ "   | " ++ show bb ++ "          | " ++ show bk
              ++ "  " ++ match)) allE
  putStrLn ""
  putStrLn "T-up открытые при t=0.5:"
  print (scottOpenUp 0.5 [minBound..maxBound])

demo_ex1

Line 16: Eta reduce
Found:
scottOpenUp t xs = filter (\ x -> tauEx1 x > t) xs
Why not:
scottOpenUp t = filter (\ x -> tauEx1 x > t)Line 26: Avoid lambda using `infix`
Found:
(\ x -> x `notElem` xs)
Why not:
(`notElem` xs)Line 38: Avoid lambda using `infix`
Found:
(\ x -> x `notElem` xs)
Why not:
(`notElem` xs)

=== Пример 1: дискретный X ===

E                    | Pl (битопос) | Pl (Кан) | Bel (битопос) | Bel (Кан)
--------------------------------------------------------------------------------
[Low,Mid]           | 0.7        | 0.7   | 0.0          | 0.0  == СОВПАДАЮТ ==
[Mid,High]          | 1.0        | 1.0   | 0.7          | 0.7  == СОВПАДАЮТ ==
[Low]               | 0.3        | 0.3   | 0.0          | 0.0  == СОВПАДАЮТ ==
[High]              | 1.0        | 1.0   | 0.30000000000000004          | 0.30000000000000004  == СОВПАДАЮТ ==
[Low,Mid,High]      | 1.0        | 1.0   | 1.0          | 1.0  == СОВПАДАЮТ ==

T-up открытые при t=0.5:
[Mid,High]

![Diagram: Example 1 Discrete X](../diagrams/subjective/subj_example1.svg)

---

## 15.2 Пример 2 (Средний): Отображение $\varphi: X \to Y$

$X = \{\text{TempLow, TempMed, VibLow, VibHigh}\}$ с $\tau$,  
$Y = \{\text{Normal, Fault}\}$, $\varphi$ — диагностическое правило.

### 🔺 Битопологический взгляд

Нужно вычислить прямой образ топологии $\varphi_*(\mathcal{T}_{\uparrow}^X)$ и пересчитать $\mathrm{Pl}_Y$.

### 🔺 Расширение Кана

Формула из **раздела 3** ноутбука возникает автоматически как левое расширение Кана вдоль $\varphi$:

$$\tau^{\tilde{y}}(y) = \mathrm{Lan}_\varphi\,\tau\,(y) = \sup_{\varphi(x)=y} \tau(x)$$

![Пример 2: отображение phi](../diagrams/subj/subj_example2.svg)

In [17]:
-- Пример 2: phi: X -> Y, проталкивание tau

data Sensor  = TempLow | TempMed | VibLow | VibHigh deriving (Show, Eq, Ord, Enum, Bounded)
data Status  = Normal  | Fault              deriving (Show, Eq, Ord, Enum, Bounded)

tauX :: Sensor -> Double
tauX TempLow  = 0.9
tauX TempMed  = 0.5
tauX VibLow   = 0.7
tauX VibHigh  = 0.3

-- phi: диагностическое правило
phi :: Sensor -> Status
phi TempLow  = Normal
phi TempMed  = Normal
phi VibLow   = Normal
phi VibHigh  = Fault

-- Lan_phi tau: формула раздела 3 Пытьева
lanPhi :: Status -> Double
lanPhi y = maximum [tauX x | x <- [minBound..maxBound], phi x == y]

-- Битопологический подход: tau_Y(y) = sup_{phi(x)=y} tau(x)
-- Для дискретного X: прямой образ топологии совпадает с той же формулой
plYBitopo :: Status -> Double
plYBitopo y = maximum [tauX x | x <- [minBound..maxBound], phi x == y]

demo_ex2 :: IO ()
demo_ex2 = do
  putStrLn "=== Пример 2: phi: X -> Y ==="
  putStrLn ""
  putStrLn "tau на X:"
  mapM_ (\x -> putStrLn ("  " ++ show x ++ " -> " ++ show (tauX x) ++ "  phi -> " ++ show (phi x)))
        [minBound..maxBound :: Sensor]
  putStrLn ""
  putStrLn "Проталкивание tau вдоль phi:"
  mapM_ (\y -> do
    let pk = lanPhi y
        pb = plYBitopo y
        match = if abs (pk-pb) < 1e-9 then "== совпадают ==" else "!! РАСХОДЯТСЯ !!"
    putStrLn ("  " ++ show y ++ ": Lan_phi=" ++ show pk ++ "  Bitopo=" ++ show pb ++ "  " ++ match))
    [minBound..maxBound :: Status]
  putStrLn ""
  putStrLn "Вывод: tau_Y(y) = sup_{phi(x)=y} tau(x) (раздел 3 Пытьева)"
  putStrLn "       = Lan_phi tau.  Автоматически."

demo_ex2

=== Пример 2: phi: X -> Y ===

tau на X:
  TempLow -> 0.9  phi -> Normal
  TempMed -> 0.5  phi -> Normal
  VibLow -> 0.7  phi -> Normal
  VibHigh -> 0.3  phi -> Fault

Проталкивание tau вдоль phi:
  Normal: Lan_phi=0.9  Bitopo=0.9  == совпадают ==
  Fault: Lan_phi=0.3  Bitopo=0.3  == совпадают ==

Вывод: tau_Y(y) = sup_{phi(x)=y} tau(x) (раздел 3 Пытьева)
       = Lan_phi tau.  Автоматически.

![Diagram: Example 2 phi](../diagrams/subjective/subj_example2.svg)

---

## 15.3 Пример 3 (Сложный): X — Poset, $[0,1]$-обогащённая Категория

X — поset состояний: $\text{OK} \leq \text{Warn} \leq \text{Fail}$ и $\text{OK} \leq \text{Degrade} \leq \text{Fail}$.  
$\tau$ монотонна по порядку: $\tau(\text{OK})=0.2,\; \tau(\text{Warn})=0.6,\; \tau(\text{Degrade})=0.8,\; \tau(\text{Fail})=1.0$.

**Ключевое отличие:** $X$ теперь не дискретная $[0,1]$-категория. Хом-объекты:
$$\mathbf{X}(x, y) = [x \leq y] \in \{0, 1\} \subset [0,1]$$

Обогащённое левое расширение Кана:
$$\mathrm{Lan}_J^{\mathcal{V}}\,\tau\,(E) = \sup_{x \in X} \min\bigl(\sup_{e \in E} \mathbf{X}(x, e),\; \tau(x)\bigr)$$

Когда $\mathbf{X}(x,e) \in \{0,1\}$ — совпадает с дискретным. Но если перейти к непрерывному порядку (например, $\mathbf{X}(x,y) = \tau(y) - \tau(x)$ при $x \leq y$) — расходится.

### 🔺 Где расходятся?

Битопос строится по $\tau$, не зная про $\mathbf{X}(x,y)$.  
Обогащённый Кан использует $\mathbf{X}(x,y)$ как «вес перехода».  
При $\mathbf{X}(x,y) = \tau(y)/\tau(x)$ (нормированный) — расхождение становится измеримым.

![Пример 3: poset X](../diagrams/subj/subj_example3.svg)

In [18]:
-- Пример 3: X как poset, [0,1]-обогащённая категория

data State3 = OK3 | Warn3 | Degrade3 | Fail3 deriving (Show, Eq, Ord, Enum, Bounded)

tauP :: State3 -> Double
tauP OK3      = 0.2
tauP Warn3    = 0.6
tauP Degrade3 = 0.8
tauP Fail3    = 1.0

-- Порядок: OK <= Warn <= Fail, OK <= Degrade <= Fail
leq :: State3 -> State3 -> Bool
leq OK3      _         = True
leq Warn3    Warn3     = True
leq Warn3    Fail3     = True
leq Degrade3 Degrade3  = True
leq Degrade3 Fail3     = True
leq Fail3    Fail3     = True
leq _        _         = False

-- Хом-объекты трёх вариантов [0,1]-обогащения:

-- Вариант A: дискретная категория X(x,y) = [x==y]
homDiscrete :: State3 -> State3 -> Double
homDiscrete x y = if x == y then 1.0 else 0.0

-- Вариант B: poset X(x,y) = [x<=y] in {0,1}
homPoset :: State3 -> State3 -> Double
homPoset x y = if leq x y then 1.0 else 0.0

-- Вариант C: "непрерывный" вес перехода X(x,y) = tau(y)/tau(x) при x<=y
homContinuous :: State3 -> State3 -> Double
homContinuous x y
  | leq x y   = tauP y / tauP x
  | otherwise = 0.0

-- Обобщённый обогащённый Lan_J tau (E) через заданный hom
-- Lan_J tau (E) = sup_x min(sup_{e in E} hom(x,e), tau(x))
lanEnriched :: (State3 -> State3 -> Double) -> [State3] -> Double
lanEnriched hom e
  | null e    = 0.0
  | otherwise = maximum
      [ min (maximum [hom x ex | ex <- e]) (tauP x)
      | x <- [minBound..maxBound] ]

-- Pl обычный (битопос, без структуры X)
plFlat :: [State3] -> Double
plFlat [] = 0.0
plFlat xs = maximum (map tauP xs)

demo_ex3 :: IO ()
demo_ex3 = do
  putStrLn "=== Пример 3: poset X, три варианта обогащения ==="
  putStrLn ""
  let testSets = [ [Warn3, Fail3]
                 , [OK3, Degrade3]
                 , [Warn3, Degrade3]
                 , [Fail3]
                 , [OK3] ]
  putStrLn "E                  | Pl(flat) | Lan(дискр) | Lan(poset) | Lan(непрерыв)"
  putStrLn (replicate 80 '-')
  mapM_ (\e -> do
    let pf = plFlat e
        ld = lanEnriched homDiscrete e
        lp = lanEnriched homPoset e
        lc = lanEnriched homContinuous e
    putStrLn (show e ++ replicate (18 - length (show e)) ' '
              ++ "| " ++ show pf
              ++ "   | " ++ show ld
              ++ "      | " ++ show lp
              ++ "      | " ++ show lc)) testSets
  putStrLn ""
  putStrLn "Наблюдения:"
  putStrLn "  Lan(дискр) == Pl(flat)        -- дискретный X = без структуры"
  putStrLn "  Lan(poset) == Pl(flat)        -- poset {0,1}: coend = sup, тот же результат"
  putStrLn "  Lan(непрерыв) может отличаться -- вес перехода масштабирует tau(x)"
  putStrLn ""
  putStrLn "Пример: E = [Warn3]"
  let e = [Warn3]
  putStrLn ("  Pl(flat)  = " ++ show (plFlat e))
  putStrLn ("  Lan(poset) = " ++ show (lanEnriched homPoset e))
  putStrLn ("  Lan(непрерыв) = " ++ show (lanEnriched homContinuous e))
  putStrLn "  (OK->Warn: hom = 0.6/0.2 = 3.0, но min(..., tau(OK)=0.2) = 0.2; Warn->Warn: 1.0, tau=0.6)"

demo_ex3

=== Пример 3: poset X, три варианта обогащения ===

E                  | Pl(flat) | Lan(дискр) | Lan(poset) | Lan(непрерыв)
--------------------------------------------------------------------------------
[Warn3,Fail3]     | 1.0   | 1.0      | 1.0      | 1.0
[OK3,Degrade3]    | 0.8   | 0.8      | 0.8      | 0.8
[Warn3,Degrade3]  | 0.8   | 0.8      | 0.8      | 0.8
[Fail3]           | 1.0   | 1.0      | 1.0      | 1.0
[OK3]             | 0.2   | 0.2      | 0.2      | 0.2

Наблюдения:
  Lan(дискр) == Pl(flat)        -- дискретный X = без структуры
  Lan(poset) == Pl(flat)        -- poset {0,1}: coend = sup, тот же результат
  Lan(непрерыв) может отличаться -- вес перехода масштабирует tau(x)

Пример: E = [Warn3]
  Pl(flat)  = 0.6
  Lan(poset) = 0.6
  Lan(непрерыв) = 0.6
  (OK->Warn: hom = 0.6/0.2 = 3.0, но min(..., tau(OK)=0.2) = 0.2; Warn->Warn: 1.0, tau=0.6)

![Diagram: Example 3 Poset](../diagrams/subjective/subj_example3.svg)

---

## 15.4 Сравнительный итог

| | Пример 1 (дискрет.) | Пример 2 (φ) | Пример 3 (poset) |
|-|---------------------|--------------|------------------|
| **Битопос** | ✅ даёт Pl, Bel | требует `φ_*(T)` | видит только `τ`, не `X(x,y)` |
| **Кан (дискр.)** | ✅ совпадает | `Lan_φ τ` — автоматически | = Pl(flat) |
| **Кан (обогащ.)** | = дискрет. | = дискрет. (если J фиксирован) | зависит от `X(x,y)` |

**Вывод:**
- На дискретном $X$ все подходы эквивалентны.
- Обогащённый Кан естественно обобщает формулу раздела 3 на произвольные $\varphi: X \to Y$.
- При непрерывном поряд. весе `X(x,y)` расширение Кана учитывает структуру $X$, недоступную битопосу.
- Это указывает на направление обобщения: **заменить $\mathbf{X}(x,y) \in \{0,1\}$ на $[0,1]$-значный вес** и изучить соответствующую субъективную модель.

In [19]:
-- | Совместимость: прежний внутриноутбучный API для раздела 16.
-- Библиотечные plMeasure/belMeasure (lib/KanExtension.hs) принимают домен,
-- распределение и событие; здесь восстанавливаем старую двухаргументную
-- форму (событие + распределение), перекрывая импорт — GHCi это разрешает.

plMeasure :: [a] -> (a -> Double) -> Double
plMeasure xs tau = maximum (0 : map tau xs)

belMeasure :: [a] -> (a -> Double) -> Double
belMeasure xsComp tauBar = minimum (1 : map tauBar xsComp)

putStrLn "compat OK: plMeasure/belMeasure (legacy signature)"

compat OK: plMeasure/belMeasure (legacy signature)

# 16. Практический пример: диагностика двигателя (3 эксперта)

Сведём аппарат субъективной модели на содержательной задаче. Три эксперта оценивают состояние
двигателя из множества $\{$OK, фильтр, подшипник, критическое$\}$, задавая распределения
правдоподобия $\tau$ и доверия $\bar\tau$. Мы комбинируем оценки с весами, считаем меры
события, интегралы риска (severity), применяем принцип относительности ($\gamma=\sqrt{\cdot}$)
и строим условную модель после показаний датчика.

Пример использует API субъективной модели из разделов 1--5 (`SubjModel`, `plMeasure`,
`belMeasure`, `plIntegral`, `belIntegral`).

In [20]:
-- S16: Практический пример — диагностика двигателя (3 эксперта)
-- Используем тип SubjModel и операции из разделов 1-5.

data Engine = EOK | EFilter | EBearing | ECrit deriving (Show, Eq, Ord, Enum, Bounded)

engStates :: [Engine]
engStates = [minBound .. maxBound]

-- Построить модель эксперта из таблиц; нормируем sup tau = 1
mkExpert :: [(Engine, Double)] -> [(Engine, Double)] -> SubjModel Engine
mkExpert plT belT = SubjModel engStates tau tbar
  where
    mx = maximum (map snd plT)
    tau x  = maybe 0 (/ mx) (lookup x plT)
    tbar x = maybe 0 id (lookup x belT)

expertA, expertB, expertC :: SubjModel Engine
expertA = mkExpert [(EOK,0.3),(EFilter,0.9),(EBearing,0.5),(ECrit,0.1)]
                   [(EOK,0.1),(EFilter,0.0),(EBearing,0.2),(ECrit,0.6)]
expertB = mkExpert [(EOK,0.2),(EFilter,0.7),(EBearing,0.8),(ECrit,0.3)]
                   [(EOK,0.2),(EFilter,0.1),(EBearing,0.0),(ECrit,0.5)]
expertC = mkExpert [(EOK,0.1),(EFilter,0.6),(EBearing,0.9),(ECrit,0.4)]
                   [(EOK,0.3),(EFilter,0.2),(EBearing,0.0),(ECrit,0.4)]

-- Комбинирование экспертов: взвешенная сумма tau/tauBar, затем нормировка sup tau = 1
combineExperts :: [(SubjModel Engine, Double)] -> SubjModel Engine
combineExperts ws = SubjModel engStates tau tbar
  where
    wsum      = sum (map snd ws)
    rawTau x  = sum [w * smTau m x    | (m, w) <- ws] / wsum
    rawTBar x = sum [w * smTauBar m x | (m, w) <- ws] / wsum
    mx        = maximum (map rawTau engStates)
    tau x     = if mx == 0 then rawTau x else rawTau x / mx
    tbar      = rawTBar

combined :: SubjModel Engine
combined = combineExperts [(expertA, 0.4), (expertB, 0.35), (expertC, 0.25)]

-- Условная модель: ограничить домен наблюдаемым событием и перенормировать
conditionModel :: SubjModel Engine -> [Engine] -> SubjModel Engine
conditionModel m obs = SubjModel obs tau (smTauBar m)
  where
    mx    = maximum (0 : map (smTau m) obs)
    tau x = if mx == 0 then smTau m x else smTau m x / mx

-- Серьёзность состояния (функция потерь для интеграла Сугено)
severity :: Engine -> Double
severity EOK      = 0.0
severity EFilter  = 0.3
severity EBearing = 0.6
severity ECrit    = 1.0

demo_engine :: IO ()
demo_engine = do
  putStrLn "=== Диагностика двигателя: 3 эксперта ==="
  let warn     = [EFilter, EBearing]
      compWarn = filter (`notElem` warn) engStates
  putStrLn $ "Pl({Filter,Bearing})  = " ++ show (plMeasure warn (smTau combined))
  putStrLn $ "Bel({Filter,Bearing}) = " ++ show (belMeasure compWarn (smTauBar combined))
  putStrLn $ "Зазор Pl - Bel        = "
           ++ show (plMeasure warn (smTau combined) - belMeasure compWarn (smTauBar combined))
  putStrLn ""
  putStrLn "-- Интегралы риска (severity) --"
  putStrLn $ "Pl-интеграл (оптимист):   " ++ show (plIntegral engStates (smTau combined) severity)
  putStrLn $ "Bel-интеграл (пессимист): " ++ show (belIntegral engStates (smTauBar combined) severity)
  putStrLn ""
  putStrLn "-- Принцип относительности (gamma = sqrt сохраняет порядок) --"
  let g = combined { smTau = sqrt . smTau combined }  -- принцип относительности
  putStrLn $ "Pl({Filter}) до:    " ++ show (plMeasure [EFilter] (smTau combined))
  putStrLn $ "Pl({Filter}) после: " ++ show (plMeasure [EFilter] (smTau g))
  putStrLn ""
  putStrLn "-- Условная модель (датчик исключил Critical) --"
  let c = conditionModel combined [EOK, EFilter, EBearing]
  putStrLn $ "Pl(Bearing | not Crit) = " ++ show (plMeasure [EBearing] (smTau c))
  putStrLn $ "Pl(Filter  | not Crit) = " ++ show (plMeasure [EFilter]  (smTau c))

demo_engine

Line 15: Use fromMaybe
Found:
maybe 0 id
Why not:
Data.Maybe.fromMaybe 0Line 53: Use camelCase
Found:
demo_engine :: IO ()
Why not:
demoEngine :: IO ()Line 54: Use camelCase
Found:
demo_engine = ...
Why not:
demoEngine = ...

=== Диагностика двигателя: 3 эксперта ===
Pl({Filter,Bearing})  = 1.0
Bel({Filter,Bearing}) = 0.185
Зазор Pl - Bel        = 0.815

-- Интегралы риска (severity) --
Pl-интеграл (оптимист):   0.6
Bel-интеграл (пессимист): 0.185

-- Принцип относительности (gamma = sqrt сохраняет порядок) --
Pl({Filter}) до:    1.0
Pl({Filter}) после: 1.0

-- Условная модель (датчик исключил Critical) --
Pl(Bearing | not Crit) = 0.9419252187748607
Pl(Filter  | not Crit) = 1.0

# 17. Монада Возможности — Поссибилистский Двойник Монады Гири

В [Uncertainty.ipynb](Uncertainty.ipynb) вероятность оформлена как монада: дискретные распределения (раздел 2), монада Гири (раздел 3), марковские цепи в категории Клейсли (раздел 7). Теория Пытьева даёт **точный поссибилистский аналог** этой конструкции.

## Определение

$$T(X) = \{\tau: X \to [0,1] \mid \sup_x \tau(x) = 1\} \qquad\text{— распределения правдоподобий}$$

- **Единица** (дельта Дирака): $\eta(x) = \tau$, где $\tau(x) = 1$, $\tau(y) = 0$ при $y \neq x$ — «точное знание» Пытьева
- **bind**: для $k: X \to T(Y)$

$$(\tau \mathbin{>\!\!>\!\!=} k)(y) = \sup_{x \in X} \min\{\tau(x),\, k(x)(y)\}$$

Это **pl-интеграл** из раздела 5: bind — это в точности $\mathrm{pl}_{\tau}$, применённый к ядру $k$. Теорема 1.1 Пытьева (представимость pl-интеграла) — это утверждение о том, что $T$ — монада, индуцированная кванталью $([0,1], \min, 1)$.

## Сравнение с монадой Гири

| | Гири (вероятность) | Возможность (Пытьев) |
|---|---|---|
| Объект | $\sum_x p(x) = 1$ | $\sup_x \tau(x) = 1$ |
| Полукольцо | $(\mathbb{R}_{\ge 0}, +, \cdot)$ | $([0,1], \max, \min)$ |
| bind | $\sum_x p(x) \cdot k(x)(y)$ | $\sup_x \min(\tau(x), k(x)(y))$ |
| Клейсли-композиция | умножение стох. матриц | sup-min «умножение» матриц |
| Категория Клейсли | стохастические отображения | $[0,1]$-обогащённые отношения |
| Независимое произведение | $p \otimes q$ | $\min$-произведение (Опр. 1.2 Пытьева) |

Обе — частные случаи **монады распределений над коммутативным полукольцом**; замена $(+,\cdot) \to (\max,\min)$ переводит марковские цепи Uncertainty.ipynb в поссибилистские переходные системы. Как и `Dist` через `Map`, это *ограниченная* монада (нужен `Ord` для ключей) — инстанс класса `Monad` Haskell не даётся, но законы монады выполняются и проверяются ниже.

In [21]:
-- | Раздел 17: монада возможности — из библиотеки (../lib/Distribution.hs)

data W = Sun | Rain | Fog deriving (Show, Eq, Ord)

stepW :: W -> Poss W
stepW Sun  = possOf [(Sun, 1.0), (Fog, 0.4), (Rain, 0.2)]
stepW Rain = possOf [(Rain, 1.0), (Fog, 0.7), (Sun, 0.3)]
stepW Fog  = possOf [(Fog, 1.0), (Sun, 0.6), (Rain, 0.6)]

demoPossMonad :: IO ()
demoPossMonad = do
  putStrLn "=== Razdel 17: monada vozmozhnosti (i ne tolko) ==="
  let m0 = possOf [(Sun, 1.0), (Rain, 0.5)]
      k2 w = possOf [(w, 1.0), (Fog, 0.5)]
  putStrLn $ "  [Poss]  zakony monady: " ++ show (checkMonadLaws m0 stepW k2 Sun)
  putStrLn $ "  sup posle bind = " ++ show (supPoss (bindD m0 stepW))
  mapM_ (\n -> putStrLn $ "  " ++ show n ++ " shagov ot Sun: "
               ++ showDistList (nStepsD n stepW Sun)) [1, 2, 3]
  putStrLn $ "  Stabilizaciya sup-min stepenej: "
             ++ show (eqDist (nStepsD 3 stepW Sun) (nStepsD 4 stepW Sun))
  -- ТА ЖЕ монада над вероятностным полукольцом: одна математика, две теории
  let stepP :: W -> Dist ProbW W
      stepP Sun  = distOf [(Sun, ProbW 0.7), (Fog, ProbW 0.2), (Rain, ProbW 0.1)]
      stepP Rain = distOf [(Rain, ProbW 0.5), (Fog, ProbW 0.3), (Sun, ProbW 0.2)]
      stepP Fog  = distOf [(Fog, ProbW 0.4), (Sun, ProbW 0.3), (Rain, ProbW 0.3)]
      etaP w = distOf [(w, ProbW 1.0)]
  putStrLn $ "  [ProbW] zakony monady: "
             ++ show (checkMonadLaws (etaP Sun) stepP etaP Sun)
  putStrLn $ "  [ProbW] 2 shaga: " ++ showDistList (nStepsD 2 stepP Sun)
  -- И над Bool: семантика достижимости
  let stepB :: W -> Dist Bool W
      stepB Sun  = distOf [(Sun, True), (Fog, True)]
      stepB Rain = distOf [(Rain, True)]
      stepB Fog  = distOf [(Fog, True), (Rain, True)]
  putStrLn $ "  [Bool] dostizhimost za 2 shaga iz Sun: "
             ++ showDistList (nStepsD 2 stepB Sun)

demoPossMonad

=== Razdel 17: monada vozmozhnosti (i ne tolko) ===
  [Poss]  zakony monady: True
  sup posle bind = 1.0
  1 shagov ot Sun: [(Sun,UI 1.0),(Rain,UI 0.2),(Fog,UI 0.4)]
  2 shagov ot Sun: [(Sun,UI 1.0),(Rain,UI 0.4),(Fog,UI 0.4)]
  3 shagov ot Sun: [(Sun,UI 1.0),(Rain,UI 0.4),(Fog,UI 0.4)]
  Stabilizaciya sup-min stepenej: True
  [ProbW] zakony monady: True
  [ProbW] 2 shaga: [(Sun,ProbW 0.5699999999999998),(Rain,ProbW 0.18),(Fog,ProbW 0.25)]
  [Bool] dostizhimost za 2 shaga iz Sun: [(Sun,True),(Rain,True),(Fog,True)]

# 18. Обогащённая $\mathbf{X}$: Когда Двойственность Исбелла Перестаёт Быть Тривиальной

В разделе 14 категория $\mathbf{X}$ дискретна ($\mathbf{X}(x,y) = [x=y]$), поэтому Йонеда и Исбелл вырождаются: coend схлопывается в $\sup$, и вся конструкция лишь воспроизводит формулы Пытьева. Содержательная картина возникает, если МИ оценивает ещё и **степень неразличимости состояний**.

## $\mathbf{X}$ как обобщённое метрическое пространство Ловера

Зададим $[0,1]$-обогащённую категорию: $\mathbf{X}(x,y) \in [0,1]$ — «правдоподобие того, что $x$ и $y$ неразличимы», с аксиомами

$$\mathbf{X}(x,x) = 1, \qquad \min\{\mathbf{X}(x,y),\, \mathbf{X}(y,z)\} \le \mathbf{X}(x,z)$$

(рефлексивность и $\min$-транзитивность — это в точности категория, обогащённая над кванталью $([0,1],\min,1)$; как у Ловера, у которого вместо неё берётся $([0,\infty], +, 0)$ и получается обобщённая метрика).

## Преснопы = распределения, согласованные с неразличимостью

$[0,1]$-пресноп $\varphi: \mathbf{X}^{op} \to [0,1]$ обязан удовлетворять $\min\{\mathbf{X}(x,y), \varphi(y)\} \le \varphi(x)$: похожие состояния обязаны иметь похожие правдоподобия. Произвольное $\tau$ преснопом быть не обязано — его **йонедовское пополнение** (Lan вдоль Йонеды):

$$\hat{\tau}(x) = \sup_{y} \min\{\mathbf{X}(x,y),\, \tau(y)\}$$

— наименьший пресноп над $\tau$: субъективная модель «доразмывается» по неразличимости.

## Двойственность Исбелла: $\mathcal{O} \dashv \mathrm{Spec}$ в числах

$$\mathcal{O}(\varphi)(c) = \inf_{x} \bigl(\varphi(x) \multimap \mathbf{X}(x,c)\bigr), \qquad \mathrm{Spec}(\psi)(c) = \inf_{x} \bigl(\psi(x) \multimap \mathbf{X}(c,x)\bigr)$$

Единица сопряжения даёт $\varphi \le \mathrm{Spec}(\mathcal{O}(\varphi))$, и тройка $\mathcal{O}\,\mathrm{Spec}\,\mathcal{O} = \mathcal{O}$. **Неподвижные точки** ($\varphi = \mathrm{Spec}(\mathcal{O}(\varphi))$) — это Isbell envelope / tight span: кандидат на роль «объективного пополнения» субъективной модели — состояния, которые можно добавить к $X$, не нарушив согласованности правдоподобий с неразличимостью. Гипотеза раздела 14 в этой постановке: **пара (Pl, Bel) дуально согласована по Пытьеву $\iff$ ($\tau$, $\bar\tau$) — избелловски сопряжённая пара**. Ниже всё проверяется численно.

In [22]:
-- | Раздел 18: обогащённая X и Isbell O -| Spec — из библиотеки (../lib/KanExtension.hs)

data St = StA | StB | StC deriving (Show, Eq, Ord, Enum, Bounded)

homSt :: St -> St -> UnitInterval
homSt x y | x == y = ltop
homSt StA StB = ui 0.7
homSt StB StA = ui 0.7
homSt _   _   = ui 0.5

tauRawSt :: St -> UnitInterval
tauRawSt StA = ui 1.0
tauRawSt StB = ui 0.2   -- нарушение преснопа: hom(A,B)=0.7, tau(A)=1
tauRawSt StC = ui 0.1

demoEnriched :: IO ()
demoEnriched = do
  putStrLn "=== Razdel 18: obogashchyonnaya X i Isbell ==="
  let xs = [StA, StB, StC]
      showQ f = show [ (x, unUI (f x)) | x <- xs ]
  putStrLn $ "  Tranzitivnost hom:  " ++ show (isTransitive homSt xs)
  putStrLn $ "  tauRaw - presnop?   " ++ show (isPresheaf homSt xs tauRawSt)
  let tauHat = yonedaHat homSt xs tauRawSt
  putStrLn $ "  tauHat = " ++ showQ tauHat
  putStrLn $ "  tauHat - presnop?   " ++ show (isPresheaf homSt xs tauHat)
  putStrLn $ "  Edinica Isbell (phi <= Spec(O phi)): "
             ++ show (checkIsbellUnit homSt xs tauHat)
  putStrLn $ "  Treugolnik O Spec O = O: " ++ show (checkIsbellTriangle homSt xs tauHat)
  let fix1 = isbellSpec homSt xs (isbellO homSt xs tauHat)
  putStrLn $ "  Spec(O(tauHat)) = " ++ showQ fix1 ++ " (tight span)"

demoEnriched

=== Razdel 18: obogashchyonnaya X i Isbell ===
  Tranzitivnost hom:  True
  tauRaw - presnop?   False
  tauHat = [(StA,1.0),(StB,0.7),(StC,0.5)]
  tauHat - presnop?   True
  Edinica Isbell (phi <= Spec(O phi)): True
  Treugolnik O Spec O = O: True
  Spec(O(tauHat)) = [(StA,1.0),(StB,0.7),(StC,0.5)] (tight span)

# ✅ Итоги: Теория Субъективного Моделирования Пытьева

## Основные результаты

| Раздел | Ключевое понятие | Источник |
|--------|-----------------|----------|
| 1 | Пространство $(X, \mathcal{P}(X), \mathrm{Pl}^{\tilde{x}}, \mathrm{Bel}^{\tilde{x}})$ — субъективная модель НОЭ | Часть 1, п. 1.1 |
| 2 | Шкалы $L = ([0,1], \max, \min)$ и $\bar{L} = ([0,1], \min, \max)$ | Часть 1, п. 1.3 |
| 3 | Меры Pl и Bel через $\sup$ и $\inf$ распределений $\tau, \bar{\tau}$ | Часть 1, п. 1.1 |
| 4 | Принцип относительности — группа $\Gamma$ автоморфизмов | Часть 1, п. 1.3 |
| 5 | Pl- и bel-интегралы: $\sup\min$ и $\inf\max$ (Теорема 1.1) | Часть 1, п. 1.6 |
| 6 | Субъективная независимость $\equiv$ Pl-независимость | Часть 1, п. 1.7 |
| 7 | Условные распределения через субъективные шкалы | Часть 1, п. 1.8 |
| 8 | Три варианта мер: основной, с фикс. точками, психофизический | Часть 1, п. 1.9 |
| 9 | Эмпирическое восстановление НЧ.НОЭ по наблюдениям | Часть 1, разд. 2 |
| 10 | Комбинирование через матрицы парных сравнений | Часть 1, п. 2.2 |
| 11 | Энтропии: $H(\tau) = \mathrm{pl}(\bar{\tau})$, $H(\bar{\tau}) = \mathrm{bel}(\tau)$ | Часть 2, разд. 2 |
| 12 | Оптимальное правило идентификации НО.НЧ.О. | Часть 2, разд. 3 |
| 13 | $[0,1]$ — quantale/фрейм; топологии Скотта $\mathcal{T}_{\uparrow}, \mathcal{T}_{\downarrow}$ индуцируют битопос | Категорное |
| 14 | $\mathrm{Lan}_J\,\tau = \mathrm{Pl} = \mathrm{Int}_{\mathcal{T}_{\uparrow}}$; гипотеза: единство через Isbell duality над $[0,1]$ | Гипотеза |

## Ключевые формулы

$$\mathrm{Pl}(E) = \sup_{x \in E} \tau(x), \qquad \mathrm{Bel}(E) = \inf_{x \notin E} \bar{\tau}(x)$$

$$\mathrm{pl}_{\tau}(g) = \sup_x \min(\tau(x), g(x)), \qquad \mathrm{bel}_{\bar{\tau}}(\bar{g}) = \inf_x \max(\bar{\tau}(x), \bar{g}(x))$$

$$H(\tau) = \mathrm{pl}_{\tau}(\bar{\tau}), \qquad H(\bar{\tau}) = \mathrm{bel}_{\bar{\tau}}(\tau)$$

$$\mathrm{Lan}_J\,\tau\,(E) = \int^{x} J(x,E) \otimes_{\min} \tau(x) = \sup_{x \in E}\tau(x) = \mathrm{Pl}(E)$$

## Три вида знания

| Модель | $\tau(x)$ для всех $x$ | $\bar{\tau}(x)$ для всех $x$ |
|--------|----------------------|-----------------------------|
| Абсолютное незнание | $1$ | $0$ |
| Точное знание $x_0$ | $\mathbf{1}_{x=x_0}$ | $\mathbf{1}_{x=x_0}$ |
| Произвольная модель | $\tau: X \to [0,1]$ | $\bar{\tau}: X \to [0,1]$ |

## Статус категорной гипотезы

| Утверждение | Статус |
|-------------|--------|
| Coend = sup = Interior на дискретном $X$ | ✅ Доказано |
| End = inf = Closure на дискретном $X$ | ✅ Доказано |
| $[0,1]$ — quantale (фрейм) | ✅ Доказано |
| Isbell adjunction $\mathcal{O} \dashv \mathrm{Spec}$ над $[0,1]$ = пара (Pl, Bel) | ⚓ Гипотеза |
| Кондиционирование = residuation (правый сопряжённый к $\min(-,a)$) | ✅ Доказано + проверено |
| $\mathrm{Bel} = \mathrm{Ran}$ вдоль профунктора дополнения $\theta J$ | ✅ Дыра закрыта |
| Монада возможности (разд. 17): законы монады | ✅ Проверено численно |
| Isbell-сопряжение на обогащённой $\mathbf{X}$ (разд. 18) | ✅ Проверено численно |

В разделах 15–16 теория проверена делом: три сравнительных примера (битопос vs расширения Кана) и практическая диагностика двигателя тремя экспертами.

С ветки categorical-core-lib весь код ноутбука — вызовы библиотеки `lib/` (канонический источник: https://github.com/darklordshish/SubjectiveModeling, cabal-пакет с 33 тестами законов и двумя примерами).

---

**← Предыдущий:** [Неопределённость](Uncertainty.ipynb)
